# Knowledge Distillation – Kaggle 2×T4 Edition or P100

In [1]:
# Verify Kaggle GPU environment + capture key versions for the rest of the cells.
import torch
import subprocess
import sys
import os

print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA (torch): {torch.version.cuda}")
print(f"cuDNN: {torch.backends.cudnn.version()}")
print(f"GPU count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    cc = f"sm_{p.major}{p.minor}"
    print(f"  GPU {i}: {p.name:20s}  {p.total_memory/1e9:5.1f} GB  {cc}")

# nvcc — needed for source-build of mamba-ssm / causal-conv1d.
try:
    out = subprocess.check_output(["nvcc", "--version"], text=True)
    nvcc_line = [l for l in out.splitlines() if "release" in l.lower()]
    print(f"nvcc: {nvcc_line[0].strip() if nvcc_line else out.splitlines()[-1]}")
except Exception as e:
    print(f"nvcc: NOT FOUND ({e}). Source-build of mamba kernels will fail; "
          "fall back to slow Python path or use prebuilt wheels.")

if torch.cuda.device_count() > 0:
    cc = torch.cuda.get_device_capability(0)
    assert cc[0] >= 7, (
        f"GPU compute capability {
            cc[0]}.{
            cc[1]} is too old for mamba-ssm CUDA kernels. "
        "Switch the Kaggle accelerator to 'GPU T4 x2' (sm_75) instead of P100 (sm_60)."
    )


Python: 3.12.12
PyTorch: 2.10.0+cu128
CUDA (torch): 12.8
cuDNN: 91002
GPU count: 2
  GPU 0: Tesla T4               15.6 GB  sm_75
  GPU 1: Tesla T4               15.6 GB  sm_75
nvcc: Cuda compilation tools, release 12.8, V12.8.93


## Install requirements

In [2]:
!apt-get install -y -qq ffmpeg libavcodec-extra > /dev/null 2>&1
# Don't pin transformers — the Kaggle default (currently v5.x) works with
# the mamba-ssm fast-path kernels. A hard pin gets silently overridden by
# huggingface_hub / openai-whisper anyway, so it just adds confusion.
!pip install -q -U transformers
!pip install -q "datasets[audio]<4.0.0"
!pip install -q sentencepiece jiwer evaluate
!pip install -q WeTextProcessing
!pip install -q -U openai-whisper
!pip install -q --upgrade huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 774.9/774.9 kB 51.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.2/119.2 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 56.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 11.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are i

In [3]:
# ── Mamba CUDA kernels — auto-resolve for the actual env ─────────
import sys, torch, importlib, subprocess

cc_major = torch.cuda.get_device_capability(0)[0]
assert cc_major >= 7, (
    f"GPU compute capability {cc_major}.x is too old for mamba-ssm CUDA kernels. "
    "Switch the Kaggle accelerator to 'GPU T4 x2' (sm_75) instead of P100 (sm_60)."
)

# --no-build-isolation
!pip install -q --no-build-isolation ninja packaging wheel
!pip install -q --no-build-isolation causal-conv1d
!pip install -q --no-build-isolation mamba-ssm
!pip install -q speechbrain
#   !pip install -q "speechbrain==1.0.3"


def _check():
    failures = []
    try:
        from causal_conv1d import causal_conv1d_fn, causal_conv1d_update  # noqa
    except Exception as e:
        failures.append(f"causal_conv1d import failed: {e}")
    try:
        from mamba_ssm.ops.selective_scan_interface import (
            selective_scan_fn, mamba_inner_fn,
        )  # noqa
        from mamba_ssm.ops.triton.selective_state_update import selective_state_update  # noqa
    except Exception as e:
        failures.append(f"mamba_ssm import failed: {e}")
    try:
        import speechbrain  # noqa
    except Exception as e:
        failures.append(f"speechbrain import failed: {e}")
    return failures

problems = _check()
if problems:
    print("Fast-path NOT available. Reasons:")
    for p in problems:
        print(" -", p)
    raise RuntimeError("Mamba CUDA kernels did not import; fix before continuing.")
print("mamba-ssm + causal-conv1d kernels importable — fast path enabled")

  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.4/216.4 kB 1.7 MB/s eta 0:00:00
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 MB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 89.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 358.4/358.4 kB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 97.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.4/88.4 MB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 767.7/767.7 kB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.3/29.3 MB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.8/897.8 kB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 66.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4/323.4 kB 26.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver 

In [4]:
# Should not raise ImportError
from mamba_ssm.ops.selective_scan_interface import mamba_inner_fn, selective_scan_fn
from mamba_ssm.ops.triton.selective_state_update import selective_state_update


In [5]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("huggingface_token")
wandb_key = user_secrets.get_secret("wandb_api_key")
login(token=hf_token)


## Config pipeline

In [6]:
import os
import gc
import json
import math
import time
import random
import numpy as np
from pathlib import Path
from dataclasses import dataclass, field
from typing import List
import torch
import torchaudio
import torch
import functools
from pathlib import Path
from transformers import WhisperForConditionalGeneration, WhisperProcessor, AutoModelForSpeechSeq2Seq, AutoProcessor, MambaForCausalLM, MambaConfig
from datasets import load_dataset, Audio
import datasets
from tqdm import tqdm

from torch.utils.data import Dataset, DataLoader
import sentencepiece as spm

print = functools.partial(print, flush=True)
print(datasets.__version__)

# Hardware-aware helpers
if torch.cuda.device_count() > 0:
    NUM_GPUS = torch.cuda.device_count()
    GPU_MEM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
    IS_T4 = "T4" in torch.cuda.get_device_name(0)


_KAGGLE_INPUT_ROOT = "/kaggle/input"
_KAGGLE_DATASET_OWNER = "leviettrieu369"

# These are the six immutable Kaggle dataset mounts for this project. Keep the
# paths literal and stable: processed artifacts are uploaded here between
# sessions, while every mutable/resumable artifact is written under C.root.
_READONLY_DATASET_DIRS = {
    'tokenizer': "/kaggle/input/datasets/leviettrieu369/tokenizer",
    'filtering-transcription': "/kaggle/input/datasets/leviettrieu369/filtering-transcription",
    'mel-cache': "/kaggle/input/datasets/leviettrieu369/mel-cache",
    'pseudo-labeling': "/kaggle/input/datasets/leviettrieu369/pseudo-labeling",
    'distillation-checkpoint': "/kaggle/input/datasets/leviettrieu369/distillation-checkpoint",
    'teacher-cache': "/kaggle/input/datasets/leviettrieu369/teacher-cache",
}
_RESUME_CHECKPOINT_DATASET_ROOT = _READONLY_DATASET_DIRS['distillation-checkpoint']
# Older uploads placed the run under edge_asr; discovery below also supports
# checkpoints stored directly at the immutable dataset root.
_RESUME_CHECKPOINT_ROOT = os.path.join(_RESUME_CHECKPOINT_DATASET_ROOT, 'edge_asr')


def _kaggle_dataset_roots(slug):
    """Return the one canonical, read-only mount for a project artifact."""
    try:
        return [_READONLY_DATASET_DIRS[slug]]
    except KeyError as exc:
        raise KeyError(f'Unknown immutable Kaggle dataset slug: {slug!r}') from exc


def _resolve_kaggle_dir(slug, *relative_layouts):
    """Resolve a dataset directory without ever creating it under /kaggle/input."""
    layouts = relative_layouts or ("",)
    candidates = [
        os.path.normpath(os.path.join(root, relative))
        for root in _kaggle_dataset_roots(slug)
        for relative in layouts
    ]
    for candidate in candidates:
        if os.path.isdir(candidate):
            return candidate
    # Keep a deterministic expected path for diagnostics when the dataset is
    # not attached. Consumers fail with the full path instead of silently
    # falling back to a writable directory containing different data.
    return candidates[0]


def _resolve_kaggle_prefix(slug, *relative_prefixes):
    """Resolve a file prefix such as SentencePiece's path without `.model`."""
    candidates = [
        os.path.normpath(os.path.join(root, relative))
        for relative in relative_prefixes
        for root in _kaggle_dataset_roots(slug)
    ]
    for candidate in candidates:
        if os.path.isfile(candidate) or os.path.isfile(candidate + ".model"):
            return candidate
    return candidates[0]


def _dir_has_suffix(directory, suffix):
    if not os.path.isdir(directory):
        return False
    try:
        return any(entry.is_file() and entry.name.endswith(suffix)
                   for entry in os.scandir(directory))
    except OSError:
        return False


def _prefer_populated_dir(work_dir, input_dir, suffix):
    """Prefer newly generated artifacts only when they are actually present."""
    return work_dir if _dir_has_suffix(work_dir, suffix) else input_dir


def _completed_jsonl_path(directory, safe_name):
    """Return a complete artifact only; a JSONL without `.done` is resumable."""
    jsonl = os.path.join(directory, f'{safe_name}.jsonl')
    done = os.path.join(directory, f'{safe_name}.done')
    return jsonl if os.path.isfile(jsonl) and os.path.isfile(done) else None


def _select_jsonl_source(work_dir, input_dir, safe_name):
    """Choose a source without letting a partial working file shadow an upload."""
    work_complete = _completed_jsonl_path(work_dir, safe_name)
    input_complete = _completed_jsonl_path(input_dir, safe_name)
    if work_complete:
        return work_complete
    if input_complete:
        return input_complete
    work_partial = os.path.join(work_dir, f'{safe_name}.jsonl')
    return work_partial if os.path.isfile(work_partial) else os.path.join(
        input_dir, f'{safe_name}.jsonl')


def _has_chunk_balance(directory):
    return (
        os.path.isfile(os.path.join(directory, 'chunk_balance.done')) and
        os.path.isfile(os.path.join(directory, 'chunk_balance_summary.json'))
    )


def _prefer_balanced_filter_dir(work_dir, input_dir):
    """Use a balanced working output, otherwise a completed uploaded output."""
    return work_dir if _has_chunk_balance(work_dir) else input_dir


def _cache_tree_has_shards(directory):
    if not os.path.isdir(directory):
        return False
    try:
        for entry in os.scandir(directory):
            if entry.is_dir() and _dir_has_suffix(entry.path, ".npz"):
                return True
    except OSError:
        pass
    return False


def _select_cache_pair(work_teacher, work_mel, input_teacher, input_mel):
    """Never mix one side of a generated cache with one side of an input cache."""
    if (_cache_tree_has_shards(work_teacher) and
            _cache_tree_has_shards(work_mel)):
        return work_teacher, work_mel
    return input_teacher, input_mel


@dataclass
class Config:
    # Storage (Kaggle working dir, persists across saves)
    root: str = "/kaggle/working/edge_asr"

    # Teacher
    teacher_id: str = "openai/whisper-large-v3"
    # T4 does not support flash_attention_2
    teacher_attn: str = "sdpa"
    teacher_d: int = 1280

    # Pseudo Labeling
    psuedo_batch: int = 8
    max_label_len: int = 128
    max_samples_per_dataset: int = 30_000

    # Post-filter balance is a hard data invariant for this experiment. It
    # downsamples only after every quality and CTC-length gate, never repeats
    # examples, and therefore cannot amplify duplicate recordings.
    enforce_post_filter_language_balance: bool = True
    post_filter_chunks_per_language_cap: int | None = None
    post_filter_balance_seed: int = 42

    # Filtering config
    # loop filter
    max_repeats: int = 3

    # glitch filter
    max_word_len: int = 20

    # speed filter (words per second)
    min_wps: float = 0.5
    max_wps: float = 6.0

    # WER/CER gate (percentage)
    wer_threshold: float = 10.0

    # all-caps minimum length
    allcaps_min_len: int = 5

    # Prefer a human/source transcript when Whisper timestamps span the entire
    # utterance. Partial book windows keep timestamp-local Whisper text because
    # the source transcript can cover audio outside the selected chunk.
    prefer_source_text_for_full_utterance: bool = True
    source_text_full_utterance_tolerance_s: float = 0.25

    # Tokenizer / CTC targets. The defaults preserve compatibility with the
    # supplied checkpoint. For a clean lineage, set target normalization to
    # 'metric' and force a tokenizer rebuild before running any training cell.
    spm_vocab: int = 5000
    spm_coverage: float = 0.9995
    vocab_size: int = 5001     # SP pieces + 1 CTC blank; re-derived from the tokenizer at load time
    force_retokenize_targets: bool = True
    allow_tokenizer_rebuild: bool = True
    force_tokenizer_rebuild: bool = False
    ctc_target_normalization: str = 'none'  # 'none' (resume-safe) or 'metric'
    spm_balance_languages: bool = True
    spm_language_temperature: float = 0.50
    spm_max_sentences_per_language: int = 30_000

    # Empty by default: Korean was only ~1% of effective exposure and was not
    # the cause of the observed error. Set ('korean',) to skip it in TRAINING
    # while retaining Korean validation/test examples and tokenizer coverage.
    train_excluded_languages: tuple = ()

    # Student
    mamba_pretrained: str = "state-spaces/mamba-130m-hf"
    mamba_d: int = 768
    cnn_ch: int = 768
    cnn_ks: int = 5
    n_mels: int = 80
    sr: int = 16000

    # ConMamba encoder: unidirectional, T4-sized, trained from scratch.
    cm_d_model: int = 256
    cm_layers: int = 18
    cm_d_ffn: int = 1024
    cm_d_state: int = 16
    cm_expand: int = 2
    cm_d_conv: int = 4
    cm_kernel: int = 31          # Conformer conv-module kernel (odd)
    # unidirectional/causal: streaming-friendly, uses stock mamba_ssm
    cm_bidirectional: bool = False
    # Mel path: cached 100 fps log-mel -> 4x conv subsample -> ~25 fps.
    n_fft: int = 400
    hop: int = 160
    win: int = 400

    # Chunk cache is CTC-safe only if the collator pads, never crops.
    use_chunk_cache: bool = True
    chunk_seconds: float = 8.0
    chunk_overlap: float = 1.0
    allow_audio_crop_for_ctc: bool = False

    # Use the scratch recipe and resume scratch_kd_latest.pt when present.
    # This flag selects the lineage; it does not force random initialization.
    train_from_scratch: bool = True
    # Require the attached checkpoint bundle before launching training.
    resume_from_checkpoint: bool = True

    # StageSpec owns the live loss curriculum. These remain diagnostic
    # defaults for standalone cells; feature KD is retained throughout joint
    # training but annealed after it plateaus so CTC can specialize.
    a_kl: float = 0.0
    a_ctc: float = 1.0
    ctc_blank_bias_init: float = 0.0
    joint_kl_end_weight: float = 0.25
    joint_kl_hold_steps: int = 1000
    joint_kl_decay_steps: int = 5000

    # Low-confidence pseudo transcripts still provide useful acoustic feature
    # KD, but should not drive CTC as strongly as reliable labels. Missing
    # confidence remains weight 1.0 for backward-compatible caches.
    ctc_confidence_weighting: bool = True
    ctc_confidence_min: float = 0.50
    ctc_confidence_floor_weight: float = 0.25
    # temp: float = 2.0

    # Stage names are legacy; CTC alignment is learned in the chunk stage.
    fr_epochs: int = 0
    fr_lr: float = 3e-4
    fr_bs: int = 32  # 16 utterances/GPU; measured bs=8 peak was only 1.71 GB
    fr_ga: int = 1   # effective batch remains 32 without four DP launches/update
    max_s1: float = 10.0

    # Unfrozen stage. The supplied epoch-8 checkpoint ended at an
    # effectively zero cosine LR, so the continuation extends the horizon and
    # softly restarts from a bounded fraction of the original base LR.
    un_epochs: int = 16
    un_lr: float = 3e-5
    restart_exhausted_schedule: bool = True
    resume_lr_floor_factor: float = 1.0 / 3.0
    un_bs: int = 32  # measured headroom supports one effective batch/update
    un_ga: int = 1   # avoids four DataParallel launches per optimizer update
    gc_norm: float = 1.0
    warmup: int = 500
    max_s2: float = 12.0   # safety limit only; no CTC audio cropping

    # Datasets. Each row is (repo_id, config_or_None, split, text_column,
    # language, prefer_verified_source_text). load_dataset needs a repo ID,
    # never a Hugging Face web URL.
    datasets: List[tuple] = field(default_factory=lambda: [
        ## MLS adds large, human-transcribed coverage to the three low-volume
        ## European languages. The shared per-source cap makes their source
        ## counts comparable to Zeroth rather than letting book speech dominate.
        ('facebook/multilingual_librispeech', 'french',  'train', 'transcript', 'french',  True),
        ('facebook/multilingual_librispeech', 'spanish', 'train', 'transcript', 'spanish', True),
        ('facebook/multilingual_librispeech', 'german',  'train', 'transcript', 'german',  True),
        # ('fsicoli/common_voice_22_0', 'vi', 'train', 'sentence', 'vietnamese', True),
        # ('fsicoli/common_voice_22_0', 'ja', 'train', 'sentence', 'japanese',   True),
        # ('fsicoli/common_voice_22_0', 'ko', 'train', 'sentence', 'korean',     True),
        ## Human text labels + speaker_id; the official test split stays unused.
        # ('kresnik/zeroth_korean', None, 'train', 'text', 'korean', True),
        # ('fsicoli/common_voice_22_0', 'zh-CN', 'train', 'sentence', 'chinese', True),
        # ('fsicoli/common_voice_22_0', 'en', 'train', 'sentence', 'english', True),
        # ('nguyendv02/ViMD_Dataset', 'default', 'train', 'text', 'vietnamese', True),
        # ('pnnbao-ump/VieNeu-TTS', 'default', 'train', 'text', 'vietnamese', True)
    ])

    # Speaker-disjoint evaluation where a source exposes stable speaker IDs.
    # Other sources retain existing recording/source-disjoint grouping.
    speaker_group_columns: dict = field(default_factory=lambda: {
        'kresnik/zeroth_korean': 'speaker_id',
        'facebook/multilingual_librispeech': 'speaker_id',
    })

    # Regularization / memory policy
    dropout: float = 0.1   # applied between Mamba output and heads
    # The measured peak was 0.83 GB/GPU with checkpointing on a 16 GB T4.
    # Recomputing all 18 layers therefore wastes training time; the smoke test
    # remains the OOM gate if the architecture or chunk lengths change.
    grad_checkpointing: bool = False

    # The prior run spent ~31% of active time in full mid-epoch
    # validation. Use a deterministic language-stratified probe for monitoring
    # and reserve the full held-out split for epoch-end checkpoint selection.
    mid_epoch_val_updates: int = 3000
    midval_probe_samples: int = 512
    # Merged shard tails make the full 2k validation split only ~125 batches.
    # Keep the cap above that so epoch-end selection evaluates every example.
    val_max_batches: int = 500
    validation_batch_size: int = 16

    # Temperature-style shard repetition preserves every high-resource sample
    # while repeating low-resource languages moderately: repeat_l is
    # round((n_max / n_l) ** (1 - temperature)). 1.0 disables balancing.
    train_language_temperature: float = 0.70
    merge_shard_tails: bool = True
    macro_metric_min_samples: int = 20

    # Early stopping (per-stage, monitors held-out multilingual quality)
    es_enabled: bool = True
    es_patience: int = 3    # epochs without improvement before stopping
    es_min_delta: float = 1e-4  # minimum drop in val metric to count as improvement
    es_metric: str = 'loss'     # 'loss' (val total loss) or 'wer'

    # The shipped unfrozen baseline is known-bad: its CTC head collapsed to a
    # frequent non-blank token after the old KD+CTC/-5 blank-bias recipe. On a
    # fresh Kaggle session, ignore only that read-only unfrozen checkpoint and
    # restart Stage 2 from frozen_latest with a clean CTC head; same-session
    # writable unfrozen checkpoints still resume normally.
    ignore_readonly_unfrozen_baseline: bool = True

    seed: int = 42
    workers: int = 2      # Kaggle has fewer CPU cores than Colab HM
    log_every: int = 50

    def __post_init__(self):
        if self.ctc_target_normalization not in {'none', 'metric'}:
            raise ValueError(
                'ctc_target_normalization must be "none" or "metric", '
                f'got {self.ctc_target_normalization!r}')
        if not 0.0 < float(self.resume_lr_floor_factor) <= 1.0:
            raise ValueError('resume_lr_floor_factor must be in (0, 1]')
        if not 0.0 <= float(self.ctc_confidence_floor_weight) <= 1.0:
            raise ValueError('ctc_confidence_floor_weight must be in [0, 1]')
        if not 0.0 <= float(self.ctc_confidence_min) < 1.0:
            raise ValueError('ctc_confidence_min must be in [0, 1)')

        # Do not silently duplicate a Hugging Face source: repeated specs would
        # create separate JSONL/cache artifacts for the same audio. Different
        # MLS language configurations remain distinct sources by design.
        dataset_keys = [(name, config, split) for name, config, split, *_ in self.datasets]
        if len(dataset_keys) != len(set(dataset_keys)):
            raise ValueError(
                'C.datasets contains a duplicate (repo, config, split) source; '
                'remove it instead of fetching the same audio twice.')

        # Immutable upload locations. Never create or append beneath these
        # paths; they are the exact Kaggle mounts published between sessions.
        self.tokenizer_input_dir = _READONLY_DATASET_DIRS['tokenizer']
        self.pseudo_input_dir = _READONLY_DATASET_DIRS['pseudo-labeling']
        self.filter_input_dir = _READONLY_DATASET_DIRS['filtering-transcription']
        self.teacher_cache_input_dir = _READONLY_DATASET_DIRS['teacher-cache']
        self.mel_cache_input_dir = _READONLY_DATASET_DIRS['mel-cache']
        self.checkpoint_input_dir = _READONLY_DATASET_DIRS['distillation-checkpoint']

        # Backward-compatible reader names. Cache uploads may preserve the
        # historic teacher_cache/mel_cache inner folder, so resolve only that
        # layout below each fixed immutable dataset root.
        self.pseudo_dir = self.pseudo_input_dir
        self.filter_dir = self.filter_input_dir
        self.cache_dir = _resolve_kaggle_dir('teacher-cache', 'teacher_cache')
        self.mel_dir = _resolve_kaggle_dir('mel-cache', 'mel_cache')

        # Writable staging trees for fresh or resumed work. Partial uploads are
        # hydrated here before append/replace operations; /kaggle/input remains
        # immutable for the entire notebook run.
        self.chunk_pseudo_input_dir = self.pseudo_input_dir
        self.chunk_filter_input_dir = self.filter_input_dir
        self.chunk_teacher_input_dir = self.cache_dir
        self.chunk_mel_input_dir = self.mel_dir
        self.chunk_pseudo_work_dir = os.path.join(self.root, 'chunk_pseudo')
        self.chunk_filter_work_dir = os.path.join(self.root, 'chunk_filter')
        self.chunk_teacher_work_dir = os.path.join(
            self.root, 'chunk_teacher_cache', 'teacher_cache')
        self.chunk_mel_work_dir = os.path.join(
            self.root, 'chunk_mel_cache', 'mel_cache')
        # Backward-compatible names used in older notebook commentary/cells.
        self.chunk_pseudo_dir = self.chunk_pseudo_work_dir
        self.chunk_filter_dir = self.chunk_filter_work_dir

        # Prefer the tokenizer shipped with the checkpoint bundle so the
        # model and tokenizer travel together. Fall back to the standalone
        # tokenizer dataset for older Kaggle attachments.
        _bundle_spm_prefixes = [
            os.path.join(_RESUME_CHECKPOINT_ROOT, 'tokenizer', f'spm_{self.spm_vocab}'),
            os.path.join(self.checkpoint_input_dir, 'tokenizer', f'spm_{self.spm_vocab}'),
        ]
        self.spm_prefix = next(
            (prefix for prefix in _bundle_spm_prefixes
             if os.path.isfile(prefix + '.model') or os.path.isfile(prefix)),
            _resolve_kaggle_prefix('tokenizer', f'spm_{self.spm_vocab}', 'spm'),
        )

        # Checkpoints are always loaded from the fixed read-only upload root;
        # writes go only to C.ckpt_dir under /kaggle/working.
        self.ckpt_dataset_root = self.checkpoint_input_dir
        self.ckpt_dataset_roots = [self.checkpoint_input_dir]
        self.ckpt_load_dir = os.path.join(
            self.checkpoint_input_dir, 'checkpoints')
        self.ckpt_dir = f'{self.root}/checkpoints'

        # Loss log
        self.loss_log_path = f'{self.root}/training_log.jsonl'

        self.onnx_path = f'{self.root}/student.onnx'

    def safe_name(self, i):
        n, c, _split, _text, lang, _verified = self.datasets[i]
        # Config may be None (Zeroth); keep artifact names stable and meaningful.
        return f'{n.rsplit("/", 1)[-1]}_{c or lang}'

    def speaker_column(self, dataset_name):
        return self.speaker_group_columns.get(dataset_name)


C = Config()


def _discover_checkpoint_load_dirs(configured):
    """Find prior-session checkpoints across known Kaggle mount layouts.

    A saved Version may be mounted as either edge_asr/checkpoints or
    results/edge_asr/checkpoints. Keep all populated matches: compatibility
    checks later can skip a stale recipe in one source and continue to another.
    """
    dataset_roots = list(getattr(C, 'ckpt_dataset_roots', []))
    dataset_roots.insert(0, getattr(C, 'ckpt_dataset_root', None))
    candidates = [configured]
    for root in dataset_roots:
        if not root:
            continue
        candidates.extend([
            root,
            os.path.join(root, 'checkpoints'),
            os.path.join(root, 'edge_asr', 'checkpoints'),
            os.path.join(root, 'results', 'edge_asr', 'checkpoints'),
            os.path.join(root, 'checkpoints'),
        ])

    populated = []
    seen = set()
    for path in candidates:
        normalized = os.path.normpath(path)
        if normalized in seen:
            continue
        seen.add(normalized)
        if not os.path.isdir(normalized):
            continue
        pt_names = [name for name in os.listdir(normalized) if name.endswith('.pt')]
        if pt_names:
            populated.append((normalized, set(pt_names)))

    # A current recover_kd checkpoint is the most valuable resume source.
    # File count breaks ties without opening several hundred MB checkpoints.
    populated.sort(
        key=lambda item: (
            'scratch_joint_latest.pt' in item[1],
            'scratch_joint_best.pt' in item[1],
            'scratch_kd_latest.pt' in item[1],
            'recover_kd_latest.pt' in item[1],
            'recover_kd_best.pt' in item[1],
            'recover_ctc_latest.pt' in item[1],
            len(item[1]),
        ),
        reverse=True,
    )
    return [path for path, _ in populated]


C.ckpt_load_dirs = _discover_checkpoint_load_dirs(C.ckpt_load_dir)
if C.ckpt_load_dirs:
    # Backward-compatible primary source for tokenizer and W&B lookup. Model
    # checkpoint loading uses the full list and therefore cannot be shadowed.
    C.ckpt_load_dir = C.ckpt_load_dirs[0]

# Only mkdir writable destinations; /kaggle/input is read-only.


def _is_writable_path(p):
    if not p:
        return False
    path = os.path.abspath(os.path.normpath(os.fspath(p)))
    input_root = os.path.abspath(os.path.normpath(_KAGGLE_INPUT_ROOT))
    try:
        return os.path.commonpath([path, input_root]) != input_root
    except ValueError:
        return True


for d in [
    C.root,
    C.ckpt_dir,
    os.path.dirname(C.loss_log_path),
    C.chunk_pseudo_dir,
    C.chunk_filter_dir,
    os.path.dirname(C.chunk_teacher_work_dir),
    os.path.dirname(C.chunk_mel_work_dir),
]:
    if _is_writable_path(d):
        os.makedirs(d, exist_ok=True)

# Sanity report on checkpoint source availability.
if C.ckpt_load_dirs:
    print(f'Checkpoint source(s), in priority order: {len(C.ckpt_load_dirs)}')
    for _source_dir in C.ckpt_load_dirs:
        _existing = sorted(
            f for f in os.listdir(_source_dir) if f.endswith('.pt'))
        print(f'  {_source_dir}')
        print(
            f'    Found {len(_existing)} .pt file(s): '
            f'{_existing[:5]}{"..." if len(_existing) > 5 else ""}')
else:
    print('Checkpoint source not found in either supported Kaggle layout:')
    print(f'  configured={C.ckpt_load_dir}')
    print('  Training can start only if the requested stage does not require a source.')
print(f'Checkpoint destination (writable): {C.ckpt_dir}')
print(f'Loss log path: {C.loss_log_path}')

random.seed(C.seed)


def flush():
    """Free GPU memory on ALL devices."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        for i in range(torch.cuda.device_count()):
            alloc = torch.cuda.memory_allocated(i) / 1e9
            total = torch.cuda.get_device_properties(i).total_memory / 1e9
            print(f'  [GPU {i}] {alloc:.1f}/{total:.0f} GB')


print(f'{len(C.datasets)} datasets, root: {C.root}')
if torch.cuda.device_count() > 0:
    print(f'Hardware : {NUM_GPUS}× {torch.cuda.get_device_name(0)}')
print(f'Precision: float16  |  Attention: {C.teacher_attn}')
print(f'Frozen: bs={C.fr_bs} × GA={C.fr_ga} = {C.fr_bs*C.fr_ga} eff')
print(f'Unfrozen: bs={C.un_bs} × GA={C.un_ga} = {C.un_bs*C.un_ga} eff')


3.6.0
Checkpoint source(s), in priority order: 1
  /kaggle/input/datasets/leviettrieu369/distillation-checkpoint/results/edge_asr/checkpoints
    Found 4 .pt file(s): ['scratch_joint_best.pt', 'scratch_joint_latest.pt', 'scratch_kd_best.pt', 'scratch_kd_latest.pt']
Checkpoint destination (writable): /kaggle/working/edge_asr/checkpoints
Loss log path: /kaggle/working/edge_asr/training_log.jsonl
3 datasets, root: /kaggle/working/edge_asr
Hardware : 2× Tesla T4
Precision: float16  |  Attention: sdpa
Frozen: bs=32 × GA=1 = 32 eff
Unfrozen: bs=32 × GA=1 = 32 eff


## True chunk-level cache generation

Run these cells when you need to create the first true chunk-level dataset. They replace the old utterance-level pseudo-label/filter/cache path for the chunk CTC run.

Storage contract (paths are fixed in the config cell):

- Read-only uploaded inputs: `/kaggle/input/datasets/leviettrieu369/{tokenizer,filtering-transcription,mel-cache,pseudo-labeling,distillation-checkpoint,teacher-cache}`.
- Writable/resumable work: `/kaggle/working/edge_asr/...` only. The notebook copies an uploaded partial pseudo JSONL or a paired teacher/mel cache into this tree before appending shards or records.

Order:
1. Generate timestamped Whisper pseudo labels into `C.chunk_pseudo_work_dir`; upload completed or partial JSONL files to the fixed `pseudo-labeling` input path between sessions.
2. Convert timestamped segments into filtered manifests in `C.chunk_filter_work_dir`; upload completed balanced output to the fixed `filtering-transcription` input path.
3. Build aligned teacher and mel NPZ shards in `C.chunk_teacher_work_dir` and `C.chunk_mel_work_dir`.
4. Upload those two output trees to the fixed `teacher-cache` and `mel-cache` input paths before the next session.


### Chunk timestamp pseudo labels

Runs Whisper large-v3 once per original audio file and stores timestamped segments. This is source-level labeling, not training data yet; the next cell groups whole timestamped segments into true chunk examples.


### Inspect uploaded pseudo-label progress

Run this read-only diagnostic before resuming chunk pseudo-labeling. It counts the uploaded JSONL without changing it. A row count is only a progress estimate: the pseudo stage is complete **only** when its matching `.done` marker exists after the source stream finishes.


In [7]:
# Read-only pseudo-label progress inspector.
# This cell never writes beneath /kaggle/input. It reports the uploaded JSONL
# first, then the writable continuation that the generation cell will use.
from collections import Counter


def _inspect_pseudo_artifact(base_path, label):
    jsonl_path = base_path + '.jsonl'
    progress_path = base_path + '.progress.jsonl'
    state_path = base_path + '.state.json'
    done_path = base_path + '.done'
    stats = Counter()
    label_indices = set()
    processed_indices = set()

    if os.path.isfile(jsonl_path):
        with open(jsonl_path, 'r', encoding='utf-8') as fin:
            for line in fin:
                if not line.strip():
                    continue
                try:
                    idx = int(json.loads(line)['idx'])
                except Exception:
                    stats['invalid_label_lines'] += 1
                    continue
                stats['label_rows'] += 1
                if idx in label_indices:
                    stats['duplicate_label_indices'] += 1
                label_indices.add(idx)
                processed_indices.add(idx)

    if os.path.isfile(progress_path):
        with open(progress_path, 'r', encoding='utf-8') as fin:
            for line in fin:
                if not line.strip():
                    continue
                try:
                    progress = json.loads(line)
                    idx = int(progress['idx'])
                    status = str(progress.get('status', 'processed'))
                except Exception:
                    stats['invalid_progress_lines'] += 1
                    continue
                stats[f'progress_{status}'] += 1
                if idx in processed_indices:
                    stats['overlapping_progress_indices'] += 1
                processed_indices.add(idx)

    state = {}
    if os.path.isfile(state_path):
        try:
            with open(state_path, 'r', encoding='utf-8') as fin:
                state = json.load(fin)
        except Exception as exc:
            state = {'state_read_error': str(exc)}

    return {
        'label': label,
        'base_path': base_path,
        'exists': os.path.isfile(jsonl_path),
        'complete': os.path.isfile(jsonl_path) and os.path.isfile(done_path),
        'label_rows': int(stats['label_rows']),
        'unique_labels': len(label_indices),
        'processed_sources': len(processed_indices),
        'max_processed_idx': max(processed_indices, default=-1),
        'invalid_label_lines': int(stats['invalid_label_lines']),
        'invalid_progress_lines': int(stats['invalid_progress_lines']),
        'duplicate_label_indices': int(stats['duplicate_label_indices']),
        'state': state,
    }


for _di in range(len(C.datasets)):
    _safe = C.safe_name(_di)
    _input_base = os.path.join(C.chunk_pseudo_input_dir, _safe)
    _work_base = os.path.join(C.chunk_pseudo_work_dir, _safe)
    _input_state = _inspect_pseudo_artifact(
        _input_base, 'read-only uploaded input')
    _work_state = _inspect_pseudo_artifact(
        _work_base, 'writable continuation')

    print(f'[{_di + 1}/{len(C.datasets)}] {_safe}')
    for _state in [_input_state, _work_state]:
        if not _state['exists']:
            print(f"  {_state['label']}: absent -> {_state['base_path']}.jsonl")
            continue
        reason = _state['state'].get('completion_reason', 'legacy/unknown')
        print(
            f"  {_state['label']}: labels={_state['unique_labels']:,}, "
            f"processed_sources={_state['processed_sources']:,}, "
            f"max_idx={_state['max_processed_idx']:,}, "
            f"done={_state['complete']}, reason={reason}, "
            f"path={_state['base_path']}.jsonl")
        if (_state['invalid_label_lines'] or
                _state['invalid_progress_lines'] or
                _state['duplicate_label_indices']):
            print(
                '    warnings: '
                f"invalid_labels={_state['invalid_label_lines']}, "
                f"invalid_progress={_state['invalid_progress_lines']}, "
                f"duplicate_labels={_state['duplicate_label_indices']}")

    # Match the generator's behavior: a completed upload is authoritative.
    # Otherwise an existing writable continuation wins; a partial upload is
    # copied into working storage before generation resumes.
    if _input_state['complete']:
        _resume_state = _input_state
        _action = 'complete upload; generator will hydrate and reuse it'
    elif _work_state['exists']:
        _resume_state = _work_state
        _action = 'resume the writable continuation'
    else:
        _resume_state = _input_state
        _action = 'copy the uploaded partial artifact to working, then resume'

    _target = C.max_samples_per_dataset
    _remaining = (
        max(0, int(_target) - _resume_state['processed_sources'])
        if _target else None
    )
    print(f'  next action: {_action}')
    if _remaining is not None:
        print(
            f"  cap progress: {_resume_state['processed_sources']:,}/"
            f"{int(_target):,} processed source samples; "
            f"at most {_remaining:,} remain.")
    print(
        '  completion rule: `.done` is written after 30,000 processed source '
        'samples, or after natural exhaustion when the resource is smaller.')


[1/3] multilingual_librispeech_french
  read-only uploaded input: labels=12,581, processed_sources=12,581, max_idx=12,580, done=False, reason=legacy/unknown, path=/kaggle/input/datasets/leviettrieu369/pseudo-labeling/multilingual_librispeech_french.jsonl
  writable continuation: absent -> /kaggle/working/edge_asr/chunk_pseudo/multilingual_librispeech_french.jsonl
  next action: copy the uploaded partial artifact to working, then resume
  cap progress: 12,581/30,000 processed source samples; at most 17,419 remain.
  completion rule: `.done` is written after 30,000 processed source samples, or after natural exhaustion when the resource is smaller.
[2/3] multilingual_librispeech_spanish
  read-only uploaded input: labels=13,717, processed_sources=13,717, max_idx=13,716, done=False, reason=legacy/unknown, path=/kaggle/input/datasets/leviettrieu369/pseudo-labeling/multilingual_librispeech_spanish.jsonl
  writable continuation: absent -> /kaggle/working/edge_asr/chunk_pseudo/multilingual_lib

In [8]:
# Chunk-level timestamped pseudo labels.
# This reruns Whisper because the old pseudo labels did not store timestamps.
# Outputs are source-level JSONL files; the next cell turns them into chunk rows.

import math
from pathlib import Path
from tqdm.auto import tqdm

RUN_CHUNK_PSEUDO = True
CHUNK_PSEUDO_SESSION_HOURS = 10.5
CHUNK_PSEUDO_OVERWRITE = False
CHUNK_PSEUDO_OUT = C.chunk_pseudo_work_dir
if RUN_CHUNK_PSEUDO:
    if not _is_writable_path(CHUNK_PSEUDO_OUT):
        raise RuntimeError(f'CHUNK_PSEUDO_OUT must be writable: {CHUNK_PSEUDO_OUT}')
    os.makedirs(CHUNK_PSEUDO_OUT, exist_ok=True)
else:
    print(f'RUN_CHUNK_PSEUDO=False; writable output would be {CHUNK_PSEUDO_OUT}')

_CHUNK_LANG_CODES = {
    'english': 'en',
    'french': 'fr',
    'spanish': 'es',
    'german': 'de',
    'vietnamese': 'vi',
    'japanese': 'ja',
    'korean': 'ko',
    'chinese': 'zh',
    'en': 'en',
    'fr': 'fr',
    'es': 'es',
    'de': 'de',
    'vi': 'vi',
    'ja': 'ja',
    'ko': 'ko',
    'zh': 'zh',
    'zh-CN': 'zh',
}


def _chunk_whisper_model_name(teacher_id):
    name = teacher_id.split('/')[-1]
    return name.replace('whisper-', '')


def _chunk_lang_code(lang):
    return _CHUNK_LANG_CODES.get(
        str(lang), _CHUNK_LANG_CODES.get(str(lang).lower(), None))


def _safe_float(v, default=None):
    try:
        if v is None:
            return default
        out = float(v)
        if math.isnan(out):
            return default
        return out
    except Exception:
        return default


def _segment_payload(seg):
    text = str(seg.get('text', '')).strip()
    return {
        'start': _safe_float(seg.get('start'), 0.0),
        'end': _safe_float(seg.get('end'), 0.0),
        'text': text,
        'avg_logprob': _safe_float(seg.get('avg_logprob'), None),
        'no_speech_prob': _safe_float(seg.get('no_speech_prob'), None),
        'compression_ratio': _safe_float(seg.get('compression_ratio'), None),
    }


def _load_pseudo_indices(path):
    indices = set()
    invalid = 0
    if not os.path.isfile(path):
        return indices, invalid
    with open(path, 'r', encoding='utf-8') as fin:
        for line in fin:
            if not line.strip():
                continue
            try:
                indices.add(int(json.loads(line)['idx']))
            except Exception:
                invalid += 1
    return indices, invalid


def _write_pseudo_state(path, **state):
    tmp = path + '.tmp'
    with open(tmp, 'w', encoding='utf-8') as fout:
        json.dump(state, fout, ensure_ascii=False, indent=2)
    os.replace(tmp, path)


def _hydrate_pseudo_artifact(input_base, work_base, overwrite=False):
    """Copy immutable uploaded progress into working storage before mutation."""
    import shutil

    suffixes = ['.jsonl', '.progress.jsonl', '.state.json', '.done']
    if overwrite:
        for suffix in suffixes:
            path = work_base + suffix
            if os.path.isfile(path):
                os.remove(path)
        return []

    input_complete = (
        os.path.isfile(input_base + '.jsonl') and
        os.path.isfile(input_base + '.done')
    )
    copied = []
    for suffix in suffixes:
        src = input_base + suffix
        dst = work_base + suffix
        if not os.path.isfile(src):
            continue
        # A completed upload is authoritative. Otherwise seed only missing
        # writable files so a newer in-session continuation is never replaced.
        if input_complete or not os.path.exists(dst):
            shutil.copyfile(src, dst)
            copied.append(suffix)
    return copied


if RUN_CHUNK_PSEUDO:
    import whisper

    _chunk_pseudo_deadline = time.monotonic() + CHUNK_PSEUDO_SESSION_HOURS * 3600
    _chunk_pseudo_stopped = False

    _whisper_name = _chunk_whisper_model_name(C.teacher_id)
    _whisper_device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f'Loading openai-whisper model: {_whisper_name} on {_whisper_device}')
    chunk_whisper_model = whisper.load_model(_whisper_name, device=_whisper_device)

    for di, (ds_name, ds_cfg, ds_split, text_col, lang, _) in enumerate(C.datasets):
        safe = C.safe_name(di)
        work_base = os.path.join(CHUNK_PSEUDO_OUT, safe)
        input_base = os.path.join(C.chunk_pseudo_input_dir, safe)
        jsonl = work_base + '.jsonl'
        progress_jsonl = work_base + '.progress.jsonl'
        state_json = work_base + '.state.json'
        done_flag = work_base + '.done'

        copied = _hydrate_pseudo_artifact(
            input_base, work_base, overwrite=CHUNK_PSEUDO_OVERWRITE)
        if copied:
            print(
                f'[{di+1}/{len(C.datasets)}] {safe}: copied uploaded '
                f'{copied} to writable resume paths')
        if os.path.isfile(done_flag) and os.path.isfile(jsonl):
            print(
                f'[{di+1}/{len(C.datasets)}] {safe}: already complete in '
                f'writable storage -> {done_flag}')
            continue

        label_indices, invalid_labels = _load_pseudo_indices(jsonl)
        progress_indices, invalid_progress = _load_pseudo_indices(progress_jsonl)
        processed_indices = set(label_indices) | set(progress_indices)
        if invalid_labels or invalid_progress:
            raise RuntimeError(
                f'{safe}: corrupt resume artifact: invalid_labels={invalid_labels}, '
                f'invalid_progress={invalid_progress}')
        print(
            f'[{di+1}/{len(C.datasets)}] {safe}: resume with '
            f'{len(label_indices):,} labeled and '
            f'{len(processed_indices):,} processed source samples')

        ds_stream = load_dataset(
            ds_name, ds_cfg, split=ds_split,
            streaming=True, trust_remote_code=True,
        )
        speaker_col = C.speaker_column(ds_name)
        required_columns = ['audio', text_col]
        if speaker_col is not None:
            required_columns.append(speaker_col)
        missing_columns = set(required_columns) - set(ds_stream.column_names)
        if missing_columns:
            raise ValueError(
                f'{safe}: missing columns {sorted(missing_columns)}; '
                f'available={ds_stream.column_names}')
        ds_stream = ds_stream.select_columns(required_columns)
        ds_stream = ds_stream.cast_column('audio', Audio(sampling_rate=C.sr))
        target = int(C.max_samples_per_dataset) if C.max_samples_per_dataset else None
        if target:
            ds_stream = ds_stream.take(target)

        lang_code = _chunk_lang_code(lang)
        written, skipped, errors = 0, 0, 0
        max_idx_seen = -1
        source_stream_exhausted = False
        with (
            open(jsonl, 'a', encoding='utf-8', buffering=1) as fout,
            open(progress_jsonl, 'a', encoding='utf-8', buffering=1) as progress_out,
        ):
            pbar = tqdm(
                enumerate(ds_stream),
                total=target,
                desc=f'chunk-pseudo:{safe}',
                unit='src',
            )
            for idx, sample in pbar:
                max_idx_seen = max(max_idx_seen, int(idx))
                if idx in processed_indices:
                    continue
                if time.monotonic() >= _chunk_pseudo_deadline:
                    _chunk_pseudo_stopped = True
                    print(
                        'Session deadline reached; JSONL/progress are flushed '
                        'and resumable.')
                    break

                try:
                    audio = sample['audio']
                    raw = np.asarray(audio['array'], dtype=np.float32)
                    duration = float(len(raw) / C.sr)
                    if raw.size == 0:
                        progress_out.write(json.dumps({
                            'idx': int(idx),
                            'status': 'skipped_empty_audio',
                        }) + '\n')
                        processed_indices.add(int(idx))
                        skipped += 1
                        continue

                    result = chunk_whisper_model.transcribe(
                        raw,
                        language=lang_code,
                        task='transcribe',
                        fp16=torch.cuda.is_available(),
                        verbose=False,
                        temperature=0.0,
                        condition_on_previous_text=False,
                    )

                    segments = [
                        _segment_payload(s)
                        for s in result.get('segments', [])
                        if str(s.get('text', '')).strip()
                    ]
                    whisper_text = str(result.get('text', '')).strip()
                    if not whisper_text or not segments:
                        progress_out.write(json.dumps({
                            'idx': int(idx),
                            'status': 'skipped_no_transcript',
                        }) + '\n')
                        processed_indices.add(int(idx))
                        skipped += 1
                        continue

                    obj = {
                        'idx': int(idx),
                        'source_id': f'{safe}:{idx}',
                        # Zeroth labels are human transcripts. Whisper remains
                        # timestamp-only supervision for chunk boundaries.
                        'original': str(sample.get(text_col, '') or '').strip(),
                        'source_text_verified': bool(_),
                        'speaker_id': (
                            None if speaker_col is None else
                            str(sample.get(speaker_col, '') or '').strip() or None),
                        'lang': lang,
                        'duration': round(duration, 3),
                        'whisper': whisper_text,
                        'segments': segments,
                    }
                    fout.write(json.dumps(obj, ensure_ascii=False) + '\n')
                    progress_out.write(json.dumps({
                        'idx': int(idx),
                        'status': 'labeled',
                    }) + '\n')
                    label_indices.add(int(idx))
                    processed_indices.add(int(idx))
                    written += 1

                    if (idx + 1) % 100 == 0:
                        pbar.set_postfix_str(
                            f'processed={len(processed_indices)}/{target or "all"} '
                            f'written={written} skipped={skipped} errors={errors}'
                        )

                except RuntimeError as e:
                    errors += 1
                    if torch.cuda.is_available() and 'out of memory' in str(e).lower():
                        torch.cuda.empty_cache()
                    print(f'  [ERR] {safe} idx={idx}: {e}')
                    if errors > 20:
                        raise RuntimeError(f'Too many Whisper errors for {safe}') from e
                except Exception as e:
                    errors += 1
                    print(f'  [ERR] {safe} idx={idx}: {e}')
                    if errors > 20:
                        raise RuntimeError(f'Too many Whisper errors for {safe}') from e
            else:
                source_stream_exhausted = True
            pbar.close()

        state = {
            'safe_name': safe,
            'target_source_samples': target,
            'processed_sources': len(processed_indices),
            'pseudo_label_rows': len(label_indices),
            'max_source_index_seen': max_idx_seen,
            'written_this_run': written,
            'skipped_this_run': skipped,
            'errors_this_run': errors,
            'complete': False,
            'completion_reason': None,
        }

        if _chunk_pseudo_stopped:
            state['completion_reason'] = 'session_deadline'
            _write_pseudo_state(state_json, **state)
            print(
                f'  {safe}: partial progress '
                f'{len(processed_indices):,}/{target or "all"} processed; '
                f'rerun this cell to resume -> {jsonl}')
            break
        if errors:
            state['completion_reason'] = 'retryable_errors'
            _write_pseudo_state(state_json, **state)
            print(
                f'  {safe}: {errors} sources failed; leaving incomplete so '
                'the next run retries only those sources')
            continue
        if not source_stream_exhausted:
            raise RuntimeError(
                f'{safe}: pseudo source loop ended without deadline or exhaustion')

        cap_reached = bool(target and max_idx_seen + 1 >= target)
        state['complete'] = True
        state['completion_reason'] = (
            'source_cap_reached' if cap_reached else 'source_exhausted')
        _write_pseudo_state(state_json, **state)
        Path(done_flag).touch()
        print(
            f'  {safe}: COMPLETE ({state["completion_reason"]}); '
            f'processed={len(processed_indices):,}, labels={len(label_indices):,}, '
            f'skipped_this_run={skipped}, errors=0 -> {done_flag}')
else:
    print('RUN_CHUNK_PSEUDO=False; skipping timestamped pseudo-label generation.')


Loading openai-whisper model: large-v3 on cuda


100%|█████████████████████████████████████| 2.88G/2.88G [01:29<00:00, 34.3MiB/s]


[1/3] multilingual_librispeech_french: copied uploaded ['.jsonl'] to writable resume paths
[1/3] multilingual_librispeech_french: resume with 12,581 labeled and 12,581 processed source samples


README.md:   0%|          | 0.00/18.1k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/34 [00:00<?, ?it/s]

chunk-pseudo:multilingual_librispeech_french:   0%|          | 0/30000 [00:00<?, ?src/s]


100%|██████████| 1318/1318 [00:03<00:00, 357.35frames/s]

100%|██████████| 1129/1129 [00:02<00:00, 547.06frames/s]

100%|██████████| 1083/1083 [00:01<00:00, 561.11frames/s]

100%|██████████| 1756/1756 [00:02<00:00, 603.63frames/s]

100%|██████████| 1782/1782 [00:03<00:00, 524.94frames/s]

100%|██████████| 1647/1647 [00:03<00:00, 512.44frames/s]

100%|██████████| 1690/1690 [00:03<00:00, 528.85frames/s]

100%|██████████| 1103/1103 [00:02<00:00, 482.93frames/s]

100%|██████████| 1528/1528 [00:02<00:00, 535.74frames/s]

100%|██████████| 1168/1168 [00:01<00:00, 785.07frames/s]

100%|██████████| 1080/1080 [00:02<00:00, 539.25frames/s]

100%|██████████| 1271/1271 [00:02<00:00, 518.73frames/s]

100%|██████████| 1646/1646 [00:02<00:00, 549.64frames/s]

100%|██████████| 1719/1719 [00:03<00:00, 491.22frames/s]

100%|██████████| 1108/1108 [00:01<00:00, 581.08frames/s]

100%|██████████| 1481/1481 [00:02<00:00, 607.20frames/s]

100%|██████████| 1584/1584 [00:03<00:00, 522.56frames/s]

100%|████████

Session deadline reached; JSONL/progress are flushed and resumable.
  multilingual_librispeech_french: partial progress 25,269/30000 processed; rerun this cell to resume -> /kaggle/working/edge_asr/chunk_pseudo/multilingual_librispeech_french.jsonl


### Chunk manifest and filtering

Groups only whole Whisper timestamp segments into chunks. The overlap repeats complete segments in the next chunk, so every chunk transcript still matches the audio span used later by the cache builder.


In [9]:
# Build true chunk-level manifests from timestamped source pseudo labels.
# Each output line is one CTC training example: audio span + transcript + token IDs.

import collections
import math
import re
import unicodedata
from pathlib import Path
from tqdm.auto import tqdm

RUN_CHUNK_FILTER = False
# Rebuild stale manifests after the old source-level gate.
# False reuses a completed mounted manifest from the prior Kaggle session.
# Set True only when deliberately changing filtering/tokenizer rules.
CHUNK_FILTER_OVERWRITE = False
# Each source is selected independently below so a partial writable Korean
# JSONL cannot shadow a completed upload from a prior Kaggle session.
CHUNK_PSEUDO_IN = C.chunk_pseudo_input_dir
CHUNK_FILTER_OUT = C.chunk_filter_work_dir
CHUNK_BALANCE_SUMMARY = os.path.join(
    CHUNK_FILTER_OUT, 'chunk_balance_summary.json')
CHUNK_BALANCE_DONE = os.path.join(CHUNK_FILTER_OUT, 'chunk_balance.done')

# A balanced CTC manifest is valid only when every enabled source has finished
# timestamp pseudo-labeling.  A Kaggle "Run all" continues after the pseudo
# cell returns at its session deadline; without this preflight it could filter
# a partial Korean JSONL and publish an irreproducibly unbalanced cache.  The
# `.done` marker is written only after the source stream is exhausted, while a
# mounted completed manifest is reusable from a prior session.
_chunk_filter_pending_pseudo = []
if RUN_CHUNK_FILTER:
    for _di, _spec in enumerate(C.datasets):
        _safe = C.safe_name(_di)
        if not (
            _completed_jsonl_path(C.chunk_pseudo_work_dir, _safe) or
            _completed_jsonl_path(C.chunk_pseudo_input_dir, _safe)
        ):
            _chunk_filter_pending_pseudo.append(_safe)
    if _chunk_filter_pending_pseudo:
        print(
            'Chunk filter deferred: pseudo-labeling is incomplete for '
            f'{_chunk_filter_pending_pseudo}. Rerun the pseudo cell until it '
            'writes each `.done` marker, then rerun filtering.')
        RUN_CHUNK_FILTER = False

if RUN_CHUNK_FILTER:
    if not _is_writable_path(CHUNK_FILTER_OUT):
        raise RuntimeError(f'CHUNK_FILTER_OUT must be writable: {CHUNK_FILTER_OUT}')
    os.makedirs(CHUNK_FILTER_OUT, exist_ok=True)
    if 'sp' not in globals() or sp.GetPieceSize() != int(C.spm_vocab):
        sp = spm.SentencePieceProcessor()
        if not sp.Load(C.spm_prefix + '.model'):
            raise FileNotFoundError(f'Tokenizer not found: {C.spm_prefix}.model')
    if sp.GetPieceSize() != int(C.spm_vocab):
        raise RuntimeError(
            f'Chunk filter requires {C.spm_vocab} SentencePiece tokens, but '
            f'{C.spm_prefix}.model contains {sp.GetPieceSize()}. Attach the '
            'matching tokenizer dataset or build it in the writable work tree.')
    BLANK = sp.GetPieceSize()
    C.vocab_size = BLANK + 1
    print(f'Chunk filter input: {CHUNK_PSEUDO_IN}')
    print(f'Chunk filter tokenizer: {sp.GetPieceSize()} SP tokens, blank={BLANK}')

try:
    import jiwer
except Exception:
    jiwer = None

_MULTI_SPACE = re.compile(r'\s+')
_PUNCT = re.compile(
    r'[\u3000-\u303F\uFF00-\uFFEF\u2000-\u206F\u2E00-\u2E7F!"#$%&\'()*+,\-./:;<=>?@\[\]^_`{|}~]')
_HAS_CJK = re.compile(
    r'[\u4E00-\u9FFF\u3400-\u4DBF\uF900-\uFAFF\u3040-\u30FF\u31F0-\u31FF\uAC00-\uD7AF\u1100-\u11FF\u3130-\u318F]')
_CJK_LANGS = {'chinese', 'japanese', 'korean', 'zh', 'ja', 'ko', 'zh-CN'}


def _norm_text(text):
    text = unicodedata.normalize('NFKC', str(text or '')).lower()
    text = _PUNCT.sub(' ', text)
    return _MULTI_SPACE.sub(' ', text).strip()


def _training_target_text(text):
    """Mirror the target normalization selected for the training cell."""
    mode = getattr(C, 'ctc_target_normalization', 'none')
    if mode == 'none':
        return str(text or '').strip()
    if mode == 'metric':
        return _norm_text(text)
    raise ValueError(f'Unsupported ctc_target_normalization={mode!r}')


def _is_cjk(lang, text=''):
    return str(lang) in _CJK_LANGS or bool(_HAS_CJK.search(str(text or '')))


def _edit_distance(a, b):
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(
                prev[j] + 1,
                cur[j - 1] + 1,
                prev[j - 1] + (ca != cb),
            ))
        prev = cur
    return prev[-1]


def _wer(ref, hyp):
    ref, hyp = _norm_text(ref), _norm_text(hyp)
    if not ref:
        return 0.0 if not hyp else 1.0
    if jiwer is not None:
        return float(jiwer.wer(ref, hyp))
    return _edit_distance(ref.split(), hyp.split()) / max(1, len(ref.split()))


def _cer(ref, hyp):
    ref, hyp = _norm_text(ref), _norm_text(hyp)
    if not ref:
        return 0.0 if not hyp else 1.0
    if jiwer is not None:
        return float(jiwer.cer(ref, hyp))
    return _edit_distance(list(ref), list(hyp)) / max(1, len(ref))


def _max_consecutive_repeats(text, cjk=False):
    if cjk:
        tokens = list(_norm_text(text).replace(' ', ''))
    else:
        tokens = _norm_text(text).split()
    if len(tokens) <= 1:
        return len(tokens)
    best = cur = 1
    for i in range(1, len(tokens)):
        if tokens[i] == tokens[i - 1]:
            cur += 1
            best = max(best, cur)
        else:
            cur = 1
    return best


def _speech_rate_ok(text, duration_s, cjk=False):
    if duration_s <= 0:
        return False
    if cjk:
        cps = len(_norm_text(text).replace(' ', '')) / duration_s
        return 1.0 <= cps <= 12.0
    wps = len(_norm_text(text).split()) / duration_s
    return C.min_wps <= wps <= C.max_wps


def _has_long_word(text):
    words = _norm_text(text).split()
    return bool(words) and max(len(w) for w in words) > C.max_word_len


def _student_enc_len_from_mel_len(mel_len):
    lens = int(max(1, mel_len))
    k, p, s = C.cnn_ks, C.cnn_ks // 2, 2
    for _ in range(2):
        lens = ((lens + 2 * p - k) // s) + 1
    return max(1, lens)


def _ctc_required_len(token_ids):
    repeats = sum(1 for a, b in zip(token_ids, token_ids[1:]) if a == b)
    return len(token_ids) + repeats


def _safe_float(v, default=None):
    try:
        if v is None:
            return default
        out = float(v)
        if math.isnan(out):
            return default
        return out
    except Exception:
        return default


def _valid_segments(source_obj):
    out = []
    source_duration = _safe_float(source_obj.get('duration'), None)
    for seg in source_obj.get('segments', []):
        text = str(seg.get('text', '')).strip()
        start = _safe_float(seg.get('start'), None)
        end = _safe_float(seg.get('end'), None)
        if source_duration is not None and end is not None:
            # Whisper timestamps can overshoot the decoded file duration,
            # especially on short Common Voice clips. Clamp at filter time so
            # duration/rate/CTC checks describe the audio we can actually load.
            end = min(end, source_duration)
        if not text or start is None or end is None or end <= start:
            continue
        out.append({
            'start': max(0.0, start),
            'end': max(0.0, end),
            'text': text,
            'avg_logprob': _safe_float(seg.get('avg_logprob'), None),
            'no_speech_prob': _safe_float(seg.get('no_speech_prob'), None),
            'compression_ratio': _safe_float(seg.get('compression_ratio'), None),
        })
    return sorted(out, key=lambda s: (s['start'], s['end']))


def _segment_groups(segments, target_s, overlap_s, max_s):
    groups = []
    i = 0
    while i < len(segments):
        start = segments[i]['start']
        end = segments[i]['end']
        j = i
        while j + 1 < len(segments):
            cand_end = max(end, segments[j + 1]['end'])
            if cand_end - start > target_s:
                break
            if cand_end - start > max_s:
                break
            j += 1
            end = cand_end
        group = segments[i:j + 1]
        groups.append(group)

        overlap_start = end - max(0.0, overlap_s)
        next_i = j + 1
        if overlap_s > 0:
            for k in range(i, j + 1):
                if segments[k]['end'] > overlap_start:
                    next_i = k
                    break
        if next_i <= i:
            next_i = j + 1
        i = next_i
    return groups


def _chunk_from_group(source_obj, group, chunk_idx):
    teacher_text = _MULTI_SPACE.sub(
        ' ', ' '.join(s['text'].strip() for s in group)).strip()
    start = min(s['start'] for s in group)
    end = max(s['end'] for s in group)
    duration_s = end - start

    # Ground-truth/source text is safer than a pseudo transcript only when the
    # timestamp group covers the whole utterance. For partial MLS/book windows,
    # the original field can describe speech outside this chunk.
    original = str(source_obj.get('original', '') or '').strip()
    source_duration = _safe_float(source_obj.get('duration'), None)
    tolerance = float(getattr(
        C, 'source_text_full_utterance_tolerance_s', 0.25))
    covers_full_utterance = (
        source_duration is not None and start <= tolerance and
        end >= source_duration - tolerance)
    use_source_text = (
        bool(getattr(C, 'prefer_source_text_for_full_utterance', True)) and
        bool(source_obj.get('source_text_verified', False)) and
        bool(original) and covers_full_utterance)
    text = original if use_source_text else teacher_text
    label_source = 'source_transcript' if use_source_text else 'whisper_segment'

    avg_logs = [s['avg_logprob'] for s in group if s['avg_logprob'] is not None]
    no_speech = [s['no_speech_prob'] for s in group if s['no_speech_prob'] is not None]
    comp = [s['compression_ratio'] for s in group if s['compression_ratio'] is not None]
    mean_avg_logprob = float(np.mean(avg_logs)) if avg_logs else None
    teacher_conf = float(min(1.0, math.exp(mean_avg_logprob))
                         ) if mean_avg_logprob is not None else None
    source_id = source_obj.get('source_id') or f'source:{source_obj.get("idx")}'
    return {
        'idx': int(source_obj['idx']),
        'chunk_id': f'{source_id}:{chunk_idx:04d}',
        'source_id': source_id,
        # Zeroth groups on speaker; sources without speaker metadata group on
        # recording, preserving the existing anti-overlap split invariant.
        'split_id': str(source_obj.get('speaker_id') or source_id),
        'speaker_id': source_obj.get('speaker_id'),
        'chunk_start_s': round(float(start), 3),
        'chunk_end_s': round(float(end), 3),
        'duration_s': round(float(duration_s), 3),
        'text': text,
        'teacher_text': teacher_text,
        'label_source': label_source,
        'lang': source_obj.get('lang', 'unknown'),
        'original': original,
        'full_whisper': source_obj.get('whisper', ''),
        'teacher_confidence': teacher_conf,
        'avg_logprob': mean_avg_logprob,
        'max_no_speech_prob': max(no_speech) if no_speech else None,
        'max_compression_ratio': max(comp) if comp else None,
        'segment_count': len(group),
    }


def _reject_reason(chunk):
    text = chunk['text']
    lang = chunk['lang']
    duration_s = float(chunk['duration_s'])
    cjk = _is_cjk(lang, text)

    if not text.strip():
        return 'empty'
    if duration_s <= 0 or duration_s > C.max_s2:
        return 'duration'
    if chunk.get('avg_logprob') is not None and chunk['avg_logprob'] < -1.0:
        return 'confidence'
    if chunk.get(
            'max_no_speech_prob') is not None and chunk['max_no_speech_prob'] > 0.6:
        return 'no_speech'
    if chunk.get(
            'max_compression_ratio') is not None and chunk['max_compression_ratio'] > 2.4:
        return 'compression'
    if _max_consecutive_repeats(text, cjk=cjk) >= C.max_repeats:
        return 'repeat'
    if not cjk and _has_long_word(text):
        return 'long_word'
    if not _speech_rate_ok(text, duration_s, cjk=cjk):
        return 'speech_rate'

    target_text = _training_target_text(text)
    token_ids = sp.EncodeAsIds(target_text)
    if not token_ids:
        return 'empty_tokens'
    if max(token_ids) >= BLANK or min(token_ids) < 0:
        return 'token_range'

    mel_len = max(1, int(round(duration_s * C.sr)) // C.hop)
    enc_len = _student_enc_len_from_mel_len(mel_len)
    required = _ctc_required_len(token_ids)
    if required > enc_len:
        return 'ctc_len'

    chunk['token_ids'] = [int(x) for x in token_ids]
    chunk['tok_len'] = len(token_ids)
    chunk['ctc_required_len'] = required
    chunk['mel_len_est'] = mel_len
    chunk['enc_len_est'] = enc_len
    return None


if RUN_CHUNK_FILTER:
    summary_path = os.path.join(
        CHUNK_FILTER_OUT, 'chunk_filter_summary.json')
    all_stats = {}
    # Points to either a freshly filtered writable manifest or the completed
    # read-only artifact mounted from a prior Kaggle session. Balancing below
    # always publishes its selected subset into CHUNK_FILTER_OUT.
    raw_manifest_paths = {}
    if os.path.exists(summary_path):
        try:
            with open(summary_path, 'r', encoding='utf-8') as f:
                prior_stats = json.load(f)
            if isinstance(prior_stats, dict):
                all_stats.update(prior_stats)
                print(
                    f'Loaded {len(prior_stats)} prior filtering summaries; '
                    'processed datasets will be updated in place.')
        except Exception as exc:
            print(f'Could not read prior summary {summary_path}: {exc}')

    for di, (_ds_name, _ds_cfg, _ds_split, _text_col, lang, _) in enumerate(C.datasets):
        safe = C.safe_name(di)
        print(safe)
        src = _select_jsonl_source(
            C.chunk_pseudo_work_dir, C.chunk_pseudo_input_dir, safe)
        print(src)
        dst = os.path.join(CHUNK_FILTER_OUT, f'{safe}.jsonl')
        done_flag = os.path.join(CHUNK_FILTER_OUT, f'{safe}.done')
        input_manifest = os.path.join(C.chunk_filter_input_dir, f'{safe}.jsonl')
        input_done = os.path.join(C.chunk_filter_input_dir, f'{safe}.done')

        # Keep prior filtered MLS chunks immutable and reusable. This prevents
        # duplicate cache rows and avoids re-reading the full pseudo JSONL.
        if (not CHUNK_FILTER_OVERWRITE and os.path.isfile(input_manifest) and
                os.path.isfile(input_done)):
            with open(input_manifest, 'r', encoding='utf-8') as fin:
                kept = sum(1 for line in fin if line.strip())
            all_stats[safe] = {'kept': kept, 'reused_mounted_manifest': True}
            raw_manifest_paths[safe] = input_manifest
            print(f'[{di+1}/{len(C.datasets)}] {safe}: reusing mounted filtered manifest ({kept} chunks)')
            continue

        if CHUNK_FILTER_OVERWRITE:
            for p in [dst, done_flag]:
                if os.path.exists(p):
                    os.remove(p)

        if os.path.exists(done_flag):
            print(f'[{di+1}/{len(C.datasets)}] {safe}: already filtered')
            raw_manifest_paths[safe] = dst
            if safe not in all_stats and os.path.exists(dst):
                # Older runs wrote `.done` before summary merging existed.
                # Recover the most important count without touching the
                # completed manifest or pretending rejection details exist.
                with open(dst, 'r', encoding='utf-8') as fin:
                    kept = sum(1 for line in fin if line.strip())
                all_stats[safe] = {
                    'kept': kept,
                    'summary_reconstructed_from_manifest': True,
                }
            continue
        if not os.path.exists(src):
            print(f'[{di+1}/{len(C.datasets)}] {safe}: missing pseudo file {src}; skip')
            continue

        stats = collections.Counter()
        dst_tmp = dst + '.tmp'
        with open(src, 'r', encoding='utf-8') as fin, open(dst_tmp, 'w', encoding='utf-8') as fout:
            for line in tqdm(fin, desc=f'chunk-filter:{safe}', unit='src'):
                stats['sources'] += 1
                source_obj = json.loads(line)
                original = source_obj.get('original', '')
                full_whisper = source_obj.get('whisper', '')
                if original and full_whisper:
                    if _is_cjk(lang, original + full_whisper):
                        source_score = 100.0 * _cer(original, full_whisper)
                    else:
                        source_score = 100.0 * _wer(original, full_whisper)
                    # Source-level WER/CER is only a diagnostic for chunked data.
                    # The full dataset transcript and Whisper's timestamped pass can
                    # differ in casing, elisions, accents, or partial-book boundaries.
                    # Hard-rejecting the whole source here was wiping every chunk for
                    # French/Spanish; rely on chunk-local confidence, no-speech,
                    # compression, rate, token-range, and CTC-length gates instead.
                    if source_score >= C.wer_threshold:
                        stats['source_high_error'] += 1
                else:
                    source_score = None

                segments = _valid_segments(source_obj)
                if not segments:
                    stats['rejected_no_segments'] += 1
                    continue

                for chunk_idx, group in enumerate(_segment_groups(
                    segments,
                    target_s=C.chunk_seconds,
                    overlap_s=C.chunk_overlap,
                    max_s=C.max_s2,
                )):
                    stats['chunks_total'] += 1
                    chunk = _chunk_from_group(source_obj, group, chunk_idx)
                    chunk['source_error_rate'] = source_score
                    stats[f'label_{chunk["label_source"]}'] += 1
                    reason = _reject_reason(chunk)
                    if reason is not None:
                        stats[f'rejected_{reason}'] += 1
                        continue
                    fout.write(json.dumps(chunk, ensure_ascii=False) + '\n')
                    stats['kept'] += 1

        os.replace(dst_tmp, dst)
        Path(done_flag).touch()
        all_stats[safe] = dict(stats)
        raw_manifest_paths[safe] = dst
        print(
            f'[{di+1}/{len(C.datasets)}] {safe}: kept {stats["kept"]}/{stats["chunks_total"]} chunks')
        rejected = {k: v for k, v in stats.items() if k.startswith('rejected_')}
        print(f'  rejected: {rejected}')


    # ?? Deterministic post-filter language balance ??????????????????????
    # Quality filtering happens first. We then retain an equal number of
    # *unique* chunks per language, bounded by the smallest accepted language
    # (or a lower explicit cap). This avoids both duplicate oversampling and a
    # high-resource language silently dominating the CTC/KD objective.
    if C.enforce_post_filter_language_balance:
        import hashlib

        candidates_by_lang = collections.defaultdict(list)
        seen_chunks = set()
        expected_languages = set()
        for di, (_name, _config, _split, _text_col, language, _verified) in enumerate(C.datasets):
            safe = C.safe_name(di)
            expected_languages.add(str(language))
            manifest = raw_manifest_paths.get(safe)
            if not manifest or not os.path.isfile(manifest):
                raise FileNotFoundError(
                    f'Cannot balance {safe}: no completed filtered manifest. '
                    'Attach the prior filtering dataset or run filtering first.')
            with open(manifest, 'r', encoding='utf-8') as fin:
                for line in fin:
                    if not line.strip():
                        continue
                    chunk = json.loads(line)
                    lang = str(chunk.get('lang', language))
                    if lang != str(language):
                        raise RuntimeError(
                            f'{safe} has unexpected manifest language {lang!r}; '
                            f'expected {language!r}')
                    dedupe_key = (
                        lang,
                        str(chunk.get('source_id', '')),
                        float(chunk.get('chunk_start_s', -1.0)),
                        float(chunk.get('chunk_end_s', -1.0)),
                    )
                    if dedupe_key in seen_chunks:
                        raise RuntimeError(
                            f'Duplicate filtered chunk detected before balancing: {dedupe_key}')
                    seen_chunks.add(dedupe_key)
                    chunk_id = str(chunk.get('chunk_id') or dedupe_key)
                    rank = hashlib.blake2b(
                        f'{C.post_filter_balance_seed}|{safe}|{chunk_id}'.encode('utf-8'),
                        digest_size=16).digest()
                    candidates_by_lang[lang].append((rank, safe, line))

        missing_languages = expected_languages - set(candidates_by_lang)
        if missing_languages:
            raise RuntimeError(
                f'No accepted chunks after filtering for {sorted(missing_languages)}; '
                'do not build a supposedly balanced cache.')
        raw_counts = {lang: len(rows) for lang, rows in sorted(candidates_by_lang.items())}
        target = min(raw_counts.values())
        cap = C.post_filter_chunks_per_language_cap
        if cap is not None:
            cap = int(cap)
            if cap <= 0:
                raise ValueError('post_filter_chunks_per_language_cap must be positive or None')
            target = min(target, cap)
        if target <= 0:
            raise RuntimeError(f'Cannot balance empty filtered counts: {raw_counts}')

        selected_by_safe = collections.defaultdict(list)
        retained_counts = {}
        for lang, rows in candidates_by_lang.items():
            # The hash rank is stable across sessions and independent of stream
            # ordering, so resumed Kaggle runs produce the same selected subset.
            selected = sorted(rows, key=lambda row: row[0])[:target]
            if len(selected) != target:
                raise RuntimeError(f'Balance selection shortfall for {lang}: {len(selected)}/{target}')
            retained_counts[lang] = len(selected)
            for _rank, safe, line in selected:
                selected_by_safe[safe].append(line)

        # Publish only complete balanced manifests. Never alter read-only input
        # datasets; the cache builder will prefer these writable outputs.
        for di in range(len(C.datasets)):
            safe = C.safe_name(di)
            dst = os.path.join(CHUNK_FILTER_OUT, f'{safe}.jsonl')
            done_flag = os.path.join(CHUNK_FILTER_OUT, f'{safe}.done')
            tmp = dst + '.balanced.tmp'
            with open(tmp, 'w', encoding='utf-8') as fout:
                for line in selected_by_safe[safe]:
                    fout.write(line if line.endswith('\n') else line + '\n')
            os.replace(tmp, dst)
            Path(done_flag).touch()
            all_stats.setdefault(safe, {})['balanced_kept'] = len(selected_by_safe[safe])

        balance_summary = {
            'strategy': 'deterministic_equal_chunks_v1',
            'seed': int(C.post_filter_balance_seed),
            'configured_cap_per_language': cap,
            'raw_accepted_counts': raw_counts,
            'target_chunks_per_language': target,
            'retained_counts': retained_counts,
            'duplicate_policy': 'reject',
        }
        summary_tmp = CHUNK_BALANCE_SUMMARY + '.tmp'
        with open(summary_tmp, 'w', encoding='utf-8') as f:
            json.dump(balance_summary, f, ensure_ascii=False, indent=2)
        os.replace(summary_tmp, CHUNK_BALANCE_SUMMARY)
        Path(CHUNK_BALANCE_DONE).touch()
        assert len(set(retained_counts.values())) == 1, retained_counts
        print(f'Balanced filtered chunks: raw={raw_counts}; retained={retained_counts}')
    else:
        print('WARNING: post-filter language balance is disabled; cache may be imbalanced.')

    summary_tmp = summary_path + '.tmp'
    with open(summary_tmp, 'w', encoding='utf-8') as f:
        json.dump(all_stats, f, ensure_ascii=False, indent=2)
    os.replace(summary_tmp, summary_path)
    print(
        f'Chunk filtering summary -> {summary_path} '
        f'({len(all_stats)} dataset(s))')
else:
    print('RUN_CHUNK_FILTER=False; skipping chunk manifest/filtering.')


RUN_CHUNK_FILTER=False; skipping chunk manifest/filtering.


### Combined chunk teacher and mel cache

Replays the source audio once, slices each accepted chunk span, and writes the teacher-cache and mel-cache shards in the same order. This is the cache pair used by the refactored CTC training loader.


In [10]:
# Build aligned chunk-level teacher and mel caches from the filtered chunk manifest.
# The two output trees must be uploaded as Kaggle datasets after this cell finishes:
#   C.chunk_teacher_work_dir -> chunk-teacher-cache/teacher_cache
#   C.chunk_mel_work_dir     -> chunk-mel-cache/mel_cache

from pathlib import Path
from transformers import WhisperForConditionalGeneration, WhisperFeatureExtractor
from tqdm.auto import tqdm

RUN_CHUNK_CACHE_BUILD = False
CHUNK_CACHE_OVERWRITE = False
CHUNK_CACHE_SHARD_SIZE = 128
CHUNK_CACHE_SESSION_HOURS = 10.5

# A stale/partial writable filter must never shadow a completed uploaded
# balanced manifest. The cache preflight below validates the selected tree.
CHUNK_FILTER_IN = _prefer_balanced_filter_dir(
    C.chunk_filter_work_dir, C.chunk_filter_input_dir)
CHUNK_TEACHER_OUT = C.chunk_teacher_work_dir
CHUNK_MEL_OUT = C.chunk_mel_work_dir

# A Kaggle "Run all" may reach this cell after pseudo-label generation stops
# at its session deadline.  Cache construction must never consume a partial or
# unbalanced manifest, so defer it until filtering has published both balance
# artifacts.  This mirrors the filter cell's `.done` preflight and keeps the
# session resumable instead of failing after valid partial upstream work.
if RUN_CHUNK_CACHE_BUILD and C.enforce_post_filter_language_balance:
    _chunk_balance_marker = os.path.join(CHUNK_FILTER_IN, 'chunk_balance.done')
    _chunk_balance_summary = os.path.join(
        CHUNK_FILTER_IN, 'chunk_balance_summary.json')
    if not (os.path.isfile(_chunk_balance_marker) and
            os.path.isfile(_chunk_balance_summary)):
        print(
            'Chunk cache build deferred: balanced manifests are not ready. '
            'Finish pseudo-labeling, run filtering with RUN_CHUNK_FILTER=True, '
            'then rerun this cache cell.')
        RUN_CHUNK_CACHE_BUILD = False

if RUN_CHUNK_CACHE_BUILD:
    if C.enforce_post_filter_language_balance:
        balance_marker = os.path.join(CHUNK_FILTER_IN, 'chunk_balance.done')
        balance_summary = os.path.join(CHUNK_FILTER_IN, 'chunk_balance_summary.json')
        if not (os.path.isfile(balance_marker) and os.path.isfile(balance_summary)):
            raise RuntimeError(
                'Chunk cache build requires balanced manifests. Run the filtering '
                'cell with RUN_CHUNK_FILTER=True, then use its writable balanced output '
                'or attach the uploaded output including chunk_balance.done.')
        with open(balance_summary, 'r', encoding='utf-8') as f:
            balance_state = json.load(f)
        retained = balance_state.get('retained_counts', {})
        expected_languages = {str(row[4]) for row in C.datasets}
        if (set(retained) != expected_languages or not retained or
                len(set(retained.values())) != 1):
            raise RuntimeError(
                f'Invalid language balance summary: retained={retained}, '
                f'expected={sorted(expected_languages)}')
        print(f'Validated equal post-filter counts: {retained}')
    for _output_dir in [CHUNK_TEACHER_OUT, CHUNK_MEL_OUT]:
        if not _is_writable_path(_output_dir):
            raise RuntimeError(f'Chunk cache output must be writable: {_output_dir}')
        os.makedirs(_output_dir, exist_ok=True)
    if 'sp' not in globals() or sp.GetPieceSize() != int(C.spm_vocab):
        sp = spm.SentencePieceProcessor()
        if not sp.Load(C.spm_prefix + '.model'):
            raise FileNotFoundError(f'Tokenizer not found: {C.spm_prefix}.model')
    if sp.GetPieceSize() != int(C.spm_vocab):
        raise RuntimeError(
            f'Chunk cache build requires {C.spm_vocab} SentencePiece tokens, but '
            f'{C.spm_prefix}.model contains {sp.GetPieceSize()}.')
    BLANK = sp.GetPieceSize()
    C.vocab_size = BLANK + 1
    print(f'Chunk manifest input: {CHUNK_FILTER_IN}')


def chunk_quantize_to_int8(arr):
    arr = np.asarray(arr, dtype=np.float32)
    fmin, fmax = float(arr.min()), float(arr.max())
    if fmax - fmin < 1e-8:
        return np.zeros_like(arr, dtype=np.int8), np.float32(1.0), np.float32(0.0)
    scale = np.float32((fmax - fmin) / 254.0)
    zero = np.float32(np.round(-127.0 - (fmin / scale)))
    q = np.clip(np.round(arr / scale + zero), -127, 127).astype(np.int8)
    return q, scale, zero


def _existing_npz_count(lang_dir):
    shards = sorted(f for f in os.listdir(lang_dir) if f.endswith(
        '.npz')) if os.path.isdir(lang_dir) else []
    n = 0
    for sf in shards:
        with np.load(os.path.join(lang_dir, sf), allow_pickle=True) as data:
            n += len(data['scales'])
    return n, shards


def _save_chunk_shard(buf, teacher_dir, mel_dir, shard_idx):
    hidden_chunks, h_scales, h_zeros, h_offsets = [], [], [], [0]
    mel_chunks, m_scales, m_zeros, m_offsets = [], [], [], [0]
    mel_lens, texts, source_ids, split_ids, speaker_ids, langs = [], [], [], [], [], []
    chunk_start_s, chunk_end_s, duration_s, teacher_conf = [], [], [], []
    token_ids_flat, token_offsets, token_lens = [], [0], []

    for item in buf:
        hq, hs, hz = chunk_quantize_to_int8(item['teacher_h'])
        mq, ms, mz = chunk_quantize_to_int8(item['mel'])

        hidden_chunks.append(hq)
        h_scales.append(hs)
        h_zeros.append(hz)
        h_offsets.append(h_offsets[-1] + hq.shape[0])

        mel_chunks.append(mq)
        m_scales.append(ms)
        m_zeros.append(mz)
        m_offsets.append(m_offsets[-1] + mq.shape[0])

        ids = [int(x) for x in item['token_ids']]
        assert ids and max(ids) < BLANK and min(ids) >= 0
        token_ids_flat.extend(ids)
        token_offsets.append(token_offsets[-1] + len(ids))
        token_lens.append(len(ids))

        mel_lens.append(int(item['mel_len']))
        texts.append(item['text'])
        source_ids.append(item['source_id'])
        split_ids.append(item['split_id'])
        speaker_ids.append(item.get('speaker_id') or '')
        langs.append(item['lang'])
        chunk_start_s.append(float(item['chunk_start_s']))
        chunk_end_s.append(float(item['chunk_end_s']))
        duration_s.append(float(item['duration_s']))
        teacher_conf.append(np.nan if item.get('teacher_confidence')
                            is None else float(item['teacher_confidence']))

    teacher_path = os.path.join(teacher_dir, f'shard_{shard_idx:04d}.npz')
    mel_path = os.path.join(mel_dir, f'shard_{shard_idx:04d}.npz')

    teacher_tmp = teacher_path + '.tmp.npz'
    mel_tmp = mel_path + '.tmp.npz'

    np.savez_compressed(
        teacher_tmp,
        hidden_cat=np.concatenate(hidden_chunks, axis=0),
        offsets=np.array(h_offsets, dtype=np.int32),
        scales=np.array(h_scales, dtype=np.float32),
        zeros=np.array(h_zeros, dtype=np.float32),
        mel_lens=np.array(mel_lens, dtype=np.int32),
        texts=np.array(texts, dtype=object),
        token_ids=np.array(token_ids_flat, dtype=np.int32),
        token_offsets=np.array(token_offsets, dtype=np.int32),
        token_lens=np.array(token_lens, dtype=np.int32),
        source_ids=np.array(source_ids, dtype=object),
        split_ids=np.array(split_ids, dtype=object),
        speaker_ids=np.array(speaker_ids, dtype=object),
        chunk_start_s=np.array(chunk_start_s, dtype=np.float32),
        chunk_end_s=np.array(chunk_end_s, dtype=np.float32),
        duration_s=np.array(duration_s, dtype=np.float32),
        langs=np.array(langs, dtype=object),
        teacher_confidence=np.array(teacher_conf, dtype=np.float32),
    )
    np.savez_compressed(
        mel_tmp,
        mel_cat=np.concatenate(mel_chunks, axis=0),
        offsets=np.array(m_offsets, dtype=np.int32),
        scales=np.array(m_scales, dtype=np.float32),
        zeros=np.array(m_zeros, dtype=np.float32),
        mel_lens=np.array(mel_lens, dtype=np.int32),
    )
    # Publish only complete files. If Kaggle stops between these two replaces,
    # the next run promotes the surviving temporary counterpart below.
    os.replace(teacher_tmp, teacher_path)
    os.replace(mel_tmp, mel_path)
    return os.path.getsize(teacher_path), os.path.getsize(mel_path)


def _slice_chunk_audio(raw_audio, start_s, end_s):
    s = max(0, int(round(float(start_s) * C.sr)))
    e = min(len(raw_audio), int(round(float(end_s) * C.sr)))
    if e <= s:
        raise ValueError(
            f'Invalid chunk slice: start={start_s}, end={end_s}, samples={
                len(raw_audio)}')
    return np.asarray(raw_audio[s:e], dtype=np.float32)


def _chunk_teacher_encode(raw_audio):
    inputs = chunk_feat_ext(
        raw_audio,
        sampling_rate=C.sr,
        return_tensors='pt',
        padding='max_length',
        max_length=480000,
    )
    mel_len = max(1, min(len(raw_audio) // C.hop, 3000))
    with torch.inference_mode():
        h = chunk_encoder(
            inputs.input_features.to(chunk_device, dtype=chunk_teacher_dtype)
        ).last_hidden_state
    h_np = h.cpu().float().numpy()[0]
    enc_len = max(1, mel_len // 2)
    return h_np[:enc_len].astype(np.float16), mel_len


chunk_mel_tf = torchaudio.transforms.MelSpectrogram(
    sample_rate=C.sr,
    n_fft=C.n_fft,
    win_length=C.win,
    hop_length=C.hop,
    n_mels=C.n_mels,
    power=2.0,
    pad_mode='constant',
)


def _chunk_mel_sample(raw_audio, expected_mel_len):
    # torchaudio's STFT uses reflect padding when center=True, which requires
    # the waveform to be longer than n_fft//2. Whisper timestamps can produce
    # very short edge chunks, so pad with silence for the transform while keeping
    # mel_len based on the real audio length.
    raw_audio = np.asarray(raw_audio, dtype=np.float32)
    real_samples = len(raw_audio)
    if real_samples < C.n_fft:
        raw_for_mel = np.pad(raw_audio, (0, C.n_fft - real_samples), mode='constant')
    else:
        raw_for_mel = raw_audio

    wav = torch.as_tensor(raw_for_mel, dtype=torch.float32).unsqueeze(0)
    m = chunk_mel_tf(wav)[0]
    m = torch.log(m.clamp(min=1e-10)).transpose(0, 1).contiguous()
    mel_len = max(1, min(real_samples // C.hop, 3000))
    if mel_len != int(expected_mel_len):
        raise ValueError(
            f'mel_len mismatch teacher={expected_mel_len} student={mel_len}')
    m = m[:mel_len]
    if m.shape[0] < mel_len:
        m = torch.cat([m, m.new_zeros(mel_len - m.shape[0], m.shape[1])], 0)
    return m.numpy().astype(np.float16), mel_len


if RUN_CHUNK_CACHE_BUILD:
    _chunk_cache_deadline = time.monotonic() + CHUNK_CACHE_SESSION_HOURS * 3600
    _chunk_cache_stopped = False
    for _name in ['chunk_whisper_model']:
        if _name in globals():
            del globals()[_name]
    flush()

    chunk_device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    chunk_teacher_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    print(
        f'Loading teacher encoder for chunk cache: {
            C.teacher_id} ({chunk_teacher_dtype})')
    chunk_teacher = WhisperForConditionalGeneration.from_pretrained(
        C.teacher_id,
        torch_dtype=chunk_teacher_dtype,
        low_cpu_mem_usage=True,
        use_safetensors=True,
        attn_implementation=C.teacher_attn,
    ).to(chunk_device)
    chunk_teacher.eval()
    chunk_encoder = chunk_teacher.get_encoder()
    chunk_feat_ext = WhisperFeatureExtractor.from_pretrained(C.teacher_id)
    print(
        f'Teacher encoder loaded on {chunk_device}; Whisper mels={
            chunk_feat_ext.feature_size}')

    total_teacher_bytes, total_mel_bytes = 0, 0
    for di, (ds_name, ds_cfg, ds_split, _text_col, lang, _) in enumerate(C.datasets):
        safe = C.safe_name(di)
        manifest = os.path.join(CHUNK_FILTER_IN, f'{safe}.jsonl')
        if not os.path.exists(manifest):
            print(f'[{di+1}/{len(C.datasets)}] {safe}: missing manifest {manifest}; skip')
            continue

        teacher_dir = os.path.join(CHUNK_TEACHER_OUT, safe)
        mel_dir = os.path.join(CHUNK_MEL_OUT, safe)
        input_teacher_dir = os.path.join(C.chunk_teacher_input_dir, safe)
        input_mel_dir = os.path.join(C.chunk_mel_input_dir, safe)
        os.makedirs(teacher_dir, exist_ok=True)
        os.makedirs(mel_dir, exist_ok=True)

        # Cache shards are a paired artifact: resume from an uploaded partial
        # cache only by copying both immutable sides into /kaggle/working.
        # Never merge an unmatched teacher/mel pair or write to /kaggle/input.
        work_teacher_has_shards = _dir_has_suffix(teacher_dir, '.npz')
        work_mel_has_shards = _dir_has_suffix(mel_dir, '.npz')
        input_teacher_has_shards = _dir_has_suffix(input_teacher_dir, '.npz')
        input_mel_has_shards = _dir_has_suffix(input_mel_dir, '.npz')
        if not CHUNK_CACHE_OVERWRITE:
            if not work_teacher_has_shards and not work_mel_has_shards:
                if input_teacher_has_shards != input_mel_has_shards:
                    raise RuntimeError(
                        f'Uploaded chunk cache pair is incomplete for {safe}: '
                        f'teacher={input_teacher_dir}, mel={input_mel_dir}')
                if input_teacher_has_shards:
                    import shutil
                    shutil.copytree(input_teacher_dir, teacher_dir, dirs_exist_ok=True)
                    shutil.copytree(input_mel_dir, mel_dir, dirs_exist_ok=True)
                    print(
                        f'[{di+1}/{len(C.datasets)}] {safe}: copied uploaded '
                        'teacher/mel cache pair to writable resume paths')
            elif work_teacher_has_shards != work_mel_has_shards:
                raise RuntimeError(
                    f'Writable chunk cache pair is incomplete for {safe}: '
                    f'teacher={teacher_dir}, mel={mel_dir}')

        teacher_done = os.path.join(teacher_dir, '.done')
        mel_done = os.path.join(mel_dir, '.done')

        if CHUNK_CACHE_OVERWRITE:
            for root in [teacher_dir, mel_dir]:
                for name in os.listdir(root):
                    p = os.path.join(root, name)
                    if os.path.isfile(p):
                        os.remove(p)

        entries = [json.loads(line) for line in open(manifest, 'r', encoding='utf-8')]
        if not entries:
            print(f'[{di+1}/{len(C.datasets)}] {safe}: empty manifest')
            continue

        if os.path.exists(teacher_done) and os.path.exists(mel_done):
            print(f'[{di+1}/{len(C.datasets)}] {safe}: already cached ({len(entries)} chunks)')
            continue

        # Recover if interruption happened after publishing one side of a pair.
        for shard_name in set(os.listdir(teacher_dir)) | set(os.listdir(mel_dir)):
            if not shard_name.endswith('.npz.tmp.npz'):
                continue
            final_name = shard_name[:-8]
            teacher_tmp = os.path.join(teacher_dir, shard_name)
            mel_tmp = os.path.join(mel_dir, shard_name)
            teacher_final = os.path.join(teacher_dir, final_name)
            mel_final = os.path.join(mel_dir, final_name)
            if os.path.exists(teacher_final) and os.path.exists(
                    mel_tmp) and not os.path.exists(mel_final):
                os.replace(mel_tmp, mel_final)
            if os.path.exists(mel_final) and os.path.exists(
                    teacher_tmp) and not os.path.exists(teacher_final):
                os.replace(teacher_tmp, teacher_final)

        teacher_n, teacher_shards = _existing_npz_count(teacher_dir)
        mel_n, mel_shards = _existing_npz_count(mel_dir)
        if teacher_n != mel_n or len(teacher_shards) != len(mel_shards):
            raise RuntimeError(
                f'Partial chunk cache mismatch for {safe}: '
                f'teacher={teacher_n}/{len(teacher_shards)} mel={mel_n}/{len(mel_shards)}'
            )

        n_done = teacher_n
        shard_idx = len(teacher_shards)
        if n_done >= len(entries):
            Path(teacher_done).touch()
            Path(mel_done).touch()
            print(f'[{di+1}/{len(C.datasets)}] {safe}: all cached ({shard_idx} shards)')
            continue

        needed_by_idx = {}
        for pos, entry in enumerate(entries):
            if pos < n_done:
                continue
            needed_by_idx.setdefault(int(entry['idx']), []).append((pos, entry))
        max_idx = max(needed_by_idx) if needed_by_idx else -1

        ds_stream = load_dataset(
            ds_name, ds_cfg, split=ds_split,
            streaming=True, trust_remote_code=True,
        )
        ds_stream = ds_stream.select_columns(['audio'])
        ds_stream = ds_stream.cast_column('audio', Audio(sampling_rate=C.sr))

        buf = []
        n_written = n_done
        lang_teacher_bytes, lang_mel_bytes = 0, 0
        pbar = tqdm(
            enumerate(ds_stream),
            total=max_idx + 1,
            desc=f'chunk-cache:{safe}',
            unit='src',
        )
        for stream_idx, sample in pbar:
            if time.monotonic() >= _chunk_cache_deadline:
                _chunk_cache_stopped = True
                print('Session deadline reached; flushing the current cache shard.')
                break
            if stream_idx > max_idx:
                break
            if stream_idx not in needed_by_idx:
                continue

            raw_full = np.asarray(sample['audio']['array'], dtype=np.float32)
            for _pos, entry in needed_by_idx[stream_idx]:
                req_start_s = float(entry['chunk_start_s'])
                req_end_s = float(entry['chunk_end_s'])
                source_duration_s = len(raw_full) / C.sr
                eff_start_s = min(max(0.0, req_start_s), source_duration_s)
                eff_end_s = min(max(eff_start_s, req_end_s), source_duration_s)
                chunk_audio = _slice_chunk_audio(raw_full, eff_start_s, eff_end_s)

                expected_samples = max(1, int(round(float(entry['duration_s']) * C.sr)))
                requested_overshoots_source = req_end_s > source_duration_s + \
                    (1.0 / C.sr)
                # A tiny slice for a normal-duration manifest entry usually means
                # the pseudo JSONL was generated from a different dataset order or
                # config than this load_dataset stream. Common Voice/Whisper edge
                # timestamps can run past the actual clip end; those are safe to
                # clamp because the chunk starts in the right source audio.
                if (
                    expected_samples >= C.n_fft
                    and len(chunk_audio) < max(C.n_fft, int(0.90 * expected_samples))
                    and not requested_overshoots_source
                ):
                    raise RuntimeError(
                        f'{safe} pos={_pos} source_id={
                            entry.get("source_id")} audio slice too short: '
                        f'got {
                            len(chunk_audio)} samples, expected ~{expected_samples}; '
                        f'chunk=[{
                            entry["chunk_start_s"]:.3f}, {
                            entry["chunk_end_s"]:.3f}], '
                        f'source_samples={len(raw_full)}. Check that the pseudo JSONL '
                        f'matches ds_name={
                            ds_name!r}, ds_cfg={
                            ds_cfg!r}, split={
                            ds_split!r}.'
                    )
                teacher_h, mel_len = _chunk_teacher_encode(chunk_audio)
                mel, mel_len_2 = _chunk_mel_sample(chunk_audio, mel_len)
                if mel_len != mel_len_2:
                    raise RuntimeError(f'chunk mel mismatch at {safe} pos={_pos}')

                token_ids = [int(x) for x in entry.get('token_ids', [])]
                if not token_ids or max(token_ids) >= BLANK or min(token_ids) < 0:
                    raise ValueError(
                        f'Bad token_ids in {safe} pos={_pos}: {token_ids[:20]}')

                buf.append({
                    'teacher_h': teacher_h,
                    'mel': mel,
                    'mel_len': mel_len,
                    'text': entry['text'],
                    'token_ids': token_ids,
                    'source_id': entry['source_id'],
                    'split_id': entry.get('split_id', entry['source_id']),
                    'speaker_id': entry.get('speaker_id'),
                    'chunk_start_s': round(float(eff_start_s), 3),
                    'chunk_end_s': round(float(eff_end_s), 3),
                    'duration_s': round(float(len(chunk_audio) / C.sr), 3),
                    'lang': entry.get('lang', lang),
                    'teacher_confidence': entry.get('teacher_confidence'),
                })
                n_written += 1

                if len(buf) >= CHUNK_CACHE_SHARD_SIZE:
                    tb, mb = _save_chunk_shard(buf, teacher_dir, mel_dir, shard_idx)
                    total_teacher_bytes += tb
                    total_mel_bytes += mb
                    lang_teacher_bytes += tb
                    lang_mel_bytes += mb
                    shard_idx += 1
                    buf.clear()
                    pbar.set_postfix_str(
                        f'{n_written}/{len(entries)} chunks shard={shard_idx}')

        pbar.close()

        if buf:
            tb, mb = _save_chunk_shard(buf, teacher_dir, mel_dir, shard_idx)
            total_teacher_bytes += tb
            total_mel_bytes += mb
            lang_teacher_bytes += tb
            lang_mel_bytes += mb
            shard_idx += 1
            buf.clear()

        if _chunk_cache_stopped:
            print(
                f'[{di+1}/{len(C.datasets)}] {safe}: partial cache {n_written}/{len(entries)}; rerun to resume')
            break
        if n_written != len(entries):
            raise RuntimeError(
                f'{safe}: cached {n_written}/{len(entries)} chunks; source stream ended early')

        Path(teacher_done).touch()
        Path(mel_done).touch()
        print(
            f'[{di+1}/{len(C.datasets)}] {safe}: {len(entries)} chunks, '
            f'{shard_idx} shards, teacher={lang_teacher_bytes/1e9:.2f}GB, '
            f'mel={lang_mel_bytes/1e9:.2f}GB'
        )

    print(
        f'Chunk teacher cache: {CHUNK_TEACHER_OUT} ({
            total_teacher_bytes /
            1e9:.2f}GB written this run)')
    print(
        f'Chunk mel cache:     {CHUNK_MEL_OUT} ({
            total_mel_bytes /
            1e9:.2f}GB written this run)')
    print('Upload those two folders as Kaggle datasets, then rerun training with C.use_chunk_cache=True.')
else:
    print('RUN_CHUNK_CACHE_BUILD=False; skipping chunk cache build.')


RUN_CHUNK_CACHE_BUILD=False; skipping chunk cache build.


### Chunk NPZ cache sanity check

Fast structural check for the freshly generated NPZ shards before uploading them as Kaggle datasets.


In [11]:
# # Lightweight NPZ sanity check before upload.

# if 'sp' not in globals():
#     sp = spm.SentencePieceProcessor()
#     sp.Load(C.spm_prefix + '.model')

# def _first_npz(root):
#     for lang_dir in sorted(os.listdir(root)):
#         path = os.path.join(root, lang_dir)
#         if not os.path.isdir(path):
#             continue
#         shards = sorted(f for f in os.listdir(path) if f.endswith('.npz'))
#         if shards:
#             return os.path.join(path, shards[0])
#     return None


# _teacher_shard = _first_npz(C.chunk_teacher_work_dir) if os.path.isdir(C.chunk_teacher_work_dir) else None
# if _teacher_shard is None:
#     print(f'No generated chunk teacher shards found yet at {C.chunk_teacher_work_dir}')
# else:
#     _mel_shard = _teacher_shard.replace(C.chunk_teacher_work_dir, C.chunk_mel_work_dir, 1)
#     print(f'Checking teacher shard: {_teacher_shard}')
#     print(f'Checking mel shard:     {_mel_shard}')
#     with np.load(_teacher_shard, allow_pickle=True) as td, np.load(_mel_shard, allow_pickle=True) as md:
#         required_teacher = {
#             'hidden_cat', 'offsets', 'scales', 'zeros', 'mel_lens', 'texts',
#             'token_ids', 'token_offsets', 'source_ids', 'chunk_start_s',
#             'chunk_end_s', 'duration_s', 'langs', 'teacher_confidence',
#         }
#         required_mel = {'mel_cat', 'offsets', 'scales', 'zeros', 'mel_lens'}
#         missing_teacher = sorted(required_teacher - set(td.files))
#         missing_mel = sorted(required_mel - set(md.files))
#         assert not missing_teacher, f'Missing teacher fields: {missing_teacher}'
#         assert not missing_mel, f'Missing mel fields: {missing_mel}'
#         assert np.array_equal(td['mel_lens'], md['mel_lens']), 'teacher/mel mel_lens mismatch'
#         assert len(td['texts']) == len(td['scales']) == len(td['source_ids'])
#         assert len(td['token_offsets']) == len(td['texts']) + 1
#         assert td['token_ids'].size == int(td['token_offsets'][-1])
#         assert td['token_ids'].size == 0 or int(td['token_ids'].max()) < sp.GetPieceSize()
#         n_show = min(3, len(td['texts']))
#         print(f'OK: {len(td["texts"])} chunks in shard, mel_lens aligned.')
#         for i in range(n_show):
#             ts, te = int(td['token_offsets'][i]), int(td['token_offsets'][i + 1])
#             print(
#                 f'  {i}: mel_len={int(td["mel_lens"][i])} '
#                 f'tok_len={te-ts} lang={td["langs"][i]} '
#                 f'source={td["source_ids"][i]} '
#                 f'chunk=({float(td["chunk_start_s"][i]):.2f}, {float(td["chunk_end_s"][i]):.2f}) '
#                 f'text={str(td["texts"][i])[:100]}'
#             )


## Distillation  

## ConMamba encoder modules (vendored, SpeechBrain-free)


In [12]:
# # ConMamba encoder, unidirectional/causal via stock mamba_ssm.Mamba.
# # Vendored from .kdedit/conmamba-src/modules/Conmamba.py; keep the
# # ConMambaStudent fp32 encoder island for selective-scan stability.
# from typing import Optional
# import torch
# import torch.nn as nn
# import torch.nn.functional as F
# from speechbrain.nnet.activations import Swish
# from speechbrain.nnet.attention import PositionalwiseFeedForward
# from speechbrain.nnet.normalization import LayerNorm
# from speechbrain.utils.dynamic_chunk_training import DynChunkTrainConfig

# from mamba_ssm import Mamba   # unidirectional / causal SSM block

# # Fail loudly if the removed bidirectional path is re-enabled.


# class _BiMambaRemoved:
#     def __init__(self, *a, **k):
#         raise RuntimeError(
#             'Bidirectional ConMamba was removed (vendored Vim bimamba dropped for '
#             'stock mamba_ssm). Set Config.cm_bidirectional=False, or restore the '
#             'bimamba / selective_scan_interface blocks to use it.')


# BiMamba = _BiMambaRemoved


# class ConvolutionModule(nn.Module):
#     """This is an implementation of convolution module in Conmamba.
#     """

#     def __init__(
#         self,
#         input_size,
#         kernel_size=31,
#         bias=True,
#         activation=Swish,
#         dropout=0.0,
#         causal=False,
#         dilation=1,
#     ):
#         super().__init__()

#         self.kernel_size = kernel_size
#         self.causal = causal
#         self.dilation = dilation

#         if self.causal:
#             self.padding = (kernel_size - 1) * 2 ** (dilation - 1)
#         else:
#             self.padding = (kernel_size - 1) * 2 ** (dilation - 1) // 2

#         self.layer_norm = nn.LayerNorm(input_size)
#         self.bottleneck = nn.Sequential(
#             # pointwise
#             nn.Conv1d(
#                 input_size, 2 * input_size, kernel_size=1, stride=1, bias=bias
#             ),
#             nn.GLU(dim=1),
#         )
#         # depthwise
#         self.conv = nn.Conv1d(
#             input_size,
#             input_size,
#             kernel_size=kernel_size,
#             stride=1,
#             padding=self.padding,
#             dilation=dilation,
#             groups=input_size,
#             bias=bias,
#         )

#         # BatchNorm in the original Conformer replaced with a LayerNorm due to
#         # https://github.com/speechbrain/speechbrain/pull/1329
#         # see discussion
#         # https://github.com/speechbrain/speechbrain/pull/933#issuecomment-1033367884

#         self.after_conv = nn.Sequential(
#             nn.LayerNorm(input_size),
#             activation(),
#             # pointwise
#             nn.Linear(input_size, input_size, bias=bias),
#             nn.Dropout(dropout),
#         )

#     def forward(
#         self,
#         x: torch.Tensor,
#         mask: Optional[torch.Tensor] = None,
#         dynchunktrain_config: Optional[DynChunkTrainConfig] = None,
#     ):
#         """Applies the convolution to an input tensor `x`.
#         """

#         if dynchunktrain_config is not None:
#             # chances are chunking+causal is unintended; i don't know where it
#             # may make sense, but if it does to you, feel free to implement it.
#             assert (
#                 not self.causal
#             ), "Chunked convolution not supported with causal padding"

#             assert (
#                 self.dilation == 1
#             ), "Current DynChunkTrain logic does not support dilation != 1"

#             # in a causal convolution, which is not the case here, an output
#             # frame would never be able to depend on a input frame from any
#             # point in the future.

#             # but with the dynamic chunk convolution, we instead use a "normal"
#             # convolution but where, for any output frame, the future beyond the
#             # "current" chunk gets masked.
#             # see the paper linked in the documentation for details.

#             chunk_size = dynchunktrain_config.chunk_size
#             batch_size = x.shape[0]

#             # determine the amount of padding we need to insert at the right of
#             # the last chunk so that all chunks end up with the same size.
#             if x.shape[1] % chunk_size != 0:
#                 final_right_padding = chunk_size - (x.shape[1] % chunk_size)
#             else:
#                 final_right_padding = 0

#             # -> [batch_size, t, in_channels]
#             out = self.layer_norm(x)

#             # -> [batch_size, in_channels, t] for the CNN
#             out = out.transpose(1, 2)

#             # -> [batch_size, in_channels, t] (pointwise)
#             out = self.bottleneck(out)

#             # -> [batch_size, in_channels, lc+t+final_right_padding]
#             out = F.pad(out, (self.padding, final_right_padding), value=0)

#             # now, make chunks with left context.
#             # as a recap to what the above padding and this unfold do, consider
#             # each a/b/c letter represents a frame as part of chunks a, b, c.
#             # consider a chunk size of 4 and a kernel size of 5 (padding=2):
#             #
#             # input seq: 00aaaabbbbcc00
#             # chunk #1:  00aaaa
#             # chunk #2:      aabbbb
#             # chunk #3:          bbcc00
#             #
#             # a few remarks here:
#             # - the left padding gets inserted early so that the unfold logic
#             #   works trivially
#             # - the right 0-padding got inserted as the number of time steps
#             #   could not be evenly split in `chunk_size` chunks

#             # -> [batch_size, in_channels, num_chunks, lc+chunk_size]
#             out = out.unfold(2, size=chunk_size + self.padding, step=chunk_size)

#             # as we manually disable padding in the convolution below, we insert
#             # right 0-padding to the chunks, e.g. reusing the above example:
#             #
#             # chunk #1:  00aaaa00
#             # chunk #2:      aabbbb00
#             # chunk #3:          bbcc0000

#             # -> [batch_size, in_channels, num_chunks, lc+chunk_size+rpad]
#             out = F.pad(out, (0, self.padding), value=0)

#             # the transpose+flatten effectively flattens chunks into the batch
#             # dimension to be processed into the time-wise convolution. the
#             # chunks will later on be unflattened.

#             # -> [batch_size, num_chunks, in_channels, lc+chunk_size+rpad]
#             out = out.transpose(1, 2)

#             # -> [batch_size * num_chunks, in_channels, lc+chunk_size+rpad]
#             out = out.flatten(start_dim=0, end_dim=1)

#             # TODO: experiment around reflect padding, which is difficult
#             # because small chunks have too little time steps to reflect from

#             # let's keep backwards compat by pointing at the weights from the
#             # already declared Conv1d.
#             #
#             # still reusing the above example, the convolution will be applied,
#             # with the padding truncated on both ends. the following example
#             # shows the letter corresponding to the input frame on which the
#             # convolution was centered.
#             #
#             # as you can see, the sum of lengths of all chunks is equal to our
#             # input sequence length + `final_right_padding`.
#             #
#             # chunk #1:  aaaa
#             # chunk #2:      bbbb
#             # chunk #3:          cc00

#             # -> [batch_size * num_chunks, out_channels, chunk_size]
#             out = F.conv1d(
#                 out,
#                 weight=self.conv.weight,
#                 bias=self.conv.bias,
#                 stride=self.conv.stride,
#                 padding=0,
#                 dilation=self.conv.dilation,
#                 groups=self.conv.groups,
#             )

#             # -> [batch_size * num_chunks, chunk_size, out_channels]
#             out = out.transpose(1, 2)

#             out = self.after_conv(out)

#             # -> [batch_size, num_chunks, chunk_size, out_channels]
#             out = torch.unflatten(out, dim=0, sizes=(batch_size, -1))

#             # -> [batch_size, t + final_right_padding, out_channels]
#             out = torch.flatten(out, start_dim=1, end_dim=2)

#             # -> [batch_size, t, out_channels]
#             if final_right_padding > 0:
#                 out = out[:, :-final_right_padding, :]
#         else:
#             out = self.layer_norm(x)
#             out = out.transpose(1, 2)
#             out = self.bottleneck(out)
#             out = self.conv(out)

#             if self.causal:
#                 # chomp
#                 out = out[..., : -self.padding]

#             out = out.transpose(1, 2)
#             out = self.after_conv(out)

#         if mask is not None:
#             out.masked_fill_(mask, 0.0)

#         return out


# class ConmambaEncoderLayer(nn.Module):
#     """This is an implementation of Conmamba encoder layer.
#     """

#     def __init__(
#         self,
#         d_model,
#         d_ffn,
#         kernel_size=31,
#         activation=Swish,
#         bias=True,
#         dropout=0.0,
#         causal=False,
#         mamba_config=None
#     ):
#         super().__init__()
#         assert mamba_config is not None

#         bidirectional = mamba_config.pop('bidirectional')
#         if causal or (not bidirectional):
#             self.mamba = Mamba(
#                 d_model=d_model,
#                 **mamba_config
#             )
#         else:
#             self.mamba = BiMamba(
#                 d_model=d_model,
#                 bimamba_type='v2',
#                 **mamba_config
#             )
#         mamba_config['bidirectional'] = bidirectional

#         self.convolution_module = ConvolutionModule(
#             d_model, kernel_size, bias, activation, dropout, causal=causal
#         )

#         self.ffn_module1 = nn.Sequential(
#             nn.LayerNorm(d_model),
#             PositionalwiseFeedForward(
#                 d_ffn=d_ffn,
#                 input_size=d_model,
#                 dropout=dropout,
#                 activation=activation,
#             ),
#             nn.Dropout(dropout),
#         )

#         self.ffn_module2 = nn.Sequential(
#             nn.LayerNorm(d_model),
#             PositionalwiseFeedForward(
#                 d_ffn=d_ffn,
#                 input_size=d_model,
#                 dropout=dropout,
#                 activation=activation,
#             ),
#             nn.Dropout(dropout),
#         )

#         self.norm1 = LayerNorm(d_model)
#         self.norm2 = LayerNorm(d_model)
#         self.drop = nn.Dropout(dropout)

#     def forward(
#         self,
#         x,
#         src_mask: Optional[torch.Tensor] = None,
#         src_key_padding_mask: Optional[torch.Tensor] = None,
#         pos_embs: torch.Tensor = None,
#         dynchunktrain_config: Optional[DynChunkTrainConfig] = None,
#     ):
#         conv_mask: Optional[torch.Tensor] = None
#         if src_key_padding_mask is not None:
#             conv_mask = src_key_padding_mask.unsqueeze(-1)


#         # ffn module
#         x = x + 0.5 * self.ffn_module1(x)
#         # mamba module
#         skip = x
#         x = self.norm1(x)
#         x = self.mamba(x)
#         x = x + skip
#         # convolution module
#         x = x + self.convolution_module(
#             x, conv_mask, dynchunktrain_config=dynchunktrain_config
#         )
#         # ffn module
#         x = self.norm2(x + 0.5 * self.ffn_module2(x))
#         return x


# class ConmambaEncoder(nn.Module):
#     """This class implements the Conmamba encoder.
#     """

#     def __init__(
#         self,
#         num_layers,
#         d_model,
#         d_ffn,
#         kernel_size=31,
#         activation=Swish,
#         bias=True,
#         dropout=0.0,
#         causal=False,
#         mamba_config=None
#     ):
#         super().__init__()
#         print(f'dropout={str(dropout)} is not used in Mamba.')

#         self.layers = torch.nn.ModuleList(
#             [
#                 ConmambaEncoderLayer(
#                     d_model=d_model,
#                     d_ffn=d_ffn,
#                     dropout=dropout,
#                     activation=activation,
#                     kernel_size=kernel_size,
#                     bias=bias,
#                     causal=causal,
#                     mamba_config=mamba_config,
#                 )
#                 for i in range(num_layers)
#             ]
#         )
#         self.norm = LayerNorm(d_model, eps=1e-6)

#     def forward(
#         self,
#         src,
#         src_mask: Optional[torch.Tensor] = None,
#         src_key_padding_mask: Optional[torch.Tensor] = None,
#         pos_embs: Optional[torch.Tensor] = None,
#         dynchunktrain_config: Optional[DynChunkTrainConfig] = None,
#     ):
#         """
#         Arguments
#         ----------
#         src : torch.Tensor
#             The sequence to the encoder layer.
#         src_mask : torch.Tensor, optional
#             The mask for the src sequence.
#         src_key_padding_mask : torch.Tensor, optional
#             The mask for the src keys per batch.
#         pos_embs: torch.Tensor, torch.nn.Module,
#             Module or tensor containing the input sequence positional embeddings
#             If custom pos_embs are given it needs to have the shape (1, 2*S-1, E)
#             where S is the sequence length, and E is the embedding dimension.
#         dynchunktrain_config: Optional[DynChunkTrainConfig]
#             Dynamic Chunk Training configuration object for streaming,
#             specifically involved here to apply Dynamic Chunk Convolution to the
#             convolution module.
#         """

#         output = src
#         for enc_layer in self.layers:
#             output = enc_layer(
#                 output,
#                 src_mask=src_mask,
#                 src_key_padding_mask=src_key_padding_mask,
#                 pos_embs=pos_embs,
#                 dynchunktrain_config=dynchunktrain_config,
#             )
#         output = self.norm(output)

#         return output, None


In [13]:
# import sentencepiece as spm
# from transformers import MambaForCausalLM, MambaConfig
# from tqdm.auto import tqdm
# from torch.utils.data import Dataset, DataLoader
# import torch.nn.functional as F
# import torch.nn as nn
# import torch
# import gc
# import os
# import math
# import time
# import json
# import glob
# import unicodedata
# import collections
# # Free leftovers from the pseudo-labelling / encoder-cache cells.
# for _v in ['t_model', 'enc', 'feat_ext', 'processor', 'proc',
#            'model', 'whisper_model', 'feature_extractor', 'student', 'optimizer']:
#     if _v in dir():
#         try:
#             exec(f'del {_v}')
#         except Exception:
#             pass
# gc.collect()
# if torch.cuda.is_available():
#     torch.cuda.empty_cache()
#     torch.cuda.synchronize()
#     torch.cuda.ipc_collect()


# device = torch.device('cuda')

# # Training-time budget. Start it in the Training process cell, *after* setup
# # and the smoke gate. A one-hour go/no-go must mean one hour of optimizer
# # updates, not ~50 minutes after notebook startup and smoke-test overhead.
# # The wall-clock Kaggle session still bounds the kernel; this is the rolling
# # checkpoint budget owned by one training-process invocation. Eight hours
# # fits Kaggle's session model and lets the still-improving joint stage finish in
# # one continuation; rolling checkpoints retain safe interruption behavior.
# SESSION_HOURS = 8.0
# CKPT_EVERY_MINUTES = 30      # rolling save cadence during training
# SESSION_START = None
# SESSION_DEADLINE = None


# def start_training_budget(reset=False):
#     """Start one explicit training budget after smoke/setup complete."""
#     global SESSION_START, SESSION_DEADLINE
#     if SESSION_START is None or reset:
#         SESSION_START = time.monotonic()
#         SESSION_DEADLINE = SESSION_START + SESSION_HOURS * 3600
#         print(
#             f'Training budget started: {SESSION_HOURS}h; '
#             f'ckpt every {CKPT_EVERY_MINUTES} min')
#     return SESSION_DEADLINE


# def time_left_seconds():
#     # Before the launch cell, expose the full configured budget so helpers
#     # remain safe when inspected manually after the setup cell.
#     if SESSION_DEADLINE is None:
#         return SESSION_HOURS * 3600
#     return max(0.0, SESSION_DEADLINE - time.monotonic())


# def training_elapsed_seconds():
#     if SESSION_START is None:
#         return 0.0
#     return max(0.0, time.monotonic() - SESSION_START)


# def fmt_hms(seconds):
#     seconds = int(seconds)
#     return f'{seconds // 3600:d}h{(seconds % 3600) // 60:02d}m{seconds % 60:02d}s'


# print(
#     f'Training budget configured: {SESSION_HOURS}h, '
#     'starts in the Training process cell after smoke passes')


# # Use absolute C.ckpt_dir so saves do not drift into the kernel CWD.
# os.makedirs(C.ckpt_dir, exist_ok=True)

# # ── Weights & Biases setup: one resumed run across Kaggle sessions.
# WANDB_PROJECT = "edge-asr-distillation"   # ← edit if you want a different project
# _wandb_run_filename = (
#     'wandb_run_scratch.json' if C.train_from_scratch else 'wandb_run.json')
# WANDB_RUN_FILE = os.path.join(C.ckpt_dir, _wandb_run_filename)

# # Install if missing (Kaggle base image usually has it, but be safe)
# try:
#     import wandb
# except ImportError:
#     os.system("pip install -q wandb")
#     import wandb

# # Pull API key from Kaggle Secrets; fall back to offline mode if absent.
# # To set it: Kaggle notebook  : Add-ons  : Secrets  : add WANDB_API_KEY
# if "WANDB_API_KEY" not in os.environ:
#     try:
#         from kaggle_secrets import UserSecretsClient
#         os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("wandb_api_key")
#         print("WANDB_API_KEY loaded from Kaggle Secrets")
#     except Exception as e:
#         print(f"No WANDB_API_KEY found ({type(e).__name__}). Logging in OFFLINE mode.")
#         print(f"To enable online sync: add WANDB_API_KEY in Add-ons  : Secrets, then rerun.")
#         print(f"Or sync afterwards with:  wandb sync /kaggle/working/wandb/<run_dir>")

# # Resume from writable run metadata first, then read-only baseline metadata.
# prior_run_id = None
# _wandb_run_candidates = [WANDB_RUN_FILE]
# for _load_dir in (getattr(C, 'ckpt_load_dirs', None) or
#                   [getattr(C, 'ckpt_load_dir', None)]):
#     if _load_dir:
#         _wandb_run_candidates.append(os.path.join(
#             _load_dir, _wandb_run_filename))

# for _candidate in _wandb_run_candidates:
#     if os.path.exists(_candidate):
#         try:
#             prior_run_id = json.load(open(_candidate))["run_id"]
#             print(f"  ↻ Resuming wandb run {prior_run_id} (from {_candidate})")
#             break
#         except Exception:
#             continue

# wandb_mode = "online" if "WANDB_API_KEY" in os.environ else "offline"
# wandb_run = wandb.init(
#     project=WANDB_PROJECT,
#     id=prior_run_id,
#     resume="allow",
#     mode=wandb_mode,
#     config={k: v for k, v in vars(C).items()
#             if not k.startswith("_") and isinstance(v, (int, float, str, bool, list, tuple))},
#     settings=wandb.Settings(start_method="thread"),
# )
# if prior_run_id is None:
#     json.dump({"run_id": wandb_run.id}, open(WANDB_RUN_FILE, "w"))
#     print(f"Started new wandb run: {wandb_run.id} (mode={wandb_mode})")
# else:
#     print(f"Continuing wandb run: {wandb_run.id} (mode={wandb_mode})")
# print(f"  URL: {wandb_run.get_url() or '(offline — sync after training)'}")


# # ── Local JSONL loss logger: append-only backup of wandb metrics.
# class LossLogger:
#     """ Thin append-only JSONL writer keyed by wandb run + global wall clock."""

#     def __init__(self, path, run_id):
#         self.path = path
#         self.run_id = run_id
#         os.makedirs(os.path.dirname(path), exist_ok=True)
#         # Open in append mode so resumes preserve history.
#         # line_buffering=True flushes on every '\n' (each record is one line).
#         self._fh = open(path, 'a', buffering=1, encoding='utf-8')
#         n_existing = 0
#         if os.path.exists(path):
#             with open(path, 'r', encoding='utf-8') as f:
#                 for _ in f:
#                     n_existing += 1
#         self._n_records = n_existing
#         # Header record on session start — lets you separate sessions in post-hoc
#         # analysis.
#         self._fh.write(json.dumps({
#             'type': 'session_start',
#             'ts': time.time(),
#             'run_id': run_id,
#             'session_start_iso': time.strftime('%Y-%m-%dT%H:%M:%S'),
#         }) + '\n')

#     def log(self, record_type, **fields):
#         rec = {'type': record_type, 'ts': time.time(), 'run_id': self.run_id}
#         rec.update(fields)
#         self._fh.write(json.dumps(rec, default=str) + '\n')
#         self._n_records += 1

#     def close(self):
#         try:
#             self._fh.flush()
#             os.fsync(self._fh.fileno())
#         except Exception:
#             pass
#         self._fh.close()


# loss_log = LossLogger(C.loss_log_path, wandb_run.id)
# print(f"Loss log: {C.loss_log_path} ({loss_log._n_records - 1} prior records)")

# # ── T4 16 GB overrides: use the measured memory headroom for throughput.
# _T4_OVERRIDES = dict(
#     # The real bs=8/no-checkpointing smoke peaked at 1.71 GB on GPU 0. Scaling
#     # the micro-batch to the full effective batch is comfortably below 16 GB
#     # and removes four DataParallel launches per optimizer update. The mandatory
#     # smoke test remains the OOM gate for future architecture/cache changes.
#     fr_bs=32,
#     fr_ga=1,
#     un_bs=32,
#     un_ga=1,
#     max_s1=6.0,  # safety metadata; chunk CTC itself is never cropped
#     max_s2=8.0,
# )
# print('T4 overrides applied:')
# for _k, _v in _T4_OVERRIDES.items():
#     _old = getattr(C, _k)
#     setattr(C, _k, _v)
#     print(f'  C.{_k:8s} {_old!r:>8}  ->  {_v!r}')
# print(f'  effective batch (frozen)   = {C.fr_bs * C.fr_ga}')
# print(f'  effective batch (unfrozen) = {C.un_bs * C.un_ga}')


# # Decode + WER helpers (for validation logging)
# # Lightweight install of jiwer for clean WER. Falls back to a manual
# # Levenshtein implementation if jiwer can't be installed.
# try:
#     from jiwer import wer as _jiwer_wer
#     _HAS_JIWER = True
# except ImportError:
#     try:
#         os.system("pip install -q jiwer")
#         from jiwer import wer as _jiwer_wer
#         _HAS_JIWER = True
#     except Exception:
#         _HAS_JIWER = False
#         print("    jiwer unavailable, falling back to manual WER computation")


# def greedy_ctc_token_ids(ctc_logits, blank_id, lengths=None):
#     """Greedy CTC argmax -> collapse repeats -> drop blanks; returns IDs."""
#     preds = ctc_logits.argmax(dim=-1)  # [B, T]
#     out = []
#     for i, p in enumerate(preds):
#         seq = p[:lengths[i]].tolist() if lengths is not None else p.tolist()
#         cleaned, prev = [], -1
#         for tok in seq:
#             tok = int(tok)
#             if tok != prev and tok != blank_id:
#                 cleaned.append(tok)
#             prev = tok
#         out.append(cleaned)
#     return out


# def greedy_ctc_decode(ctc_logits, sp_model, blank_id, lengths=None):
#     """Greedy CTC decode: argmax -> collapse repeats -> drop blanks -> detokenize."""
#     return [sp_model.DecodeIds(ids) for ids in greedy_ctc_token_ids(
#         ctc_logits, blank_id, lengths)]


# def _flatten_id_sequences(seqs):
#     flat = []
#     for seq in seqs or []:
#         if isinstance(seq, torch.Tensor):
#             seq = seq.detach().cpu().tolist()
#         if isinstance(seq, np.ndarray):
#             seq = seq.tolist()
#         if isinstance(seq, (int, np.integer)):
#             seq = [int(seq)]
#         flat.extend(int(x) for x in seq)
#     return flat


# def compute_collapse_metrics(pred_ids, blank_id, frame_argmax_ids=None,
#                              refs=None, hyps=None, sp_model=None,
#                              blank_probability_pct=None,
#                              empty_threshold=0.95,
#                              blank_threshold=0.98,
#                              blank_probability_threshold=0.98,
#                              top_token_threshold=0.90):
#     """Summarize CTC collapse from decoded IDs and raw frame predictions.

#     This function inspects one model snapshot. Early CTC commonly has all-blank
#     greedy output while its loss is falling, especially with a 5k-way head.
#     Therefore blank collapse requires both near-total blank argmax *and* near-
#     total blank probability mass. The training canary separately compares loss
#     against a pre-training baseline before deciding that progress is terminal.
#     """
#     pred_lens = [len(seq) for seq in (pred_ids or [])]
#     nonblank = [int(x) for seq in (pred_ids or []) for x in seq if int(x) != blank_id]
#     frames = _flatten_id_sequences(frame_argmax_ids)
#     hyps = hyps or []
#     refs = refs or []

#     empty_pct = float(np.mean([len(str(h).strip()) == 0 for h in hyps])) if hyps else 0.0
#     avg_pred_len = float(np.mean(pred_lens)) if pred_lens else 0.0
#     if refs and sp_model is not None:
#         avg_ref_len = float(np.mean([len(sp_model.EncodeAsIds(str(r))) for r in refs]))
#     else:
#         avg_ref_len = 0.0

#     top_id = None
#     top_ratio = 0.0
#     if nonblank:
#         vals, counts = np.unique(nonblank, return_counts=True)
#         top_idx = int(np.argmax(counts))
#         top_id = int(vals[top_idx])
#         top_ratio = float(counts[top_idx] / len(nonblank))

#     blank_frame_pct = 0.0
#     frame_top_id = None
#     frame_top_ratio = 0.0
#     if frames:
#         blank_frame_pct = float(sum(1 for x in frames if x == blank_id) / len(frames))
#         vals, counts = np.unique(frames, return_counts=True)
#         top_idx = int(np.argmax(counts))
#         frame_top_id = int(vals[top_idx])
#         frame_top_ratio = float(counts[top_idx] / len(frames))

#     blank_probability_pct = (
#         None if blank_probability_pct is None else float(blank_probability_pct))
#     blank_argmax_dominant = blank_frame_pct >= blank_threshold
#     hard_blank_collapse = (
#         blank_argmax_dominant and blank_probability_pct is not None and
#         blank_probability_pct >= blank_probability_threshold)
#     empty_blank_warning = (
#         empty_pct >= empty_threshold and blank_argmax_dominant)
#     # Detect non-blank mode collapse from raw frames. A post-CTC sequence with
#     # one residual token per utterance is not non-blank collapse when nearly
#     # every frame actually predicts BLANK.
#     nonblank_frame_mode = (
#         frame_top_id is not None and frame_top_id != blank_id and
#         frame_top_ratio >= top_token_threshold)

#     reason = 'ok'
#     collapsed = False
#     if hard_blank_collapse:
#         collapsed, reason = True, 'blank_dominant'
#     elif nonblank_frame_mode:
#         collapsed, reason = True, 'nonblank_mode'
#     elif empty_blank_warning:
#         # Important warning, but not terminal without probability/loss evidence.
#         reason = 'empty_blank_dominant'
#     elif blank_argmax_dominant:
#         reason = 'argmax_blank_dominant'
#     elif empty_pct >= empty_threshold and not frames:
#         # Preserve the fallback when raw-frame evidence is unavailable.
#         collapsed, reason = True, 'empty_hypothesis'

#     return {
#         'collapsed': bool(collapsed),
#         'reason': reason,
#         'blank_frame_pct': blank_frame_pct,
#         'blank_probability_pct': blank_probability_pct,
#         'empty_hypothesis_pct': empty_pct,
#         'empty_blank_warning': bool(empty_blank_warning),
#         'top_id': top_id,
#         'top_ratio': top_ratio,
#         'nonblank_count': int(len(nonblank)),
#         'frame_top_id': frame_top_id,
#         'frame_top_ratio': frame_top_ratio,
#         'frame_count': int(len(frames)),
#         'avg_pred_token_len': avg_pred_len,
#         'avg_ref_token_len': avg_ref_len,
#     }


# def detect_ctc_collapse(pred_ids, blank_id, threshold=0.80):
#     """Backward-compatible wrapper for older diagnostic calls."""
#     return compute_collapse_metrics(
#         pred_ids, blank_id, top_token_threshold=threshold)


# def _manual_wer(refs, hyps):
#     """Word-level Levenshtein distance, summed over the batch / total ref words."""
#     def edit_dist(r, h):
#         # Standard DP; r, h are lists of words
#         if not r:
#             return len(h)
#         if not h:
#             return len(r)
#         prev = list(range(len(h) + 1))
#         for i, rw in enumerate(r, 1):
#             curr = [i] + [0] * len(h)
#             for j, hw in enumerate(h, 1):
#                 cost = 0 if rw == hw else 1
#                 curr[j] = min(curr[j-1] + 1,           # insertion
#                               prev[j] + 1,             # deletion
#                               prev[j-1] + cost)        # substitution / match
#             prev = curr
#         return prev[-1]

#     total_words = total_err = 0
#     for r, h in zip(refs, hyps):
#         rw, hw = r.split(), h.split()
#         total_words += len(rw)
#         total_err += edit_dist(rw, hw)
#     return total_err / max(total_words, 1)


# def compute_wer(refs, hyps):
#     if not refs:
#         return 0.0
#     if _HAS_JIWER:
#         # jiwer raises on empty hyp lists; protect just in case
#         try:
#             return float(_jiwer_wer(refs, hyps))
#         except Exception:
#             return _manual_wer(refs, hyps)
#     return _manual_wer(refs, hyps)


# def _manual_cer(refs, hyps):
#     """Char-level Levenshtein, summed over the batch / total ref chars.
#     Mirrors _manual_wer but over characters (no whitespace split) - the
#     meaningful metric for space-free scripts (zh/ja)."""
#     def edit_dist(r, h):
#         if not r:
#             return len(h)
#         if not h:
#             return len(r)
#         prev = list(range(len(h) + 1))
#         for i, rc in enumerate(r, 1):
#             curr = [i] + [0] * len(h)
#             for j, hc in enumerate(h, 1):
#                 cost = 0 if rc == hc else 1
#                 curr[j] = min(curr[j-1] + 1, prev[j] + 1, prev[j-1] + cost)
#             prev = curr
#         return prev[-1]

#     total_chars = total_err = 0
#     for r, h in zip(refs, hyps):
#         total_chars += len(r)
#         total_err += edit_dist(r, h)
#     return total_err / max(total_chars, 1)


# def compute_cer(refs, hyps):
#     if not refs:
#         return 0.0
#     if _HAS_JIWER:
#         try:
#             import jiwer
#             return float(jiwer.cer(refs, hyps))
#         except Exception:
#             return _manual_cer(refs, hyps)
#     return _manual_cer(refs, hyps)


# def normalize_asr_text(text):
#     """Unicode/case/punctuation normalization for model selection.

#     Raw metrics are still reported. Normalized CER is the primary multilingual
#     checkpoint metric because whitespace-token WER is not comparable across
#     Latin, Chinese, and Japanese scripts.
#     """
#     text = unicodedata.normalize('NFKC', str(text)).casefold()
#     text = ''.join(
#         ' ' if unicodedata.category(ch).startswith('P') else ch
#         for ch in text
#     )
#     return ' '.join(text.split())


# _LANGUAGE_ALIASES = {
#     'en': 'english', 'english': 'english',
#     'fr': 'french', 'french': 'french',
#     'es': 'spanish', 'spanish': 'spanish',
#     'de': 'german', 'german': 'german',
#     'vi': 'vietnamese', 'vietnamese': 'vietnamese',
#     'ja': 'japanese', 'japanese': 'japanese',
#     'ko': 'korean', 'korean': 'korean',
#     'zh': 'chinese', 'zh-cn': 'chinese', 'chinese': 'chinese',
# }


# def canonical_language_name(lang):
#     raw = str(lang or 'unknown').strip().casefold().replace('_', '-')
#     return _LANGUAGE_ALIASES.get(raw, raw)


# def normalize_ctc_target_text(text):
#     """Apply the explicitly versioned CTC target recipe."""
#     mode = getattr(C, 'ctc_target_normalization', 'none')
#     if mode == 'none':
#         return str(text or '').strip()
#     if mode == 'metric':
#         return normalize_asr_text(text)
#     raise ValueError(f'Unsupported ctc_target_normalization={mode!r}')


# def normalized_metric_texts(refs, hyps):
#     return ([normalize_asr_text(x) for x in refs],
#             [normalize_asr_text(x) for x in hyps])


# def per_language_metrics(refs, hyps, langs):
#     """Return normalized and raw WER/CER for each language."""
#     groups = {}
#     for r, h, lang in zip(refs, hyps, langs):
#         groups.setdefault(lang, ([], []))
#         groups[lang][0].append(r)
#         groups[lang][1].append(h)
#     out = {}
#     for lang, (rs, hs) in sorted(groups.items()):
#         norm_rs, norm_hs = normalized_metric_texts(rs, hs)
#         out[lang] = {
#             'wer': compute_wer(norm_rs, norm_hs),
#             'cer': compute_cer(norm_rs, norm_hs),
#             'wer_raw': compute_wer(rs, hs),
#             'cer_raw': compute_cer(rs, hs),
#             'n': len(rs),
#         }
#     return out


# def macro_language_cer(per_lang, min_samples=20):
#     """Equal-language normalized CER over languages with enough support.

#     Sample-weighted CER let the three largest languages dominate checkpoint
#     selection. The support floor prevents a two-example language from making
#     the metric noisy while the sampler still trains on every language.
#     """
#     eligible = {
#         lang: metrics for lang, metrics in per_lang.items()
#         if int(metrics.get('n', 0)) >= int(min_samples)
#     }
#     if not eligible:
#         return float('inf'), []
#     return (
#         float(np.mean([metrics['cer'] for metrics in eligible.values()])),
#         sorted(eligible),
#     )


# # ── 2. Load or rebuild SentencePiece tokenizer ──────────────────
# def _clean_text_scalar(v):
#     if isinstance(v, np.ndarray) and v.shape == ():
#         v = v.item()
#     if isinstance(v, bytes):
#         v = v.decode('utf-8')
#     if isinstance(v, np.generic):
#         v = v.item()
#     return str(v)


# def _iter_cache_text_records(cache_dir):
#     """Yield (canonical_language, text) without trusting pickle scalars."""
#     for shard in sorted(glob.glob(os.path.join(cache_dir, '*', 'shard_*.npz'))):
#         fallback_lang = os.path.basename(os.path.dirname(shard))
#         with np.load(shard, allow_pickle=True) as data:
#             text_arr = data['texts'] if 'texts' in data.files else (
#                 data['text'] if 'text' in data.files else None)
#             lang_arr = data['langs'] if 'langs' in data.files else (
#                 data['lang'] if 'lang' in data.files else None)
#             if text_arr is None:
#                 continue
#             for row_idx, text in enumerate(text_arr):
#                 text = _clean_text_scalar(text).strip()
#                 if not text:
#                     continue
#                 lang = fallback_lang
#                 if lang_arr is not None:
#                     try:
#                         lang = _clean_text_scalar(lang_arr[row_idx])
#                     except Exception:
#                         pass
#                 yield canonical_language_name(lang), text


# def _balanced_tokenizer_texts(cache_dir):
#     """Deterministically temper language imbalance for tokenizer training.

#     A raw concatenation assigned most pieces to Han characters because Chinese
#     dominated the corpus. Temperature sampling preserves high-resource
#     coverage while repeating small languages less aggressively than full
#     equalization.
#     """
#     buckets = collections.defaultdict(list)
#     for lang, text in _iter_cache_text_records(cache_dir):
#         target = normalize_ctc_target_text(text)
#         if target:
#             buckets[lang].append(target.replace('\n', ' '))
#     if not buckets:
#         return [], {}

#     raw_counts = {lang: len(rows) for lang, rows in sorted(buckets.items())}
#     if not getattr(C, 'spm_balance_languages', True):
#         rows = [
#             text for lang in sorted(buckets) for text in buckets[lang]]
#         return rows, raw_counts

#     temperature = float(getattr(C, 'spm_language_temperature', 0.5))
#     if not 0.0 <= temperature <= 1.0:
#         raise ValueError('spm_language_temperature must be in [0, 1]')
#     cap = max(1, int(getattr(
#         C, 'spm_max_sentences_per_language', 30_000)))
#     max_count = max(raw_counts.values())
#     reference = min(max_count, cap)
#     balanced = []
#     sampled_counts = {}
#     for lang_idx, lang in enumerate(sorted(buckets)):
#         source_rows = buckets[lang]
#         ratio = len(source_rows) / max(max_count, 1)
#         target_n = min(
#             cap,
#             max(1, int(round(reference * (ratio ** temperature)))))
#         rng = random.Random(C.seed * 1_000_003 + lang_idx)
#         order = list(range(len(source_rows)))
#         selected = []
#         while len(selected) < target_n:
#             rng.shuffle(order)
#             selected.extend(order[:target_n - len(selected)])
#         balanced.extend(source_rows[i] for i in selected)
#         sampled_counts[lang] = target_n
#     print(f'Tokenizer raw language counts: {raw_counts}')
#     print(
#         f'Tokenizer sampled counts (temperature={temperature}): '
#         f'{sampled_counts}')
#     return balanced, sampled_counts


# def _build_sentencepiece_from_cache(cache_dir, prefix):
#     os.makedirs(os.path.dirname(prefix), exist_ok=True)
#     corpus_path = prefix + '_corpus.txt'
#     texts, sampled_counts = _balanced_tokenizer_texts(cache_dir)
#     if not texts:
#         raise RuntimeError(f'No cached texts found under {cache_dir}; cannot build tokenizer.')
#     with open(corpus_path, 'w', encoding='utf-8') as f:
#         for text in texts:
#             f.write(text + '\n')
#     print(
#         f'Building {C.spm_vocab}-piece SentencePiece tokenizer from '
#         f'{len(texts):,} language-tempered texts...')
#     spm.SentencePieceTrainer.Train(
#         input=corpus_path,
#         model_prefix=prefix,
#         vocab_size=int(C.spm_vocab),
#         character_coverage=float(C.spm_coverage),
#         model_type='unigram',
#         input_sentence_size=2000000,
#         # Corpus sampling and traversal are deterministic, so disabling the
#         # trainer's shuffle makes rebuilds stable across Kaggle sessions.
#         shuffle_input_sentence=False,
#         num_threads=1,
#         hard_vocab_limit=False,
#     )
#     with open(prefix + '_corpus_summary.json', 'w', encoding='utf-8') as f:
#         json.dump({
#             'target_normalization': C.ctc_target_normalization,
#             'sampled_counts': sampled_counts,
#             'n_sentences': len(texts),
#         }, f, ensure_ascii=False, indent=2)


# def _load_sentencepiece_for_training():
#     requested = int(getattr(C, 'spm_vocab', 0) or 0)
#     work_prefix = os.path.join(C.root, 'tokenizer', f'spm_{requested}')
#     if getattr(C, 'force_tokenizer_rebuild', False):
#         if C.ctc_target_normalization == 'none':
#             print(
#                 'WARNING: rebuilding a tokenizer with unnormalized CTC '
#                 'targets. For the recommended clean lineage, set '
#                 "C.ctc_target_normalization='metric'.")
#         print(f'Forcing tokenizer rebuild at {work_prefix}.model')
#         _build_sentencepiece_from_cache(C.cache_dir, work_prefix)
#         cand = spm.SentencePieceProcessor()
#         cand.Load(work_prefix + '.model')
#         C.spm_prefix = work_prefix
#         return cand

#     checkpoint_prefixes = []
#     for load_dir in (getattr(C, 'ckpt_load_dirs', None) or
#                      [getattr(C, 'ckpt_load_dir', None)]):
#         if load_dir:
#             checkpoint_prefixes.append(os.path.join(
#                 os.path.dirname(load_dir), 'tokenizer', f'spm_{requested}'))
#     candidates = [work_prefix, *checkpoint_prefixes, C.spm_prefix]
#     seen = set()
#     mismatches = []
#     for prefix in candidates:
#         if prefix in seen:
#             continue
#         seen.add(prefix)
#         model_path = prefix + '.model'
#         if not os.path.exists(model_path):
#             continue
#         cand = spm.SentencePieceProcessor()
#         cand.Load(model_path)
#         size = cand.GetPieceSize()
#         if requested <= 0 or size == requested:
#             C.spm_prefix = prefix
#             return cand
#         mismatches.append((prefix, size))

#     if not getattr(C, 'allow_tokenizer_rebuild', True):
#         raise RuntimeError(
#             f'No tokenizer with {requested} pieces found. Mismatches={mismatches}')

#     print(f'No matching {requested}-piece tokenizer found; mismatches={mismatches}.')
#     _build_sentencepiece_from_cache(C.cache_dir, work_prefix)
#     cand = spm.SentencePieceProcessor()
#     cand.Load(work_prefix + '.model')
#     C.spm_prefix = work_prefix
#     return cand


# sp = _load_sentencepiece_for_training()
# BLANK = sp.GetPieceSize()  # CTC blank = last index
# C.vocab_size = BLANK + 1   # SP pieces + 1 CTC blank
# assert BLANK == sp.GetPieceSize(), 'BLANK should be one past SPM IDs'
# assert C.vocab_size == sp.GetPieceSize() + 1, 'ctc_head must output (vocab + 1) classes'
# print(f'Tokenizer: {sp.GetPieceSize()} tokens + blank={BLANK}, vocab_size={C.vocab_size}')
# print(f'  tokenizer_prefix={C.spm_prefix}')

# # A same-shaped CTC head is still incompatible when token IDs map to
# # different pieces. Persist and validate this fingerprint in checkpoints.
# def _tokenizer_fingerprint(sp_model):
#     import hashlib
#     h = hashlib.sha256()
#     h.update(f'pieces={sp_model.GetPieceSize()}\n'.encode('utf-8'))
#     for i in range(sp_model.GetPieceSize()):
#         piece = sp_model.IdToPiece(i).encode('utf-8')
#         h.update(len(piece).to_bytes(4, 'little'))
#         h.update(piece)
#     return h.hexdigest()

# TOKENIZER_FINGERPRINT = _tokenizer_fingerprint(sp)
# print(f'  tokenizer_sha256={TOKENIZER_FINGERPRINT[:16]}...')


# def _target_recipe_fingerprint():
#     import hashlib
#     payload = {
#         'ctc_target_normalization': C.ctc_target_normalization,
#         'version': 1,
#     }
#     raw = json.dumps(payload, sort_keys=True, separators=(',', ':'))
#     return hashlib.sha256(raw.encode('utf-8')).hexdigest(), payload


# TARGET_RECIPE_FINGERPRINT, TARGET_RECIPE = _target_recipe_fingerprint()
# print(
#     f'  target_recipe={TARGET_RECIPE}, '
#     f'sha256={TARGET_RECIPE_FINGERPRINT[:16]}...')


# def _require_checkpoint_target_recipe(state, path):
#     found = state.get('target_recipe_fingerprint')
#     if found is None:
#         # All supplied checkpoints predate recipe fingerprints and used raw
#         # cached text. Accept only that exact legacy behavior.
#         if C.ctc_target_normalization == 'none':
#             print(
#                 f'WARNING: {path} predates target-recipe fingerprints; '
#                 'accepting as legacy normalization="none".')
#             return False
#         raise RuntimeError(
#             f'Checkpoint {path} predates target-recipe fingerprints and '
#             f'cannot resume ctc_target_normalization='
#             f'{C.ctc_target_normalization!r}. Start a clean joint stage.')
#     if found != TARGET_RECIPE_FINGERPRINT:
#         raise RuntimeError(
#             f'CTC target recipe mismatch for {path}: '
#             f'checkpoint={found[:16]}..., '
#             f'active={TARGET_RECIPE_FINGERPRINT[:16]}... ({TARGET_RECIPE}).')
#     return True


# def _require_checkpoint_tokenizer(state, path, allow_missing=False):
#     found = state.get('tokenizer_fingerprint')
#     if found is None:
#         message = (
#             f'Checkpoint {path} has no tokenizer fingerprint. Its CTC head '
#             'cannot be proven compatible with the active tokenizer.')
#         if allow_missing:
#             print(f'WARNING: {message}')
#             return False
#         raise RuntimeError(message)
#     if found != TOKENIZER_FINGERPRINT:
#         raise RuntimeError(
#             f'Tokenizer/checkpoint mismatch for {path}: '
#             f'checkpoint={found[:16]}..., active={TOKENIZER_FINGERPRINT[:16]}.... '
#             'Load the tokenizer saved with that checkpoint or reset the CTC head.')
#     return True


# # ── 3. Dataset (lazy-loaded int8 ragged shards) ────────────────
# class KDDataset(Dataset):
#     def __init__(self, cache_dir, sp_model, mel_dir=None):
#         if not os.path.isdir(cache_dir):
#             raise FileNotFoundError(
#                 f'Teacher cache directory not found: {cache_dir}. Attach the '
#                 'configured Kaggle dataset or update C.cache_dir.')
#         if mel_dir is not None and not os.path.isdir(mel_dir):
#             raise FileNotFoundError(
#                 f'Mel cache directory not found: {mel_dir}. Attach the companion '
#                 'mel-cache dataset or update C.mel_dir.')
#         self.sp = sp_model
#         self.cache_dir = os.path.normpath(cache_dir)
#         self.mel_dir = os.path.normpath(mel_dir) if mel_dir is not None else None
#         self.index = []
#         # Parallel to `index`: all chunks from one recording share a group.
#         # Persisted splits use this to keep overlapping audio out of val/test.
#         self.group_ids = []
#         # Parallel semantic language labels let the training sampler balance
#         # languages without opening NPZ shards again.
#         self.sample_languages = []

#         shard_paths = []
#         for lang_dir in sorted(os.listdir(cache_dir)):
#             lang_path = os.path.join(cache_dir, lang_dir)
#             if not os.path.isdir(lang_path):
#                 continue
#             shards = sorted(f for f in os.listdir(lang_path) if f.endswith('.npz'))
#             for sf in shards:
#                 shard_paths.append(os.path.join(lang_path, sf))
#             print(f'  {lang_dir}: {len(shards)} shards')

#         print(f'Indexing {len(shard_paths)} shards...')
#         skipped_ctc = 0
#         skipped_token = 0
#         retokenized = 0
#         for shard in tqdm(shard_paths, desc='Indexing', unit='shard'):
#             data = np.load(shard, allow_pickle=True)
#             n = len(data['scales'])
#             text_arr = self._optional_npz(data, 'texts', 'text')
#             source_arr = self._optional_npz(data, 'source_ids', 'source_id')
#             split_arr = self._optional_npz(data, 'split_ids', 'split_id')
#             lang_arr = self._optional_npz(data, 'langs', 'lang')
#             for j in range(n):
#                 text = str(self._optional_value(text_arr, j, ''))
#                 cached_ids = None if getattr(C, 'force_retokenize_targets', True) else (
#                     self._token_ids_from_cache(data, j))
#                 token_ids, did_retokenize = self._validated_target_ids(cached_ids, text)
#                 if did_retokenize:
#                     retokenized += 1
#                 if not token_ids:
#                     skipped_token += 1
#                     continue

#                 mel_len = int(data['mel_lens'][j])
#                 enc_len = max(1, mel_len)
#                 k, p, stride = C.cnn_ks, C.cnn_ks // 2, 2
#                 for _ in range(2):
#                     enc_len = ((enc_len + 2 * p - k) // stride) + 1
#                 enc_len = max(1, enc_len)
#                 repeats = sum(
#                     1 for a, b in zip(token_ids, token_ids[1:]) if a == b)
#                 if len(token_ids) + repeats > enc_len:
#                     skipped_ctc += 1
#                     continue
#                 source_id = self._optional_value(source_arr, j, None)
#                 split_id = self._optional_value(split_arr, j, source_id)
#                 if split_id is None or not str(split_id).strip():
#                     # Missing provenance must not merge unrelated examples.
#                     rel = '/'.join(shard.replace('\\', '/').rsplit('/', 2)[-2:])
#                     group_id = f'{rel}:{j}'
#                 else:
#                     lang_dir = os.path.basename(os.path.dirname(shard))
#                     group_id = f'{lang_dir}|{split_id}'
#                 sample_lang = self._optional_value(
#                     lang_arr, j, os.path.basename(os.path.dirname(shard)))
#                 self.index.append((shard, j))
#                 self.group_ids.append(group_id)
#                 self.sample_languages.append(str(sample_lang))
#             del data
#         print(f'Indexed {len(self.index)} samples')
#         if skipped_ctc or skipped_token:
#             print(
#                 f'Skipped {skipped_ctc} CTC-invalid and '
#                 f'{skipped_token} token-invalid samples during indexing')
#         if retokenized:
#             print(f'Re-encoded {retokenized} sample(s) from cached text for the active tokenizer')

#         self._cache, self._cache_order = {}, []
#         self._max_cache = 4

#     @staticmethod
#     def _optional_npz(data, *names):
#         for name in names:
#             if name in data.files:
#                 return data[name]
#         return None

#     @staticmethod
#     def _clean_scalar(v):
#         if isinstance(v, np.ndarray) and v.shape == ():
#             v = v.item()
#         if isinstance(v, bytes):
#             v = v.decode('utf-8')
#         if isinstance(v, np.generic):
#             v = v.item()
#         return v

#     @classmethod
#     def _optional_value(cls, arr, local_idx, default=None):
#         if arr is None:
#             return default
#         try:
#             v = arr[local_idx]
#         except Exception:
#             return default
#         v = cls._clean_scalar(v)
#         if isinstance(v, np.ndarray) and v.shape == ():
#             v = cls._clean_scalar(v.item())
#         return v

#     def _validated_target_ids(self, cached_ids, text):
#         def _valid(ids):
#             return bool(ids) and max(ids) < BLANK and min(ids) >= 0

#         if cached_ids is not None and _valid(cached_ids):
#             return cached_ids, False

#         target_text = normalize_ctc_target_text(text)
#         token_ids = self.sp.EncodeAsIds(target_text)
#         if _valid(token_ids):
#             return [int(x) for x in token_ids], cached_ids is not None
#         return None, cached_ids is not None

#     @classmethod
#     def _token_ids_from_cache(cls, data, local_idx):
#         tok = data.get('token_ids')
#         if tok is None:
#             return None
#         if 'token_offsets' in data:
#             s = int(data['token_offsets'][local_idx])
#             e = int(data['token_offsets'][local_idx + 1])
#             ids = tok[s:e]
#         else:
#             ids = tok[local_idx]
#             if 'token_lens' in data:
#                 ids = ids[:int(data['token_lens'][local_idx])]
#         if isinstance(ids, np.ndarray):
#             if ids.dtype == object and ids.shape == ():
#                 ids = ids.item()
#             else:
#                 ids = ids.tolist()
#         if isinstance(ids, (int, np.integer)):
#             ids = [int(ids)]
#         return [int(x) for x in list(ids)]

#     def _get_shard(self, shard_path):
#         if shard_path not in self._cache:
#             if len(self._cache) >= self._max_cache:
#                 oldest = self._cache_order.pop(0)
#                 del self._cache[oldest]
#             data = np.load(shard_path, allow_pickle=True)
#             entry = {
#                 'hidden_cat': data['hidden_cat'],
#                 'offsets': data['offsets'],
#                 'scales': data['scales'],
#                 'zeros': data['zeros'],
#                 'mel_lens': data['mel_lens'],
#                 'texts': self._optional_npz(data, 'texts', 'text'),
#             }
#             for key in [
#                 'token_ids', 'token_offsets', 'token_lens',
#                 'source_ids', 'source_id', 'split_ids', 'split_id',
#                 'speaker_ids', 'speaker_id', 'chunk_start_s', 'chunk_end_s',
#                 'duration_s', 'durations_s', 'langs', 'lang',
#                 'teacher_confidence', 'teacher_confidences',
#             ]:
#                 if key in data.files:
#                     entry[key] = data[key]
#             if self.mel_dir is not None:
#                 relative_shard = os.path.relpath(shard_path, self.cache_dir)
#                 mel_path = os.path.join(self.mel_dir, relative_shard)
#                 if not os.path.isfile(mel_path):
#                     raise FileNotFoundError(
#                         f'Companion mel shard not found for {shard_path}: {mel_path}')
#                 md = np.load(mel_path, allow_pickle=True)
#                 assert np.array_equal(md['mel_lens'], data['mel_lens']), (
#                     f"mel/teacher cache misaligned for {os.path.basename(shard_path)}: "
#                     "mel_lens differ - regenerate the mel cache from the same chunk order.")
#                 entry['mel_cat'] = md['mel_cat']
#                 entry['mel_off'] = md['offsets']
#                 entry['mel_sc'] = md['scales']
#                 entry['mel_zr'] = md['zeros']
#             self._cache[shard_path] = entry
#             self._cache_order.append(shard_path)
#         return self._cache[shard_path]

#     def __len__(self):
#         return len(self.index)

#     def __getitem__(self, idx):
#         shard_path, local_idx = self.index[idx]
#         data = self._get_shard(shard_path)
#         s, e = data['offsets'][local_idx], data['offsets'][local_idx + 1]
#         h_i8 = data['hidden_cat'][s:e].astype(np.float32)
#         h_f16 = (
#             (h_i8 -
#              data['zeros'][local_idx]) *
#             data['scales'][local_idx]).astype(
#             np.float16)

#         text_arr = data.get('texts')
#         raw_text = str(
#             self._optional_value(
#                 text_arr,
#                 local_idx,
#                 '')) if text_arr is not None else ''
#         target_text = normalize_ctc_target_text(raw_text)
#         mel_len = int(data['mel_lens'][local_idx])
#         cached_ids = None if getattr(C, 'force_retokenize_targets', True) else (
#             self._token_ids_from_cache(data, local_idx))
#         token_ids, _ = self._validated_target_ids(cached_ids, raw_text)
#         if not token_ids:
#             raise ValueError(
#                 f'Could not produce valid target IDs for {os.path.basename(shard_path)} '
#                 f'idx={local_idx} with active blank={BLANK}')

#         fallback_lang = os.path.basename(os.path.dirname(shard_path))
#         lang = self._optional_value(
#             data.get(
#                 'langs',
#                 data.get('lang')),
#             local_idx,
#             fallback_lang)
#         duration = self._optional_value(
#             data.get(
#                 'duration_s',
#                 data.get('durations_s')),
#             local_idx,
#             None)
#         if duration is None:
#             duration = mel_len / 100.0

#         item = {
#             'teacher_h': torch.from_numpy(h_f16),
#             'mel_len': mel_len,
#             'token_ids': torch.tensor(token_ids, dtype=torch.long),
#             'text': target_text,
#             'raw_text': raw_text,
#             'lang': canonical_language_name(lang),
#             'source_id': self._optional_value(data.get('source_ids', data.get('source_id')), local_idx, None),
#             'speaker_id': self._optional_value(data.get('speaker_ids', data.get('speaker_id')), local_idx, None),
#             'chunk_start_s': self._optional_value(data.get('chunk_start_s'), local_idx, None),
#             'chunk_end_s': self._optional_value(data.get('chunk_end_s'), local_idx, None),
#             'duration_s': float(duration) if duration is not None else None,
#             'teacher_confidence': self._optional_value(
#                 data.get('teacher_confidence', data.get('teacher_confidences')), local_idx, None),
#         }
#         if self.mel_dir is not None:
#             ms, me = data['mel_off'][local_idx], data['mel_off'][local_idx + 1]
#             m_i8 = data['mel_cat'][ms:me].astype(np.float32)
#             m = (
#                 (m_i8 -
#                  data['mel_zr'][local_idx]) *
#                 data['mel_sc'][local_idx]).astype(
#                 np.float32)
#             item['mel'] = torch.from_numpy(m).transpose(0, 1).contiguous()
#         return item


# # ── Shard-bucketed batch sampler ─────────────────────────────────
# # Batch by shard to avoid repeated np.load on slow FUSE mounts; shuffle
# # shard order and in-shard samples each epoch.
# class ShardBucketBatchSampler:
#     def __init__(self, dataset, batch_size, generator=None,
#                  drop_last=True, shuffle=True, merge_tails=False,
#                  language_temperature=None, excluded_languages=()):
#         """Shard-local batches with optional global tail merge and balancing.

#         ``merge_tails`` keeps the fast path (full batches still come from one
#         NPZ shard) but pools each shard's remainder. With bs=32 this avoids
#         silently dropping roughly 20% of the cache.

#         ``language_temperature`` moderately repeats whole low-resource shards.
#         Repetition preserves locality and every high-resource example, unlike a
#         random weighted sampler that would thrash Kaggle's FUSE mounts.
#         """
#         self.batch_size = int(batch_size)
#         self.generator = generator
#         self.drop_last = bool(drop_last)
#         self.shuffle = bool(shuffle)
#         self.merge_tails = bool(merge_tails)
#         self.language_temperature = language_temperature
#         self.excluded_languages = {
#             canonical_language_name(lang) for lang in (excluded_languages or ())
#         }
#         self.excluded_language_counts = collections.Counter()

#         from torch.utils.data import Subset

#         def _resolve(d):
#             if isinstance(d, Subset):
#                 base, inner = _resolve(d.dataset)
#                 return base, [inner[i] for i in d.indices]
#             return d, list(range(len(d)))

#         base_dataset, idx_chain = _resolve(dataset)
#         shard_to_positions = {}
#         shard_to_language = {}
#         sample_languages = getattr(base_dataset, 'sample_languages', None)
#         for pos, real_i in enumerate(idx_chain):
#             shard_path, _ = base_dataset.index[real_i]
#             if sample_languages is not None:
#                 lang = canonical_language_name(sample_languages[real_i])
#             else:
#                 lang = canonical_language_name(
#                     os.path.basename(os.path.dirname(shard_path)))
#             if lang in self.excluded_languages:
#                 self.excluded_language_counts[lang] += 1
#                 continue
#             shard_to_positions.setdefault(shard_path, []).append(pos)
#             prior = shard_to_language.setdefault(shard_path, lang)
#             if prior != lang:
#                 raise RuntimeError(
#                     f'One cache shard contains multiple languages: {shard_path}')

#         if not shard_to_positions:
#             raise RuntimeError(
#                 'Training/validation sampler is empty after language '
#                 f'exclusion: {sorted(self.excluded_languages)}')
#         self.shard_paths = list(shard_to_positions)
#         self.shard_buckets = [
#             shard_to_positions[path] for path in self.shard_paths]
#         self.bucket_languages = [
#             shard_to_language[path] for path in self.shard_paths]

#         self.language_counts = {}
#         for lang, bucket in zip(self.bucket_languages, self.shard_buckets):
#             self.language_counts[lang] = (
#                 self.language_counts.get(lang, 0) + len(bucket))

#         if language_temperature is not None:
#             temperature = float(language_temperature)
#             if not 0.0 < temperature <= 1.0:
#                 raise ValueError(
#                     'language_temperature must be in (0, 1], '
#                     f'got {language_temperature}')
#             max_count = max(self.language_counts.values(), default=1)
#             self.language_repeats = {
#                 lang: max(
#                     1,
#                     int(math.floor(
#                         (max_count / max(count, 1)) ** (1.0 - temperature)
#                         + 0.5)),
#                 )
#                 for lang, count in self.language_counts.items()
#             }
#         else:
#             self.language_repeats = {
#                 lang: 1 for lang in self.language_counts}
#         self.bucket_repeats = [
#             self.language_repeats[lang] for lang in self.bucket_languages]

#         if self.merge_tails:
#             full_batches = sum(
#                 repeat * (len(bucket) // self.batch_size)
#                 for bucket, repeat in zip(
#                     self.shard_buckets, self.bucket_repeats)
#             )
#             tail_samples = sum(
#                 repeat * (len(bucket) % self.batch_size)
#                 for bucket, repeat in zip(
#                     self.shard_buckets, self.bucket_repeats)
#             )
#             self._n_batches = full_batches + tail_samples // self.batch_size
#             if not self.drop_last and tail_samples % self.batch_size:
#                 self._n_batches += 1
#         elif self.drop_last:
#             self._n_batches = sum(
#                 repeat * (len(bucket) // self.batch_size)
#                 for bucket, repeat in zip(
#                     self.shard_buckets, self.bucket_repeats)
#             )
#         else:
#             self._n_batches = sum(
#                 repeat * ((len(bucket) + self.batch_size - 1) // self.batch_size)
#                 for bucket, repeat in zip(
#                     self.shard_buckets, self.bucket_repeats)
#             )

#     def __len__(self):
#         return self._n_batches

#     def _shuffled_bucket(self, bucket):
#         if not self.shuffle:
#             return bucket
#         if self.generator is not None:
#             perm = torch.randperm(
#                 len(bucket), generator=self.generator).tolist()
#             return [bucket[i] for i in perm]
#         import random
#         out = bucket.copy()
#         random.shuffle(out)
#         return out

#     def __iter__(self):
#         n_buckets = len(self.shard_buckets)
#         if not self.shuffle:
#             shard_order = list(range(n_buckets))
#         elif self.generator is not None:
#             shard_order = torch.randperm(
#                 n_buckets, generator=self.generator).tolist()
#         else:
#             import random
#             shard_order = list(range(n_buckets))
#             random.shuffle(shard_order)

#         # Expand then reshuffle shard occurrences so a 4x low-resource
#         # repeat is distributed through the epoch instead of presenting the
#         # same shard four times consecutively. Each occurrence still emits its
#         # full shard-local batches together, preserving cache locality.
#         expanded_order = []
#         for s_idx in shard_order:
#             expanded_order.extend([s_idx] * self.bucket_repeats[s_idx])
#         if self.shuffle and expanded_order:
#             if self.generator is not None:
#                 perm = torch.randperm(
#                     len(expanded_order), generator=self.generator).tolist()
#                 expanded_order = [expanded_order[i] for i in perm]
#             else:
#                 import random
#                 random.shuffle(expanded_order)

#         tail_pool = []
#         for s_idx in expanded_order:
#             bucket = self._shuffled_bucket(self.shard_buckets[s_idx])
#             if self.merge_tails:
#                 full_end = (len(bucket) // self.batch_size) * self.batch_size
#                 for i in range(0, full_end, self.batch_size):
#                     yield bucket[i:i + self.batch_size]
#                 tail_pool.extend(bucket[full_end:])
#             else:
#                 for i in range(0, len(bucket), self.batch_size):
#                     batch = bucket[i:i + self.batch_size]
#                     if len(batch) < self.batch_size and self.drop_last:
#                         continue
#                     yield batch

#         if self.merge_tails:
#             for i in range(0, len(tail_pool), self.batch_size):
#                 batch = tail_pool[i:i + self.batch_size]
#                 if len(batch) < self.batch_size and self.drop_last:
#                     continue
#                 yield batch


# # ── Persistent train/val/test splits ─────────────────────────────
# # Persist train/val/test splits across Kaggle sessions; fingerprint shards
# # so stale splits fail loudly.
# def _dataset_fingerprint(ds):
#     """Short hash of dataset state. Changes if shards are added, removed,
#     or reordered. Sampled across the dataset, not just the head."""
#     import hashlib
#     h = hashlib.sha256()
#     n = len(ds)
#     h.update(f'len={n};split=speaker_or_source_v2'.encode())
#     step = max(1, n // 100)
#     for i in range(0, n, step):
#         shard_path, local_idx = ds.index[i]
#         # Use last two path components (language_dir/shard_file) so the
#         # fingerprint survives moving the cache to a different mount.
#         parts = shard_path.replace('\\', '/').rsplit('/', 2)[-2:]
#         group_id = ds.group_ids[i] if hasattr(ds, 'group_ids') else ''
#         h.update(f'{"/".join(parts)}:{local_idx}:{group_id}'.encode())
#     return h.hexdigest()[:16]


# SPLIT_STRATEGY = 'speaker_or_source_v2'


# def get_or_create_splits(ds, val_size, test_size, seed, splits_path):
#     """Persist speaker- or source-grouped train/val/test splits.

#     Chunk caches contain overlapping windows. Zeroth uses speaker IDs, keeping
#     every recording from one speaker in one split; other sources retain
#     recording-level grouping. Actual held-out sizes can exceed targets when the
#     final group contributes many chunks.
#     """
#     from torch.utils.data import Subset

#     fingerprint = _dataset_fingerprint(ds)
#     group_ids = getattr(ds, 'group_ids', None)
#     if group_ids is None or len(group_ids) != len(ds):
#         raise RuntimeError(
#             'KDDataset group_ids are missing/misaligned; rerun the distillation '
#             'definition cell before creating splits.')

#     if os.path.exists(splits_path):
#         with open(splits_path) as f:
#             saved = json.load(f)
#         saved_strategy = saved.get('strategy')
#         if saved_strategy != SPLIT_STRATEGY or saved.get('fingerprint') != fingerprint:
#             raise RuntimeError(
#                 f'Splits in {splits_path} are stale or unsafe:\n'
#                 f'  saved strategy      = {saved_strategy!r}\n'
#                 f'  required strategy   = {SPLIT_STRATEGY!r}\n'
#                 f'  saved fingerprint   = {saved.get("fingerprint")}\n'
#                 f'  current fingerprint = {fingerprint}\n'
#                 'Delete only the writable splits.json and rerun this cell. Old '
#                 'per-chunk test results are invalid because overlapping chunks '
#                 'from one source could cross split boundaries.'
#             )
#         train_idx = saved['train_indices']
#         val_idx = saved['val_indices']
#         test_idx = saved['test_indices']
#         train_groups = {group_ids[i] for i in train_idx}
#         val_groups = {group_ids[i] for i in val_idx}
#         test_groups = {group_ids[i] for i in test_idx}
#         assert train_groups.isdisjoint(val_groups)
#         assert train_groups.isdisjoint(test_groups)
#         assert val_groups.isdisjoint(test_groups)
#         print(f'Loaded persisted {SPLIT_STRATEGY} splits from {splits_path}')
#         print(f'  seed={saved["seed"]}, fingerprint={fingerprint}, '
#               f'train/val/test = {len(train_idx):,} / {len(val_idx):,} / '
#               f'{len(test_idx):,}')
#         return Subset(ds, train_idx), Subset(ds, val_idx), Subset(ds, test_idx)

#     n = len(ds)
#     if val_size + test_size >= n:
#         raise ValueError(
#             f'val_size + test_size = {val_size + test_size} >= dataset size {n}')

#     groups = {}
#     for idx, group_id in enumerate(group_ids):
#         groups.setdefault(str(group_id), []).append(idx)
#     group_keys = sorted(groups)
#     gen = torch.Generator().manual_seed(seed)
#     order = torch.randperm(len(group_keys), generator=gen).tolist()

#     train_idx, val_idx, test_idx = [], [], []
#     for pos in order:
#         rows = groups[group_keys[pos]]
#         if len(val_idx) < val_size or len(test_idx) < test_size:
#             # Fill the currently smaller fraction of its target. This keeps the
#             # two held-out splits close in size without ever breaking a source.
#             val_fill = len(val_idx) / max(val_size, 1)
#             test_fill = len(test_idx) / max(test_size, 1)
#             if len(val_idx) < val_size and (val_fill <= test_fill or len(test_idx) >= test_size):
#                 val_idx.extend(rows)
#             else:
#                 test_idx.extend(rows)
#         else:
#             train_idx.extend(rows)

#     if not train_idx or not val_idx or not test_idx:
#         raise RuntimeError(
#             f'Grouped split failed: train/val/test={len(train_idx)}/{len(val_idx)}/{len(test_idx)}')

#     train_groups = {group_ids[i] for i in train_idx}
#     val_groups = {group_ids[i] for i in val_idx}
#     test_groups = {group_ids[i] for i in test_idx}
#     assert train_groups.isdisjoint(val_groups)
#     assert train_groups.isdisjoint(test_groups)
#     assert val_groups.isdisjoint(test_groups)

#     os.makedirs(os.path.dirname(splits_path) or '.', exist_ok=True)
#     with open(splits_path, 'w') as f:
#         json.dump({
#             'strategy': SPLIT_STRATEGY,
#             'fingerprint': fingerprint,
#             'seed': seed,
#             'dataset_len': n,
#             'requested_val_size': val_size,
#             'requested_test_size': test_size,
#             'val_size': len(val_idx),
#             'test_size': len(test_idx),
#             'train_indices': train_idx,
#             'val_indices': val_idx,
#             'test_indices': test_idx,
#         }, f)
#     print(f'Created {SPLIT_STRATEGY} splits at {splits_path}: '
#           f'train={len(train_idx):,}, val={len(val_idx):,}, '
#           f'test={len(test_idx):,} (seed={seed}, groups={len(groups):,})')
#     return Subset(ds, train_idx), Subset(ds, val_idx), Subset(ds, test_idx)


# def collate_kd(batch, max_mel_frames=None, allow_crop=False):
#     """Pad chunk-level examples without corrupting CTC alignment.

#     True chunk data already pairs each audio span with the transcript spoken in
#     that span. If a chunk exceeds a safety cap, raise so the data can be
#     filtered/regenerated; do not crop the audio while keeping the full target.
#     """
#     n_mels = batch[0]['mel'].shape[0]
#     raw_max_m = max(b['mel'].shape[1] for b in batch)
#     if max_mel_frames is not None and raw_max_m > max_mel_frames and not allow_crop:
#         offenders = [
#             (i, b.get('source_id'), b['mel'].shape[1], b.get('text', '')[:80])
#             for i, b in enumerate(batch) if b['mel'].shape[1] > max_mel_frames
#         ]
#         raise ValueError(
#             f'Batch contains chunk longer than max_mel_frames={max_mel_frames}. '
#             f'Do not crop audio for CTC; regenerate/filter shorter chunks. '
#             f'First offenders: {offenders[:3]}')

#     max_m = raw_max_m if max_mel_frames is None else min(raw_max_m, max_mel_frames)
#     mel = torch.zeros(len(batch), n_mels, max_m, dtype=torch.float32)
#     mel_lens = []
#     for i, b in enumerate(batch):
#         t = min(b['mel'].shape[1], max_m)
#         mel[i, :, :t] = b['mel'][:, :t]
#         mel_lens.append(t)

#     max_t = max(b['teacher_h'].shape[0] for b in batch)
#     if allow_crop and max_mel_frames is not None:
#         max_t = min(max_t, max_m // 2)
#     D = batch[0]['teacher_h'].shape[1]
#     teacher_h = torch.zeros(len(batch), max_t, D, dtype=torch.float16)
#     teacher_mask = torch.zeros(len(batch), max_t, dtype=torch.bool)
#     for i, b in enumerate(batch):
#         t = min(b['teacher_h'].shape[0], max_t)
#         teacher_h[i, :t] = b['teacher_h'][:t]
#         teacher_mask[i, :t] = True

#     token_lists = []
#     for b in batch:
#         ids = b['token_ids']
#         if isinstance(ids, torch.Tensor):
#             ids = ids.tolist()
#         ids = [int(x) for x in ids]
#         if ids:
#             assert max(
#                 ids) < BLANK, f'Target contains blank/invalid id: max={max(ids)}, blank={BLANK}'
#             assert min(ids) >= 0, f'Target contains negative id: min={min(ids)}'
#         token_lists.append(ids)

#     max_tok = max((len(ids) for ids in token_lists), default=0)
#     tokens = torch.full((len(batch), max_tok), BLANK, dtype=torch.long)
#     tok_lens = []
#     for i, ids in enumerate(token_lists):
#         tl = len(ids)
#         if tl:
#             tokens[i, :tl] = torch.tensor(ids, dtype=torch.long)
#         tok_lens.append(tl)

#     return {
#         'mel': mel,
#         'mel_lens': torch.tensor(mel_lens, dtype=torch.long),
#         'teacher_h': teacher_h,
#         'teacher_mask': teacher_mask,
#         'teacher_lens': teacher_mask.sum(dim=1, dtype=torch.long),
#         'tokens': tokens,
#         'tok_lens': torch.tensor(tok_lens, dtype=torch.long),
#         'texts': [b['text'] for b in batch],
#         'raw_texts': [b.get('raw_text', b['text']) for b in batch],
#         'langs': [canonical_language_name(
#             b.get('lang', 'unknown')) for b in batch],
#         'source_ids': [b.get('source_id', None) for b in batch],
#         'chunk_start_s': [b.get('chunk_start_s', None) for b in batch],
#         'chunk_end_s': [b.get('chunk_end_s', None) for b in batch],
#         'audio_seconds': [b.get('duration_s', None) for b in batch],
#         'teacher_confidence': torch.tensor([
#             float('nan') if b.get('teacher_confidence', None) is None
#             else float(b['teacher_confidence'])
#             for b in batch
#         ], dtype=torch.float32),
#     }

# # --- 4. Student Model (ConMamba encoder, single mel path) ---
# # Single mel path: mel -> conv -> ConMamba -> {CTC, feature KD}; keep the
# # fp32 encoder island because selective-scan can overflow in fp16.


# class CausalConv1d(nn.Conv1d):
#     """Left-pad a strided Conv1d; weight/state-dict shapes stay unchanged."""

#     def forward(self, x):
#         left = (self.kernel_size[0] - 1) * self.dilation[0]
#         return super().forward(F.pad(x, (left, 0)))


# class ConMambaStudent(nn.Module):
#     def __init__(self, config):
#         super().__init__()
#         self.config = config
#         d = config.cm_d_model

#         # Front-end: two stride-2 convs -> 4x downsample (mel 100 fps -> ~25 fps).
#         self.cnn = nn.Sequential(
#             CausalConv1d(
#                 config.n_mels,
#                 d,
#                 config.cnn_ks,
#                 stride=2,
#                 padding=0),
#             nn.GELU(),
#             CausalConv1d(d, d, config.cnn_ks, stride=2, padding=0),
#             nn.GELU(),
#         )
#         self.in_norm = nn.LayerNorm(d)

#         # Real ConmambaEncoder (SpeechBrain), UNIDIRECTIONAL: causal=True (since
#         # cm_bidirectional=False) selects stock mamba_ssm.Mamba + a causal conv.
#         self.encoder = ConmambaEncoder(
#             num_layers=config.cm_layers, d_model=d, d_ffn=config.cm_d_ffn,
#             kernel_size=config.cm_kernel, dropout=config.dropout,
#             causal=not config.cm_bidirectional,
#             mamba_config=dict(d_state=config.cm_d_state, expand=config.cm_expand,
#                               d_conv=config.cm_d_conv,
#                               bidirectional=config.cm_bidirectional),
#         )

#         self.dropout = nn.Dropout(getattr(config, 'dropout', 0.0))
#         self.ctc_head = nn.Linear(d, config.vocab_size)    # CTC over SP vocab + blank
#         self.kl_head = nn.Linear(d, config.teacher_d)      # output distillation target

#     def _subsample_len(self, lens):
#         """Valid lengths after two stride-2 causal convs (odd k, left pad k-1)."""
#         k, p, s = self.config.cnn_ks, self.config.cnn_ks // 2, 2
#         for _ in range(2):
#             lens = torch.div(lens + 2 * p - k, s, rounding_mode='floor') + 1
#         return lens.clamp(min=1)

#     def enable_grad_ckpt(self):
#         """Per-layer activation checkpointing on the ConMamba encoder layers.
#         Patches the LAYER CLASS once (not instances) so DataParallel replicas on
#         cuda:1 don't capture cuda:0-bound closures. Idempotent. Same trick the old
#         mamba-130m path used, retargeted to ConMambaEncoderLayer."""
#         from torch.utils.checkpoint import checkpoint
#         if not self.encoder.layers:
#             return
#         block_cls = type(self.encoder.layers[0])
#         if getattr(block_cls, '_grad_ckpt_patched', False):
#             print(f'Gradient checkpointing already enabled on {block_cls.__name__}')
#             return
#         original_forward = block_cls.forward

#         def ckpt_forward(self, hidden_states, *args, **kwargs):
#             if not self.training:
#                 return original_forward(self, hidden_states, *args, **kwargs)
#             return checkpoint(
#                 lambda h: original_forward(self, h, *args, **kwargs),
#                 hidden_states, use_reentrant=False,
#             )

#         block_cls.forward = ckpt_forward
#         block_cls._grad_ckpt_patched = True
#         print(f'Gradient checkpointing enabled on {len(self.encoder.layers)} '
#               f'{block_cls.__name__} blocks (DataParallel-safe)')

#     def forward(self, mel, mel_lens):
#         # mel: [B, n_mels, T_mel]
#         x = self.cnn(mel)                  # [B, d, T_s]  (autocast fp16 ok here)
#         x = x.transpose(1, 2)              # [B, T_s, d]
#         x = self.in_norm(x)
#         enc_lens = self._subsample_len(mel_lens).clamp(max=x.shape[1])
#         padding_mask = (torch.arange(x.shape[1], device=x.device)[None, :]
#                         >= enc_lens[:, None])
#         # fp32 island: the selective-scan recurrence overflows fp16 over long
#         # sequences. Force fp32 activations through the encoder (weights are fp32).
#         with torch.amp.autocast('cuda', enabled=False):
#             h, _ = self.encoder(
#                 x.float(), src_key_padding_mask=padding_mask)
#         h = h.masked_fill(padding_mask.unsqueeze(-1), 0.0)
#         h = self.dropout(h)                # no-op in eval()
#         ctc_logits = self.ctc_head(h)      # [B, T_s, vocab_size]
#         kl_feat = self.kl_head(h)          # [B, T_s, teacher_d]
#         return ctc_logits, kl_feat, enc_lens

# # --- 5. Loss Functions ---


# def kd_feature_loss(student_feat, teacher_feat, student_lens,
#                     teacher_lens=None, chunk=128):
#     """Per-sample output feature distillation via 1 - cosine similarity.

#     Teacher (~50 fps) and student (~25 fps) sequences are aligned independently
#     for every utterance. Interpolating a batch-padded teacher tensor as one block
#     compresses shorter examples and aligns their valid student frames to padding.
#     """
#     B, T_s, _ = student_feat.shape
#     if teacher_lens is None:
#         teacher_lens = torch.full(
#             (B,), teacher_feat.shape[1], dtype=torch.long,
#             device=teacher_feat.device)
#     student_lens = student_lens.to(device=student_feat.device, dtype=torch.long)
#     teacher_lens = teacher_lens.to(device=teacher_feat.device, dtype=torch.long)

#     total = student_feat.new_zeros((), dtype=torch.float32)
#     n_valid = 0
#     for bi in range(B):
#         s_len = max(1, min(int(student_lens[bi].item()), T_s))
#         t_len = max(1, min(int(teacher_lens[bi].item()), teacher_feat.shape[1]))
#         target = teacher_feat[bi:bi + 1, :t_len].float().transpose(1, 2)
#         if t_len != s_len:
#             target = F.interpolate(
#                 target, size=s_len, mode='linear', align_corners=False)
#         target = target.transpose(1, 2)[0]
#         for s in range(0, s_len, chunk):
#             e = min(s + chunk, s_len)
#             cos = F.cosine_similarity(
#                 student_feat[bi, s:e].float(), target[s:e], dim=-1)
#             total = total + (1.0 - cos).sum()
#         n_valid += s_len
#     return total / max(n_valid, 1)


# def ctc_required_lens(tokens, tok_lens):
#     """Minimum CTC frames per sample, including blank slots between repeats.

#     PyTorch CTC returns inf when a target cannot be aligned. The simple
#     tok_len <= enc_len check misses repeated adjacent tokens, which need an
#     extra frame each (for the separating blank). Keep zero_infinity=False below
#     so real data bugs stay visible, but catch them here with useful context.
#     """
#     req = tok_lens.to(dtype=torch.long).clone()
#     if tokens.numel() == 0 or tokens.shape[1] <= 1:
#         return req
#     dev_lens = tok_lens.to(tokens.device)
#     for pos in range(tokens.shape[1] - 1):
#         valid_pair = (pos + 1) < dev_lens
#         repeat_pair = tokens[:, pos] == tokens[:, pos + 1]
#         req = req + (valid_pair & repeat_pair).to(req.device, dtype=torch.long)
#     return req


# def describe_invalid_ctc_batch(batch, enc_lens, tok_lens, tokens=None, limit=5):
#     req_lens = tok_lens.to(enc_lens.device)
#     if tokens is not None:
#         req_lens = ctc_required_lens(tokens, tok_lens).to(enc_lens.device)
#     bad_mask = req_lens > enc_lens
#     bad = torch.nonzero(bad_mask, as_tuple=False).flatten().tolist()
#     if bad:
#         print('Invalid CTC samples in batch:')
#         for bi in bad[:limit]:
#             src_id = batch.get('source_ids', [None] * len(batch['texts']))[bi]
#             start = batch.get('chunk_start_s', [None] * len(batch['texts']))[bi]
#             end = batch.get('chunk_end_s', [None] * len(batch['texts']))[bi]
#             print(
#                 f'  sample={bi}, enc_len={enc_lens[bi].item()}, '
#                 f'tok_len={tok_lens[bi].item()}, req_len={req_lens[bi].item()}, '
#                 f'lang={batch["langs"][bi]}, source={src_id}, '
#                 f'chunk=({start}, {end}), text={batch["texts"][bi][:120]}'
#             )
#     return bad


# def ctc_confidence_weights(confidence, device):
#     """Map pseudo-label confidence to bounded per-sample CTC weights.

#     Acoustic feature KD remains unweighted: a low-confidence transcript does
#     not make the teacher encoder representation invalid. NaN/missing values
#     receive weight 1.0 so older cache schemas remain behaviorally compatible.
#     """
#     if not getattr(C, 'ctc_confidence_weighting', False):
#         return None
#     values = torch.as_tensor(
#         confidence, dtype=torch.float32, device=device).flatten()
#     weights = torch.ones_like(values)
#     valid = torch.isfinite(values)
#     if valid.any():
#         floor_conf = float(C.ctc_confidence_min)
#         floor_weight = float(C.ctc_confidence_floor_weight)
#         scaled = (
#             (values[valid].clamp(0.0, 1.0) - floor_conf) /
#             max(1.0 - floor_conf, 1e-8)
#         ).clamp(0.0, 1.0)
#         weights[valid] = floor_weight + (1.0 - floor_weight) * scaled
#     return weights


# def ctc_loss_fn(ctc_logits, tokens, enc_lens, tok_lens,
#                 sample_weights=None):
#     assert ctc_logits.shape[-1] == BLANK + 1, (
#         f'CTC vocab mismatch: logits={ctc_logits.shape[-1]}, expected={BLANK + 1}'
#     )
#     enc_lens = enc_lens.to(dtype=torch.long)
#     tok_lens = tok_lens.to(dtype=torch.long)

#     if tokens.numel() > 0:
#         pos = torch.arange(tokens.shape[1], device=tokens.device)[None, :]
#         valid = pos < tok_lens.to(tokens.device)[:, None]
#         valid_targets = tokens[valid]
#         if valid_targets.numel() > 0:
#             assert valid_targets.max().item() < BLANK, 'CTC targets must not contain blank ID'
#             assert valid_targets.min().item() >= 0, 'CTC targets contain negative IDs'

#     req_lens = ctc_required_lens(tokens, tok_lens).to(enc_lens.device)
#     if not torch.all(req_lens <= enc_lens):
#         bad = torch.nonzero(req_lens > enc_lens, as_tuple=False).flatten().tolist()
#         raise ValueError(
#             f'Invalid CTC lengths. Examples={bad[:5]}, '
#             f'enc_lens={enc_lens[bad[:5]].tolist()}, '
#             f'tok_lens={tok_lens[bad[:5]].tolist()}, '
#             f'req_lens={req_lens[bad[:5]].tolist()}'
#         )

#     log_probs = F.log_softmax(ctc_logits.float(), dim=-1).transpose(0, 1)
#     if sample_weights is None:
#         return F.ctc_loss(
#             log_probs, tokens, enc_lens, tok_lens,
#             blank=BLANK, zero_infinity=False,
#         )

#     # PyTorch's default CTC mean divides each sample loss by target length.
#     # Reproduce that definition before applying confidence weights so enabling
#     # weighting does not silently change the scale of the objective.
#     per_sample = F.ctc_loss(
#         log_probs, tokens, enc_lens, tok_lens,
#         blank=BLANK, reduction='none', zero_infinity=False,
#     )
#     normalized = per_sample / tok_lens.to(
#         per_sample.device, dtype=per_sample.dtype).clamp_min(1)
#     weights = sample_weights.to(
#         per_sample.device, dtype=per_sample.dtype).flatten()
#     if len(weights) != len(normalized):
#         raise ValueError(
#             f'CTC sample weight count {len(weights)} != batch '
#             f'count {len(normalized)}')
#     return (normalized * weights).sum() / weights.sum().clamp_min(1e-8)


### Chunk cache KDDataset sanity-load

Runs after the distillation definitions create `KDDataset`. It checks the freshly generated working cache when present, otherwise the mounted chunk cache.


In [14]:
# # KDDataset-level sanity check for chunk caches.
# _sanity_teacher_dir, _sanity_mel_dir = _select_cache_pair(
#     C.chunk_teacher_work_dir, C.chunk_mel_work_dir, C.cache_dir, C.mel_dir)

# if not os.path.isdir(_sanity_teacher_dir) or not os.path.isdir(_sanity_mel_dir):
#     print('Chunk KDDataset sanity skipped: teacher/mel cache dirs are not available yet.')
#     print(f'  teacher: {_sanity_teacher_dir}')
#     print(f'  mel:     {_sanity_mel_dir}')
# else:
#     print('Chunk KDDataset sanity loading:')
#     print(f'  teacher: {_sanity_teacher_dir}')
#     print(f'  mel:     {_sanity_mel_dir}')
#     _chunk_sanity_ds = KDDataset(_sanity_teacher_dir, sp, _sanity_mel_dir)
#     assert len(_chunk_sanity_ds) > 0, 'Chunk cache is empty.'
#     for _i in range(min(3, len(_chunk_sanity_ds))):
#         _item = _chunk_sanity_ds[_i]
#         assert 'mel' in _item and 'teacher_h' in _item
#         assert int(_item['token_ids'].numel()) > 0
#         assert int(_item['token_ids'].max().item()) < BLANK
#         print(
#             f'  {_i}: mel={tuple(_item["mel"].shape)} '
#             f'teacher={tuple(_item["teacher_h"].shape)} '
#             f'tok_len={int(_item["token_ids"].numel())} '
#             f'lang={_item["lang"]} source={_item["source_id"]} '
#             f'chunk=({_item["chunk_start_s"]}, {_item["chunk_end_s"]}) '
#             f'text={_item["text"][:100]}'
#         )
#     del _chunk_sanity_ds


### Training configuration (run before smoke test)

Defines the full-training dataset/model setup, checkpoint helpers, evaluation helper, and `train_stage`. Run this before the smoke test so the training process cell only launches the guarded stage after `_SMOKE_PASSED = True`.


In [15]:
# # ── 6. Build everything ────────────────────────────────────────
# print('Loading dataset...')
# ds = KDDataset(C.cache_dir, sp, C.mel_dir)
# flush()

# # Persist splits so resumed sessions never leak train samples into eval.
# SPLITS_PATH = os.path.join(C.ckpt_dir, 'splits.json')
# _val_size = max(50, min(2000, len(ds) // 50))   # ~2%, capped at [50, 2000]
# _test_size = _val_size                           # symmetric val/test
# train_ds, val_ds, test_ds = get_or_create_splits(
#     ds, val_size=_val_size, test_size=_test_size,
#     seed=C.seed, splits_path=SPLITS_PATH,
# )
# with open(SPLITS_PATH) as _f:
#     ACTIVE_SPLIT_METADATA = json.load(_f)
# assert ACTIVE_SPLIT_METADATA.get('strategy') == SPLIT_STRATEGY
# print(f'  train: {len(train_ds):,} samples')
# print(f'  val:   {len(val_ds):,} samples (held out, deterministic)')
# print(f'  test:  {len(test_ds):,} samples (held out for final eval — '
#       f'never touched by training)')

# print('\nBuilding student model...')
# student = ConMambaStudent(C).to(device)
# if C.grad_checkpointing:
#     student.enable_grad_ckpt()
# else:
#     print('Gradient checkpointing disabled: using measured T4 memory headroom')

# # Multi-GPU: DataParallel is notebook-friendly; access internals via .module.
# if torch.cuda.device_count() > 1:
#     student = torch.nn.DataParallel(student)
#     print(f'Wrapped student in DataParallel across {torch.cuda.device_count()} GPUs')
# student_core = student.module if isinstance(student, torch.nn.DataParallel) else student

# n_params = sum(p.numel() for p in student_core.parameters())
# n_trainable = sum(p.numel() for p in student_core.parameters() if p.requires_grad)
# print(f'Student: {n_params/1e6:.1f}M params ({n_trainable/1e6:.1f}M trainable)')
# flush()


# # ── 7. Checkpoint helpers ──────────────────────────────────────
# def _atomic_save(state, path):
#     """Write to .tmp then rename. POSIX rename is atomic, so a kill mid-write
#     leaves the previous checkpoint intact instead of corrupting it."""
#     # Fail early on directory/empty paths before torch.save emits opaque errors.
#     if not path or path.endswith(os.sep) or os.path.isdir(path):
#         raise ValueError(
#             f'_atomic_save got an invalid path: {path!r}. '
#             f'Expected a full file path like '
#             f'{os.path.join(C.ckpt_dir, "frozen_latest.pt")!r}.'
#         )
#     # Recreate the writable parent if Kaggle cleanup removed it.
#     os.makedirs(os.path.dirname(path) or '.', exist_ok=True)
#     tmp = path + '.tmp'
#     torch.save(state, tmp)
#     os.replace(tmp, path)


# def _save_path(stage_name, suffix='latest'):
#     """Always writable destination — every save uses this."""
#     return os.path.join(C.ckpt_dir, f'{stage_name}_{suffix}.pt')


# def _load_candidates(stage_name, suffix='latest', include_readonly=True):
#     """Ordered list of paths to try when loading.

#     Priority:
#       1. C.ckpt_dir       — writable. Wins if a prior step in THIS session
#                             already saved here (within-session resume).
#       2. C.ckpt_load_dirs — read-only prior-session/baseline sources across
#                             supported Kaggle layouts, in priority order.
#     """
#     fname = f'{stage_name}_{suffix}.pt'
#     paths = [os.path.join(C.ckpt_dir, fname)]
#     if include_readonly:
#         load_dirs = getattr(C, 'ckpt_load_dirs', None)
#         if load_dirs is None:
#             load_dirs = [getattr(C, 'ckpt_load_dir', None)]
#         for load_dir in load_dirs:
#             if load_dir and load_dir != C.ckpt_dir:
#                 candidate = os.path.join(load_dir, fname)
#                 if candidate not in paths:
#                     paths.append(candidate)
#     return paths


# # Back-compat shims so the rest of the code (and external scripts) still work.
# def _latest_path(stage_name):  # treated as save path everywhere it's used
#     return _save_path(stage_name, 'latest')


# def _best_path(stage_name):
#     return _save_path(stage_name, 'best')


# def _unwrap(model):
#     """Return the inner module if `model` is DataParallel, else `model`.
#     Use this whenever you would call `.state_dict()` or `.load_state_dict()`
#     on the model — checkpoints stay portable between single- and multi-GPU
#     sessions because the `module.` prefix never enters the on-disk format.
#     """
#     return model.module if isinstance(model, torch.nn.DataParallel) else model


# def _normalize_state_dict(sd):
#     """Strip any `module.` prefix from keys, so the dict matches an
#     un-wrapped model. No-op if no keys are prefixed."""
#     if not any(k.startswith('module.') for k in sd):
#         return sd
#     return {(k[len('module.'):] if k.startswith('module.') else k): v
#             for k, v in sd.items()}


# def _load_model_tolerant(model, sd, allowed_missing_prefixes=()):
#     """Load `sd` into `model`, dropping any param whose shape no longer matches
#     the current model, and erroring on anything else missing/unexpected.

#     Migrates a checkpoint across a *deliberate* shape change — e.g. the ctc_head
#     going 120001 -> 12001 after the vocab_size fix — without throwing away the
#     rest of a long run. Dropped params keep their fresh init; the encoder +
#     kl_head (unchanged shapes) are restored. Returns the list of dropped keys
#     (truthy => a migration happened)."""
#     model_sd = model.state_dict()
#     dropped = [k for k, v in sd.items()
#                if k in model_sd and v.shape != model_sd[k].shape]
#     if dropped:
#         print(f'  Dropping {len(dropped)} shape-mismatched param(s) from '
#               f'checkpoint (kept fresh init): {dropped}')
#         sd = {k: v for k, v in sd.items() if k not in dropped}
#     incompat = model.load_state_dict(sd, strict=False)
#     if incompat.unexpected_keys:
#         raise RuntimeError(f'Unexpected keys in checkpoint: {incompat.unexpected_keys}')
#     missing = set(incompat.missing_keys) - set(dropped)
#     missing = {k for k in missing if not any(k.startswith(p) for p in allowed_missing_prefixes)}
#     if missing:
#         raise RuntimeError(f'Checkpoint missing expected params: {sorted(missing)}')
#     return dropped


# def save_full_checkpoint(path, *, model, optimizer, scheduler, scaler,
#                          stage_name, epoch, batch_idx, global_step, best_loss,
#                          best_val_metric=float('inf'),
#                          epochs_since_improvement=0,
#                          early_stopped=False,
#                          last_train_loss=None,
#                          resume_tag=None,
#                          canary_baseline_loss=None,
#                          canary_done=False,
#                          evaluation_lineage_clean=False,
#                          best_metric_name=None):
#     """Save everything needed to resume training exactly.

#     Includes early-stopping state so resumes across sessions correctly
#     track epochs-without-improvement.
#     """
#     display_loss = best_loss
#     if last_train_loss is not None and math.isfinite(float(last_train_loss)):
#         display_loss = float(last_train_loss)
#     state = {
#         # Save unwrapped weights so checkpoints load with or without DP.
#         'model':        _unwrap(model).state_dict(),
#         'optimizer':    optimizer.state_dict(),
#         'scheduler':    scheduler.state_dict(),
#         'scaler':       scaler.state_dict(),  # preserves fp16 loss scale
#         'stage_name':   stage_name,
#         'resume_tag':   resume_tag,
#         'epoch':        epoch,
#         'batch_idx':    batch_idx,            # for within-epoch resume
#         # A micro-batch change preserves effective batch/optimizer state but
#         # changes the meaning of batch_idx. Persist shape metadata so a future
#         # resume can restart the partially consumed epoch instead of silently
#         # skipping most of it.
#         'micro_batch_size': (
#             C.fr_bs if stage_name.endswith('_kd') else C.un_bs),
#         'grad_accum_steps': (
#             C.fr_ga if stage_name.endswith('_kd') else C.un_ga),
#         'samples_into_epoch': batch_idx * (
#             C.fr_bs if stage_name.endswith('_kd') else C.un_bs),
#         'global_step':  global_step,
#         'best_loss':    best_loss,
#         'loss':         display_loss,         # human-facing resume/pause metric
#         'last_train_loss': last_train_loss,
#         # Persist the one-shot canary state. Without this, a resumed stage
#         # at step > canary_updates reruns the canary immediately and can
#         # reject a checkpoint that already passed it.
#         'canary_baseline_loss': canary_baseline_loss,
#         'canary_done': bool(canary_done),
#         # Split identity describes this run; lineage_clean describes whether
#         # every ancestor was also trained with source-disjoint held-out data.
#         'split_strategy': ACTIVE_SPLIT_METADATA.get('strategy'),
#         'split_fingerprint': ACTIVE_SPLIT_METADATA.get('fingerprint'),
#         'evaluation_lineage_clean': bool(evaluation_lineage_clean),
#         # Early-stopping state (carried across session boundaries).
#         'best_val_metric':           best_val_metric,
#         'best_metric_name':          best_metric_name,
#         'epochs_since_improvement':  epochs_since_improvement,
#         'early_stopped':             early_stopped,
#         'config':       C,
#         'torch_version': torch.__version__,
#         'tokenizer_fingerprint': TOKENIZER_FINGERPRINT,
#         'target_recipe_fingerprint': TARGET_RECIPE_FINGERPRINT,
#         'target_recipe': TARGET_RECIPE,
#     }
#     _atomic_save(state, path)


# def save_best_model_only(path, model, *, stage_name, resume_tag,
#                          epoch, global_step, loss, metric=None, collapse=None,
#                          evaluation_lineage_clean=False):
#     """Save inference weights plus lineage needed for safe selection."""
#     _atomic_save({
#         'model': _unwrap(model).state_dict(),
#         'stage_name': stage_name,
#         'resume_tag': resume_tag,
#         'epoch': epoch,
#         'global_step': global_step,
#         'loss': loss,
#         'metric': metric,
#         'best_metric_name': next(
#             (key for key in (metric or {})
#              if key not in {'source', 'normalization'}), None),
#         'collapse': collapse,
#         'split_strategy': ACTIVE_SPLIT_METADATA.get('strategy'),
#         'split_fingerprint': ACTIVE_SPLIT_METADATA.get('fingerprint'),
#         'evaluation_lineage_clean': bool(evaluation_lineage_clean),
#         'tokenizer_fingerprint': TOKENIZER_FINGERPRINT,
#         'target_recipe_fingerprint': TARGET_RECIPE_FINGERPRINT,
#         'target_recipe': TARGET_RECIPE,
#     }, path)


# def try_load_full_checkpoint(path_or_paths):
#     """Try a single path or an ordered list of candidates.

#     Returns (state_dict, loaded_from_path) on success, (None, None) on failure.
#     Each candidate also looks at its .tmp sibling in case a rename was
#     interrupted (extremely rare with atomic os.replace).
#     """
#     if isinstance(path_or_paths, str):
#         candidates = [path_or_paths]
#     else:
#         candidates = list(path_or_paths)

#     for path in candidates:
#         if not os.path.exists(path):
#             continue
#         try:
#             state = torch.load(path, map_location='cpu', weights_only=False)
#             return state, path
#         except Exception as e:
#             print(f'Could not load {path}: {e}')
#             tmp = path + '.tmp'
#             if os.path.exists(tmp):
#                 try:
#                     state = torch.load(tmp, map_location='cpu', weights_only=False)
#                     return state, tmp
#                 except Exception:
#                     pass
#             # Try the next candidate.

#     return None, None


# # ── 8. Training Loop ───────────────────────────────────────────
# # Bump these whenever architecture, source-lineage rules, or loss recipes change.
# # Both full and best checkpoints persist the tag; stale uploaded attempts cannot
# # silently shadow the current recipe on a fresh Kaggle session.
# SCRATCH_KD_RECIPE_TAG = 'scratch_kd_causal_v1'
# LEGACY_SCRATCH_JOINT_RECIPE_TAG = 'scratch_joint_kd_v1'
# PRIOR_SCRATCH_JOINT_RECIPE_TAG = 'scratch_joint_kd_v2'
# PREVIOUS_SCRATCH_JOINT_RECIPE_TAG = 'scratch_joint_kd_v3'
# SCRATCH_JOINT_RECIPE_TAG = 'scratch_joint_kd_v4'
# RECOVER_KD_RECIPE_TAG = 'recover_kd_causal_v2'
# PRIOR_RECOVER_CTC_RECIPE_TAG = 'recover_ctc_joint_kd_v8'
# RECOVER_CTC_RECIPE_TAG = 'recover_ctc_joint_kd_v9'
# # Input-only compatibility for the supplied edge_asr checkpoint.
# EDGE_ASR_INPUT_RECOVER_CTC_TAG = 'recover_ctc_ctc_only_headlr_v7'


# @dataclass
# class StageSpec:
#     name: str
#     source_checkpoint_stage: str = None
#     # Primary source stages may require an exact recipe tag. Fallback stages are
#     # deliberately allowed to be legacy checkpoints because the CTC head is reset.
#     source_checkpoint_tag: str = None
#     fallback_source_stages: tuple = ()
#     source_checkpoint_suffix: str = 'best'
#     fallback_source_suffix: str = 'latest'
#     require_source_checkpoint: bool = False
#     lr: float = 5e-5
#     ctc_head_lr: float = None
#     warmup_steps: int = None
#     epochs: int = 1
#     bs: int = 2
#     ga: int = 16
#     max_seconds: float = None
#     a_kl: float = 0.0
#     kl_end_weight: float = None
#     kl_hold_steps: int = 0
#     kl_decay_steps: int = 0
#     ctc_start_weight: float = 0.0
#     ctc_end_weight: float = 0.0
#     ctc_warmup_steps: int = 0
#     reset_ctc_head: bool = False
#     ctc_blank_bias: float = 0.0
#     best_metric: str = 'wer'
#     allow_resume: bool = True
#     allow_readonly_resume: bool = False
#     fail_fast_on_collapse: bool = False
#     # True only for a fresh training lineage that never used legacy
#     # per-chunk-split weights. Recovery warm-starts must remain False.
#     evaluation_lineage_clean: bool = False
#     # Entries may be 'stage' or ('stage', required_resume_tag). A stale
#     # downstream checkpoint must never prove that an upstream stage completed.
#     skip_if_checkpoint_stages: tuple = ()
#     canary_updates: int = 0
#     canary_min_relative_loss_drop: float = 0.05
#     empty_hypothesis_threshold: float = 0.95
#     blank_frame_threshold: float = 0.98
#     blank_probability_threshold: float = 0.98
#     top_token_threshold: float = 0.90
#     validation_interval_updates: int = 0
#     validation_max_batches: int = 500
#     validation_probe_samples: int = 0
#     midval_selects_best: bool = False
#     resume_tag: str = None
#     # Optional input-only tags for a compatible external checkpoint.
#     resume_source_tags: tuple = ()


# def _stratified_validation_subset(dataset, max_samples, seed):
#     """Build one deterministic, language-balanced validation probe."""
#     if dataset is None or max_samples is None or int(max_samples) <= 0:
#         return dataset, {}
#     max_samples = min(int(max_samples), len(dataset))
#     if max_samples >= len(dataset):
#         return dataset, {}

#     from torch.utils.data import Subset

#     def _resolve(d):
#         if isinstance(d, Subset):
#             base, inner = _resolve(d.dataset)
#             return base, [inner[i] for i in d.indices]
#         return d, list(range(len(d)))

#     base, real_indices = _resolve(dataset)
#     sample_languages = getattr(base, 'sample_languages', None)
#     if sample_languages is None:
#         # No metadata: deterministic uniform subset is still cheaper than the
#         # full validation set, but cannot claim language stratification.
#         rng = random.Random(seed)
#         positions = list(range(len(dataset)))
#         rng.shuffle(positions)
#         selected = sorted(positions[:max_samples])
#         return Subset(dataset, selected), {'unknown': len(selected)}

#     groups = collections.defaultdict(list)
#     for pos, real_idx in enumerate(real_indices):
#         groups[canonical_language_name(
#             sample_languages[real_idx])].append(pos)
#     rng = random.Random(seed)
#     for rows in groups.values():
#         rng.shuffle(rows)

#     selected = []
#     offsets = {lang: 0 for lang in groups}
#     languages = sorted(groups)
#     while len(selected) < max_samples:
#         made_progress = False
#         for lang in languages:
#             offset = offsets[lang]
#             if offset >= len(groups[lang]):
#                 continue
#             selected.append(groups[lang][offset])
#             offsets[lang] += 1
#             made_progress = True
#             if len(selected) >= max_samples:
#                 break
#         if not made_progress:
#             break
#     counts = {
#         lang: offsets[lang] for lang in languages if offsets[lang] > 0}
#     return Subset(dataset, sorted(selected)), counts


# def _collapse_thresholds(spec):
#     return {
#         'empty_threshold': spec.empty_hypothesis_threshold,
#         'blank_threshold': spec.blank_frame_threshold,
#         'blank_probability_threshold': spec.blank_probability_threshold,
#         'top_token_threshold': spec.top_token_threshold,
#     }


# def _is_collapsed(metrics):
#     return bool(metrics.get('collapse', {}).get('collapsed', False))


# def _metric_for_best(metrics, spec, avg_loss):
#     if metrics is None:
#         return float(avg_loss), 'train/loss'
#     if spec.best_metric == 'wer':
#         return float(metrics['wer']), 'val/wer'
#     if spec.best_metric == 'cer':
#         return float(metrics['cer']), 'val/cer'
#     if spec.best_metric == 'macro_cer':
#         return float(metrics['macro_cer']), 'val/macro_cer'
#     return float(metrics['loss']), 'val/loss'


# @torch.no_grad()
# def evaluate(student, val_dataset, max_seconds=None, bs=2, max_batches=None,
#              a_kl=0.0, a_ctc=1.0, collapse_thresholds=None):
#     """Run validation and return loss, WER/CER, and CTC-collapse metrics."""
#     student.eval()
#     max_mel_frames = None if max_seconds is None else int(max_seconds * 100)
#     val_sampler = ShardBucketBatchSampler(
#         val_dataset, batch_size=bs, generator=None,
#         drop_last=False, shuffle=False, merge_tails=True,
#     )
#     loader = DataLoader(
#         val_dataset, batch_sampler=val_sampler,
#         num_workers=C.workers, persistent_workers=(C.workers > 0),
#         pin_memory=True,
#         collate_fn=lambda b: collate_kd(
#             b, max_mel_frames=max_mel_frames, allow_crop=False),
#     )

#     total_loss = total_kl = total_ctc = 0.0
#     loss_sample_count = 0
#     blank_probability_sum = 0.0
#     blank_probability_n = 0
#     n_batches = 0
#     refs, hyps, langs = [], [], []
#     pred_id_seqs, frame_argmax_seqs = [], []

#     for batch_idx, batch in enumerate(loader):
#         if max_batches is not None and batch_idx >= max_batches:
#             break

#         mel = batch['mel'].to(device, non_blocking=True)
#         teacher_h = (batch['teacher_h'].to(device, non_blocking=True)
#                      if a_kl > 0 else None)
#         teacher_lens = (batch['teacher_lens'].to(device, non_blocking=True)
#                         if a_kl > 0 else None)
#         tokens = batch['tokens'].to(device, non_blocking=True)
#         tok_lens = batch['tok_lens'].to(device, non_blocking=True)
#         mel_lens = batch['mel_lens'].to(device, non_blocking=True)

#         with torch.amp.autocast('cuda', dtype=torch.float16):
#             ctc_logits, kl_feat, enc_lens = student(mel, mel_lens)
#             loss_kl = (kd_feature_loss(
#                 kl_feat, teacher_h, enc_lens, teacher_lens=teacher_lens)
#                 if a_kl > 0 else kl_feat.new_zeros((), dtype=torch.float32))
#             ctc_sample_weights = ctc_confidence_weights(
#                 batch['teacher_confidence'], ctc_logits.device)
#             loss_ctc = (ctc_loss_fn(
#                 ctc_logits, tokens, enc_lens, tok_lens,
#                 sample_weights=ctc_sample_weights)
#                 if a_ctc > 0 else
#                 ctc_logits.new_zeros((), dtype=torch.float32))
#             loss = a_kl * loss_kl + a_ctc * loss_ctc

#         # Shard buckets produce many final batches of size one. Weight batch
#         # means by sample count so those tails do not dominate model selection.
#         batch_n = int(mel.shape[0])
#         total_loss += loss.item() * batch_n
#         total_kl += loss_kl.item() * batch_n
#         total_ctc += loss_ctc.item() * batch_n
#         loss_sample_count += batch_n
#         n_batches += 1

#         # Greedy blank argmax is common while a fresh 5k-way CTC head is still
#         # calibrating. Record actual blank probability mass so the fail-fast
#         # canary only stops a genuinely degenerate blank distribution.
#         logits32 = ctc_logits.float()
#         blank_log_probs = logits32[..., BLANK] - torch.logsumexp(logits32, dim=-1)
#         valid_frame_mask = (
#             torch.arange(ctc_logits.shape[1], device=ctc_logits.device)[None, :]
#             < enc_lens[:, None])
#         blank_probability_sum += float(
#             blank_log_probs.exp().masked_select(valid_frame_mask).sum().cpu())
#         blank_probability_n += int(valid_frame_mask.sum().item())

#         logits_cpu = logits32.cpu()
#         lengths = enc_lens.cpu().tolist()
#         pred_ids = greedy_ctc_token_ids(logits_cpu, BLANK, lengths=lengths)
#         frame_argmax = logits_cpu.argmax(dim=-1)
#         for seq, valid_len in zip(frame_argmax, lengths):
#             frame_argmax_seqs.append(seq[:valid_len].tolist())
#         decoded = [sp.DecodeIds(ids) for ids in pred_ids]

#         pred_id_seqs.extend(pred_ids)
#         refs.extend(batch['texts'])
#         hyps.extend(decoded)
#         langs.extend(batch.get('langs', ['unknown'] * len(decoded)))

#         del mel, teacher_h, teacher_lens, tokens, tok_lens, mel_lens
#         del ctc_sample_weights
#         del kl_feat, ctc_logits, logits32, blank_log_probs, valid_frame_mask
#         del loss_kl, loss_ctc, loss

#     student.train()
#     n = max(loss_sample_count, 1)
#     norm_refs, norm_hyps = normalized_metric_texts(refs, hyps)
#     collapse_thresholds = collapse_thresholds or {}
#     collapse = compute_collapse_metrics(
#         pred_id_seqs, BLANK,
#         frame_argmax_ids=frame_argmax_seqs,
#         refs=refs, hyps=hyps, sp_model=sp,
#         blank_probability_pct=blank_probability_sum / max(blank_probability_n, 1),
#         **collapse_thresholds,
#     )
#     if a_ctc <= 0:
#         # A random/reset CTC head is irrelevant during KD-only warmup and
#         # must not block saving the best feature-distillation checkpoint.
#         collapse['collapsed'] = False
#         collapse['reason'] = 'ctc_disabled'
#     per_lang = per_language_metrics(refs, hyps, langs)
#     macro_cer, macro_languages = macro_language_cer(
#         per_lang, min_samples=C.macro_metric_min_samples)
#     return {
#         'loss':     total_loss / n,
#         'loss_kl':  total_kl / n,
#         'loss_ctc': total_ctc / n,
#         'wer':      compute_wer(norm_refs, norm_hyps),
#         'cer':      compute_cer(norm_refs, norm_hyps),
#         'wer_raw':  compute_wer(refs, hyps),
#         'cer_raw':  compute_cer(refs, hyps),
#         'per_lang': per_lang,
#         'macro_cer': macro_cer,
#         'macro_languages': macro_languages,
#         'collapse': collapse,
#         'n_samples': len(refs),
#     }


# def _source_checkpoint_candidates(spec):
#     """Return (path, required_tag) pairs in source-preference order."""
#     stages = []
#     if spec.source_checkpoint_stage:
#         stages.append((spec.source_checkpoint_stage, spec.source_checkpoint_tag))
#     stages.extend((stage, None) for stage in (spec.fallback_source_stages or ()))
#     out = []
#     for stage, required_tag in stages:
#         out.extend((p, required_tag) for p in _load_candidates(
#             stage, suffix=spec.source_checkpoint_suffix))
#         if spec.fallback_source_suffix != spec.source_checkpoint_suffix:
#             out.extend((p, required_tag) for p in _load_candidates(
#                 stage, suffix=spec.fallback_source_suffix))
#     return out


# def _is_readonly_checkpoint_path(path):
#     if not path:
#         return False
#     load_dirs = getattr(C, 'ckpt_load_dirs', None)
#     if load_dirs is None:
#         load_dirs = [getattr(C, 'ckpt_load_dir', None)]
#     path_abs = os.path.abspath(path)
#     for load_dir in load_dirs:
#         if not load_dir or load_dir == C.ckpt_dir:
#             continue
#         try:
#             load_root = os.path.abspath(load_dir)
#             if os.path.commonpath([path_abs, load_root]) == load_root:
#                 return True
#         except ValueError:
#             continue
#     return False


# def try_load_compatible_checkpoint(path_or_paths, *, expected_tag=None,
#                                    purpose='checkpoint'):
#     """Try every candidate, skipping stale recipe tags instead of stopping.

#     The old code loaded the first existing path, rejected its tag, and never
#     tried the remaining candidates. A stale writable file could therefore hide
#     a valid uploaded checkpoint (or vice versa).
#     """
#     candidates = [path_or_paths] if isinstance(path_or_paths, str) else list(path_or_paths)
#     for path in candidates:
#         state, loaded_from = try_load_full_checkpoint(path)
#         if state is None:
#             continue
#         found_tag = state.get('resume_tag')
#         if expected_tag is not None:
#             expected_tags = ({expected_tag} if isinstance(expected_tag, str)
#                              else set(expected_tag))
#             if found_tag not in expected_tags:
#                 location = 'read-only' if _is_readonly_checkpoint_path(loaded_from) else 'writable'
#                 print(
#                     f'Ignoring {location} {purpose} {loaded_from}: '
#                     f'resume_tag={found_tag!r}, expected one of '
#                     f'{sorted(expected_tags)!r}.')
#                 continue
#         return state, loaded_from
#     return None, None


# def _load_stage_source_weights(student, spec):
#     candidates = _source_checkpoint_candidates(spec)
#     prev = prev_src = None
#     for path, required_tag in candidates:
#         prev, prev_src = try_load_compatible_checkpoint(
#             path, expected_tag=required_tag,
#             purpose=f'{spec.name} source checkpoint')
#         if prev is not None:
#             break
#     if prev is None:
#         msg = f'[{spec.name}] No compatible source checkpoint found in {candidates}'
#         if spec.require_source_checkpoint:
#             raise FileNotFoundError(
#                 msg + '\nAttach the expected Kaggle checkpoint dataset or run '
#                 'the required upstream recovery stage first.')
#         print(msg + '; starting from current initialization.')
#         return

#     sd = _normalize_state_dict(prev['model'])
#     allowed_missing = ()
#     if spec.reset_ctc_head:
#         sd = {k: v for k, v in sd.items() if not k.startswith('ctc_head.')}
#         allowed_missing = ('ctc_head.',)
#     elif any(k.startswith('ctc_head.') for k in sd):
#         _require_checkpoint_tokenizer(prev, prev_src)
#     _load_model_tolerant(_unwrap(student), sd, allowed_missing_prefixes=allowed_missing)
#     print(f'Loaded compatible source weights for {spec.name} from {prev_src}')


# def _reset_ctc_head(student, blank_bias):
#     head = _unwrap(student).ctc_head
#     head.reset_parameters()
#     with torch.no_grad():
#         head.bias.zero_()
#         head.bias[BLANK] = float(blank_bias)
#     print(f'Reset ctc_head to fresh init; blank bias = {blank_bias}')


# def train_stage(student, dataset, spec, val_dataset=None):
#     """Train one StageSpec with collapse-aware validation and best checkpoints."""
#     # The launch cell starts this after smoke; retain a safe direct-call path
#     # for notebook debugging and ad-hoc recovery runs.
#     if SESSION_START is None:
#         start_training_budget()
#     stage_name = spec.name
#     _core = student.module if isinstance(student, torch.nn.DataParallel) else student
#     for p in _core.parameters():
#         p.requires_grad = True
#     trainable = sum(p.numel() for p in _core.parameters() if p.requires_grad)
#     print(f'[{stage_name}] {trainable/1e6:.1f}M trainable parameters')
#     _kl_target = (
#         spec.a_kl if spec.kl_end_weight is None else spec.kl_end_weight)
#     print(f'[{stage_name}] loss = '
#           f'{spec.a_kl}->{_kl_target} * loss_kl + '
#           f'{spec.ctc_start_weight}->{spec.ctc_end_weight} * loss_ctc')
#     if spec.kl_decay_steps:
#         print(f'[{stage_name}] KD hold={spec.kl_hold_steps} update(s), '
#               f'decay={spec.kl_decay_steps} update(s)')

#     max_mel_frames = None if spec.max_seconds is None else int(spec.max_seconds * 100)
#     _tmp_sampler = ShardBucketBatchSampler(
#         dataset, batch_size=spec.bs, generator=None, drop_last=True,
#         merge_tails=C.merge_shard_tails,
#         language_temperature=C.train_language_temperature,
#         excluded_languages=C.train_excluded_languages,
#     )
#     steps_per_epoch = len(_tmp_sampler) // spec.ga
#     print(f'[{stage_name}] sampler language counts: '
#           f'{_tmp_sampler.language_counts}')
#     print(f'[{stage_name}] sampler language repeats '
#           f'(temperature={C.train_language_temperature}): '
#           f'{_tmp_sampler.language_repeats}')
#     if _tmp_sampler.excluded_language_counts:
#         print(
#             f'[{stage_name}] TRAIN-ONLY excluded language samples: '
#             f'{dict(_tmp_sampler.excluded_language_counts)}')
#     print(f'[{stage_name}] {len(_tmp_sampler):,} micro-batches/epoch, '
#           f'{steps_per_epoch:,} optimizer updates/epoch')
#     del _tmp_sampler
#     total_steps = max(1, spec.epochs * steps_per_epoch)

#     if spec.ctc_head_lr is not None:
#         ctc_params = list(_core.ctc_head.parameters())
#         ctc_param_ids = {id(p) for p in ctc_params}
#         backbone_params = [
#             p for p in _core.parameters()
#             if p.requires_grad and id(p) not in ctc_param_ids
#         ]
#         optimizer = torch.optim.AdamW(
#             [
#                 {'params': backbone_params, 'lr': spec.lr},
#                 {'params': ctc_params, 'lr': spec.ctc_head_lr},
#             ],
#             weight_decay=0.01,
#         )
#         print(f'[{stage_name}] optimizer lr: backbone={spec.lr:g}, '
#               f'ctc_head={spec.ctc_head_lr:g}')
#     else:
#         optimizer = torch.optim.AdamW(
#             filter(lambda p: p.requires_grad, student.parameters()),
#             lr=spec.lr, weight_decay=0.01,
#         )

#     warmup_steps = (
#         C.warmup if spec.warmup_steps is None else int(spec.warmup_steps)
#     )

#     # A resumed stage may change micro-batch shape, merge shard tails, or add
#     # language repeats. Anchor a fresh cosine tail at the loaded LR so the
#     # schedule stays continuous and reaches zero at the new planned end.
#     resume_schedule = None

#     def lr_lambda(step):
#         if resume_schedule is not None and step >= resume_schedule['anchor_step']:
#             span = max(
#                 resume_schedule['end_step'] - resume_schedule['anchor_step'], 1)
#             progress = (step - resume_schedule['anchor_step']) / span
#             progress = min(1.0, max(0.0, progress))
#             return (resume_schedule['anchor_factor'] * 0.5 *
#                     (1 + math.cos(math.pi * progress)))
#         if step < warmup_steps:
#             return step / max(warmup_steps, 1)
#         # A batch-shape migration may safely replay the current partial epoch.
#         # Clamp at the schedule horizon so cosine decay stays at zero instead
#         # of rebounding if global_step slightly exceeds total_steps.
#         progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
#         progress = min(1.0, max(0.0, progress))
#         return 0.5 * (1 + math.cos(math.pi * progress))

#     scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
#     scaler = torch.amp.GradScaler('cuda')

#     ckpt_path = _save_path(stage_name)
#     best_path = _save_path(stage_name, 'best')
#     start_epoch = 0
#     start_batch_idx = 0
#     global_step = 0
#     best_loss = float('inf')
#     last_train_loss = None
#     best_val_metric = float('inf')
#     best_tuple = (float('inf'), float('inf'))
#     epochs_since_improvement = 0
#     es_enabled = getattr(C, 'es_enabled', True) and (val_dataset is not None)
#     es_patience = getattr(C, 'es_patience', 3)
#     es_min_delta = getattr(C, 'es_min_delta', 1e-4)
#     canary_baseline_loss = None
#     canary_done = False

#     _resume_candidates = []
#     if spec.allow_resume:
#         _resume_candidates = _load_candidates(
#             stage_name, include_readonly=spec.allow_readonly_resume)
#     _resume_expected_tags = spec.resume_source_tags or spec.resume_tag
#     ckpt, ckpt_src = try_load_compatible_checkpoint(
#         _resume_candidates, expected_tag=_resume_expected_tags,
#         purpose=f'{stage_name} resume checkpoint')
#     if ckpt is not None:
#         if spec.ctc_start_weight > 0 or spec.ctc_end_weight > 0:
#             _require_checkpoint_tokenizer(ckpt, ckpt_src)
#             _require_checkpoint_target_recipe(ckpt, ckpt_src)
#         _load_model_tolerant(_unwrap(student), _normalize_state_dict(ckpt['model']))
#         optimizer.load_state_dict(ckpt['optimizer'])
#         scheduler.load_state_dict(ckpt['scheduler'])
#         scaler.load_state_dict(ckpt['scaler'])
#         start_epoch = ckpt['epoch']
#         start_batch_idx = ckpt.get('batch_idx', 0)
#         # The effective batch remains 32 across bs=2/GA=16, bs=8/GA=4,
#         # and bs=32/GA=1, so optimizer/scaler state is valid. A changed micro-
#         # batch index is not, so restart only the current partial epoch and
#         # re-anchor the cosine tail below to the new sampler horizon.
#         saved_bs = ckpt.get('micro_batch_size')
#         saved_config = ckpt.get('config')
#         if saved_bs is None:
#             config_field = (
#                 'fr_bs' if stage_name.endswith('_kd') else 'un_bs')
#             saved_bs = getattr(saved_config, config_field, spec.bs)
#         saved_sampler_recipe = {
#             'micro_batch_size': int(saved_bs),
#             'merge_shard_tails': bool(getattr(
#                 saved_config, 'merge_shard_tails', True)),
#             'language_temperature': float(getattr(
#                 saved_config, 'train_language_temperature', 0.70)),
#             'excluded_languages': tuple(sorted(
#                 canonical_language_name(lang) for lang in getattr(
#                     saved_config, 'train_excluded_languages', ()))),
#         }
#         active_sampler_recipe = {
#             'micro_batch_size': int(spec.bs),
#             'merge_shard_tails': bool(C.merge_shard_tails),
#             'language_temperature': float(C.train_language_temperature),
#             'excluded_languages': tuple(sorted(
#                 canonical_language_name(lang)
#                 for lang in C.train_excluded_languages)),
#         }
#         if start_batch_idx and saved_sampler_recipe != active_sampler_recipe:
#             print(
#                 f'  Training sampler changed; restarting partial epoch '
#                 f'{start_epoch + 1} at batch 0 for complete sample coverage.\n'
#                 f'    saved={saved_sampler_recipe}\n'
#                 f'    active={active_sampler_recipe}\n'
#                 '  Optimizer/global_step remain resumed.')
#             start_batch_idx = 0
#         global_step = ckpt['global_step']
#         best_loss = ckpt.get('best_loss', float('inf'))
#         last_train_loss = ckpt.get('last_train_loss', ckpt.get('loss'))
#         best_val_metric = ckpt.get('best_val_metric', float('inf'))
#         epochs_since_improvement = ckpt.get('epochs_since_improvement', 0)
#         saved_best_metric_name = ckpt.get('best_metric_name')
#         if (start_epoch < spec.epochs and spec.best_metric == 'macro_cer' and
#                 saved_best_metric_name != spec.best_metric):
#             # v2 stored sample-weighted CER. It is numerically incomparable to
#             # v3 macro CER, so keep weights/optimizer/scheduler but restart only
#             # model-selection and early-stop state.
#             print(
#                 f'  Model-selection metric changed '
#                 f'{saved_best_metric_name!r} -> {spec.best_metric!r}; '
#                 'resetting best/early-stop state while preserving training.')
#             best_loss = float('inf')
#             best_val_metric = float('inf')
#             epochs_since_improvement = 0
#         canary_baseline_loss = ckpt.get('canary_baseline_loss')
#         canary_done = bool(ckpt.get(
#             'canary_done', spec.canary_updates > 0 and global_step >= spec.canary_updates))
#         best_tuple = (best_val_metric, best_loss)
#         _last_msg = 'n/a' if last_train_loss is None else f'{float(last_train_loss):.4f}'
#         print(f'Resumed {stage_name} from {ckpt_src}')
#         print(f'  epoch {start_epoch+1}/{spec.epochs}, batch_idx={start_batch_idx}, '
#               f'step {global_step}, best_metric={best_val_metric:.4f}, '
#               f'best_loss={best_loss:.4f}, last_train_loss={_last_msg}')
#         if start_epoch < spec.epochs:
#             remaining_updates = max(
#                 1,
#                 (spec.epochs - start_epoch) * steps_per_epoch -
#                 start_batch_idx // max(spec.ga, 1),
#             )
#             current_factor = (
#                 scheduler.get_last_lr()[0] /
#                 max(float(scheduler.base_lrs[0]), 1e-12)
#             )
#             raw_factor = float(current_factor)
#             floor_factor = float(getattr(
#                 C, 'resume_lr_floor_factor', 1.0 / 3.0))
#             if (getattr(C, 'restart_exhausted_schedule', True) and
#                     current_factor < floor_factor):
#                 current_factor = floor_factor
#                 restarted_lrs = []
#                 for group, base_lr in zip(
#                         optimizer.param_groups, scheduler.base_lrs):
#                     group['lr'] = float(base_lr) * current_factor
#                     restarted_lrs.append(group['lr'])
#                 scheduler._last_lr = restarted_lrs
#                 # A previously exhausted schedule can also carry a saturated
#                 # early-stop counter. Give the bounded continuation a fair
#                 # opportunity without discarding the historical best metric.
#                 epochs_since_improvement = 0
#                 print(
#                     f'  Soft LR restart: exhausted factor {raw_factor:.3e} '
#                     f'-> {current_factor:.4f}; lrs={restarted_lrs}.')
#             resume_schedule = {
#                 'anchor_step': int(global_step),
#                 'anchor_factor': float(current_factor),
#                 'end_step': int(global_step + remaining_updates),
#             }
#             print(
#                 f'  Re-anchored cosine schedule at lr factor '
#                 f'{current_factor:.4f}; zero at step '
#                 f'{resume_schedule["end_step"]:,} after '
#                 f'{remaining_updates:,} planned updates.')
#     else:
#         _downstream_candidates = []
#         for _entry in getattr(spec, 'skip_if_checkpoint_stages', ()) or ():
#             if isinstance(_entry, (tuple, list)):
#                 _stage, _required_tag = _entry
#             else:
#                 _stage, _required_tag = _entry, None
#             for _suffix in ('latest', 'best'):
#                 _downstream_candidates.extend(
#                     (p, _required_tag) for p in _load_candidates(
#                         _stage, suffix=_suffix, include_readonly=True))
#         _downstream = _downstream_src = None
#         for _path, _required_tag in _downstream_candidates:
#             _downstream, _downstream_src = try_load_compatible_checkpoint(
#                 _path, expected_tag=_required_tag,
#                 purpose=f'{stage_name} downstream proof')
#             if _downstream is not None:
#                 break
#         if _downstream is not None:
#             _loss = _downstream.get('loss', _downstream.get('best_loss', 0.0))
#             print(f'[{stage_name}] Found compatible downstream checkpoint '
#                   f'{_downstream_src}; treating this stage as already complete.')
#             return float(_loss), True

#         _load_stage_source_weights(student, spec)
#         if spec.reset_ctc_head:
#             _reset_ctc_head(student, spec.ctc_blank_bias)

#     if start_epoch >= spec.epochs:
#         print(f'[{stage_name}] Already complete (epoch {start_epoch}/{spec.epochs})')
#         return best_loss, True

#     def _kl_weight():
#         if spec.kl_end_weight is None or spec.kl_decay_steps <= 0:
#             return spec.a_kl
#         if global_step <= spec.kl_hold_steps:
#             return spec.a_kl
#         frac = min(
#             1.0,
#             (global_step - spec.kl_hold_steps) /
#             max(spec.kl_decay_steps, 1),
#         )
#         return spec.a_kl + (spec.kl_end_weight - spec.a_kl) * frac

#     def _ctc_weight():
#         if spec.ctc_warmup_steps > 0 and global_step < spec.ctc_warmup_steps:
#             frac = global_step / max(spec.ctc_warmup_steps, 1)
#             return spec.ctc_start_weight + (spec.ctc_end_weight - spec.ctc_start_weight) * frac
#         return spec.ctc_end_weight

#     def _checkpoint_batch_idx(batch_idx):
#         completed = batch_idx + 1
#         return completed - (completed % spec.ga)

#     def _report_loss():
#         if math.isfinite(float(best_loss)):
#             return float(best_loss)
#         if last_train_loss is not None and math.isfinite(float(last_train_loss)):
#             return float(last_train_loss)
#         return float('inf')

#     validation_probe, validation_probe_counts = (
#         _stratified_validation_subset(
#             val_dataset, spec.validation_probe_samples,
#             seed=C.seed + 17_017)
#         if val_dataset is not None else (None, {}))
#     if validation_probe_counts:
#         print(
#             f'[{stage_name}] validation probe: '
#             f'{len(validation_probe):,} samples, '
#             f'per-language={validation_probe_counts}')

#     def _run_validation(label, epoch_for_log):
#         use_full_validation = label == 'val'
#         eval_dataset = (
#             val_dataset if use_full_validation else validation_probe)
#         val_max_batches = (
#             int(spec.validation_max_batches)
#             if use_full_validation else None)
#         # Keep the validation objective fixed at the final curriculum weights
#         # so losses remain comparable while training weights change.
#         eval_kl_weight = (
#             spec.a_kl if spec.kl_end_weight is None else spec.kl_end_weight)
#         scope = (
#             f'up to {val_max_batches} shard-batches'
#             if val_max_batches is not None else
#             f'all {len(eval_dataset):,} probe samples')
#         print(f'  Running {label} validation on {scope}...')
#         val_metrics = evaluate(
#             student, eval_dataset,
#             max_seconds=spec.max_seconds,
#             bs=C.validation_batch_size,
#             max_batches=val_max_batches,
#             a_kl=eval_kl_weight,
#             a_ctc=spec.ctc_end_weight,
#             collapse_thresholds=_collapse_thresholds(spec),
#         )
#         collapse = val_metrics['collapse']
#         print(f'  {label} val - loss={val_metrics["loss"]:.4f} '
#               f'wer_norm={val_metrics["wer"]*100:.2f}% '
#               f'cer_norm={val_metrics["cer"]*100:.2f}% '
#               f'macro_cer={val_metrics["macro_cer"]*100:.2f}% '
#               f'cer_raw={val_metrics["cer_raw"]*100:.2f}% '
#               f'empty={collapse["empty_hypothesis_pct"]*100:.1f}% '
#               f'blank={collapse["blank_frame_pct"]*100:.1f}% '
#               f'blank_p={collapse["blank_probability_pct"]*100:.1f}% '
#               f'top={collapse["top_ratio"]*100:.1f}% '
#               f'collapsed={collapse["collapsed"]} ({collapse["reason"]}) '
#               f'(n={val_metrics["n_samples"]})')
#         payload = {
#             f'{stage_name}/{label}/loss': val_metrics['loss'],
#             f'{stage_name}/{label}/loss_kl': val_metrics['loss_kl'],
#             f'{stage_name}/{label}/loss_ctc': val_metrics['loss_ctc'],
#             f'{stage_name}/{label}/wer': val_metrics['wer'],
#             f'{stage_name}/{label}/cer': val_metrics['cer'],
#             f'{stage_name}/{label}/macro_cer': val_metrics['macro_cer'],
#             f'{stage_name}/{label}/wer_raw': val_metrics['wer_raw'],
#             f'{stage_name}/{label}/cer_raw': val_metrics['cer_raw'],
#             f'{stage_name}/{label}/collapsed': int(collapse['collapsed']),
#             f'{stage_name}/{label}/blank_frame_pct': collapse['blank_frame_pct'],
#             f'{stage_name}/{label}/blank_probability_pct': collapse['blank_probability_pct'],
#             f'{stage_name}/{label}/empty_hypothesis_pct': collapse['empty_hypothesis_pct'],
#             f'{stage_name}/{label}/top_token_ratio': collapse['top_ratio'],
#             f'{stage_name}/{label}/avg_pred_token_len': collapse['avg_pred_token_len'],
#             f'{stage_name}/{label}/avg_ref_token_len': collapse['avg_ref_token_len'],
#             f'{stage_name}/{label}/epoch': epoch_for_log,
#         }
#         wandb_run.log(payload)
#         return val_metrics

#     def _save_best_from_metrics(val_metrics, epoch_marker, label):
#         nonlocal best_loss, best_val_metric, best_tuple
#         current_metric, best_source = _metric_for_best(
#             val_metrics, spec, float('inf'))
#         current_loss = float(val_metrics['loss'])
#         collapsed = _is_collapsed(val_metrics)
#         improved = (
#             current_metric < best_tuple[0] - es_min_delta or
#             (abs(current_metric - best_tuple[0]) <= es_min_delta and
#              current_loss < best_tuple[1] - es_min_delta)
#         )
#         if improved and not collapsed:
#             best_loss = current_loss
#             best_tuple = (current_metric, current_loss)
#             best_val_metric = current_metric
#             save_best_model_only(
#                 best_path, student, stage_name=stage_name,
#                 resume_tag=spec.resume_tag, epoch=epoch_marker,
#                 global_step=global_step, loss=best_loss,
#                 metric={
#                     spec.best_metric: current_metric,
#                     'source': best_source,
#                     'normalization': 'NFKC+casefold+punctuation-strip',
#                 },
#                 collapse=val_metrics.get('collapse'),
#                 evaluation_lineage_clean=spec.evaluation_lineage_clean,
#             )
#             print(f'  * New best from {label}: {best_source}='
#                   f'{current_metric:.4f}, loss={best_loss:.4f}')
#         elif collapsed:
#             print(f'  {label}: not saving best because validation is collapsed.')
#         return improved, current_metric, current_loss, collapsed, best_source

#     # Establish a like-for-like validation baseline after source loading and
#     # CTC-head reset. A single all-blank snapshot is common early in CTC; only a
#     # collapsed snapshot with insufficient loss improvement is fail-fast worthy.
#     if (spec.canary_updates and not canary_done and val_dataset is not None and
#             canary_baseline_loss is None and time_left_seconds() > 600):
#         baseline = _run_validation('canary_baseline', start_epoch)
#         canary_baseline_loss = float(baseline['loss'])
#         print(f'  Canary baseline loss={canary_baseline_loss:.4f}')

#     student.train()
#     last_save = time.monotonic()
#     save_interval_s = CKPT_EVERY_MINUTES * 60
#     last_validation_step = global_step

#     for epoch in range(start_epoch, spec.epochs):
#         epoch_had_improvement = False
#         epoch_gen = torch.Generator().manual_seed(C.seed * 1_000_003 + epoch)
#         train_sampler = ShardBucketBatchSampler(
#             dataset, batch_size=spec.bs, generator=epoch_gen, drop_last=True,
#             merge_tails=C.merge_shard_tails,
#             language_temperature=C.train_language_temperature,
#             excluded_languages=C.train_excluded_languages,
#         )
#         loader = DataLoader(
#             dataset, batch_sampler=train_sampler,
#             num_workers=C.workers, persistent_workers=(C.workers > 0),
#             pin_memory=True,
#             collate_fn=lambda b: collate_kd(
#                 b, max_mel_frames=max_mel_frames, allow_crop=False),
#         )

#         skip_batches = start_batch_idx if epoch == start_epoch else 0
#         if skip_batches:
#             print(f'  Skipping first {skip_batches} batches of epoch {epoch+1} (resume)')
#         start_batch_idx = 0

#         epoch_loss = epoch_kl = epoch_ctc = 0.0
#         n_batches = 0
#         optimizer.zero_grad(set_to_none=True)
#         pbar = tqdm(loader, desc=f'[{stage_name}] E{epoch+1}/{spec.epochs}',
#                     unit='batch', leave=True, initial=skip_batches,
#                     total=len(loader))
#         time_up = False

#         for batch_idx, batch in enumerate(loader):
#             if batch_idx < skip_batches:
#                 pbar.update(0)
#                 continue
#             pbar.update(1)

#             mel = batch['mel'].to(device, non_blocking=True)
#             teacher_h = (batch['teacher_h'].to(device, non_blocking=True)
#                          if spec.a_kl > 0 else None)
#             teacher_lens = (batch['teacher_lens'].to(device, non_blocking=True)
#                             if spec.a_kl > 0 else None)
#             tokens = batch['tokens'].to(device, non_blocking=True)
#             tok_lens = batch['tok_lens'].to(device, non_blocking=True)
#             mel_lens = batch['mel_lens'].to(device, non_blocking=True)

#             with torch.amp.autocast('cuda', dtype=torch.float16):
#                 ctc_logits, kl_feat, enc_lens = student(mel, mel_lens)
#                 loss_kl = (kd_feature_loss(
#                     kl_feat, teacher_h, enc_lens, teacher_lens=teacher_lens)
#                     if spec.a_kl > 0 else
#                     kl_feat.new_zeros((), dtype=torch.float32))
#                 kl_w = _kl_weight()
#                 ctc_w = _ctc_weight()
#                 if ctc_w > 0:
#                     bad = describe_invalid_ctc_batch(
#                         batch, enc_lens, tok_lens, tokens)
#                     if bad:
#                         loss_log.log('invalid_ctc_batch', stage=stage_name,
#                                      epoch=epoch, batch_idx=batch_idx)
#                         raise RuntimeError(
#                             'Invalid CTC batch; see printed examples above.')
#                     ctc_sample_weights = ctc_confidence_weights(
#                         batch['teacher_confidence'], ctc_logits.device)
#                     loss_ctc = ctc_loss_fn(
#                         ctc_logits, tokens, enc_lens, tok_lens,
#                         sample_weights=ctc_sample_weights)
#                 else:
#                     ctc_sample_weights = None
#                     loss_ctc = ctc_logits.new_zeros((), dtype=torch.float32)
#                 raw_loss = kl_w * loss_kl + ctc_w * loss_ctc
#                 if not (torch.isfinite(loss_kl) and torch.isfinite(loss_ctc) and
#                         torch.isfinite(raw_loss)):
#                     loss_log.log(
#                         'nonfinite_loss', stage=stage_name, epoch=epoch,
#                         batch_idx=batch_idx, loss=float(raw_loss.detach().float().cpu()),
#                         loss_kl=float(loss_kl.detach().float().cpu()),
#                         loss_ctc=float(loss_ctc.detach().float().cpu()),
#                         ctc_weight=ctc_w,
#                     )
#                     raise RuntimeError(
#                         f'Non-finite loss in {stage_name}: '
#                         f'loss={raw_loss.detach().float().cpu().item()}, '
#                         f'kl={loss_kl.detach().float().cpu().item()}, '
#                         f'ctc={loss_ctc.detach().float().cpu().item()}, '
#                         f'ctc_weight={ctc_w}')
#                 last_train_loss = float(raw_loss.detach().float().cpu())
#                 loss = raw_loss / spec.ga

#             scaler.scale(loss).backward()

#             if (batch_idx + 1) % spec.ga == 0:
#                 if C.gc_norm:
#                     scaler.unscale_(optimizer)
#                     torch.nn.utils.clip_grad_norm_(student.parameters(), C.gc_norm)
#                 scale_before = scaler.get_scale()
#                 scaler.step(optimizer)
#                 scaler.update()
#                 optimizer.zero_grad(set_to_none=True)
#                 if scaler.get_scale() >= scale_before:
#                     scheduler.step()
#                     global_step += 1
#                     step_loss = loss.item() * spec.ga
#                     wandb_run.log({
#                         f'{stage_name}/train/loss': step_loss,
#                         f'{stage_name}/train/loss_kl': loss_kl.item(),
#                         f'{stage_name}/train/loss_ctc': loss_ctc.item(),
#                         f'{stage_name}/train/kl_weight': kl_w,
#                         f'{stage_name}/train/ctc_weight': ctc_w,
#                         f'{stage_name}/train/ctc_confidence_weight_mean': (
#                             float(ctc_sample_weights.mean().detach().cpu())
#                             if ctc_sample_weights is not None else 1.0),
#                         f'{stage_name}/train/lr': scheduler.get_last_lr()[0],
#                         f'{stage_name}/train/grad_scale': scaler.get_scale(),
#                         f'{stage_name}/train/gpu_mem_gb': torch.cuda.memory_allocated() / 1e9,
#                         f'{stage_name}/train/elapsed_h': training_elapsed_seconds() / 3600,
#                         f'{stage_name}/train/global_step': global_step,
#                         f'{stage_name}/train/epoch_frac': epoch + (batch_idx + 1) / max(len(loader), 1),
#                     })
#                     loss_log.log(
#                         'train_step', stage=stage_name, epoch=epoch,
#                         batch_idx=batch_idx, global_step=global_step,
#                         loss=step_loss, loss_kl=loss_kl.item(),
#                         loss_ctc=loss_ctc.item(), kl_weight=kl_w,
#                         ctc_weight=ctc_w,
#                         ctc_confidence_weight_mean=(
#                             float(ctc_sample_weights.mean().detach().cpu())
#                             if ctc_sample_weights is not None else 1.0),
#                         lr=scheduler.get_last_lr()[0],
#                         grad_scale=scaler.get_scale(),
#                         gpu_mem_gb=torch.cuda.memory_allocated() / 1e9,
#                     )
#                 else:
#                     loss_log.log('scaler_skip', stage=stage_name, epoch=epoch,
#                                  batch_idx=batch_idx, scale_before=scale_before,
#                                  scale_after=scaler.get_scale())
#                     if batch_idx % C.log_every == 0:
#                         pbar.write(f'  [scaler skip] inf/nan grads; scale '
#                                    f'{scale_before:g} -> {scaler.get_scale():g}')

#             epoch_loss += loss.item() * spec.ga
#             epoch_kl += loss_kl.item()
#             epoch_ctc += loss_ctc.item()
#             n_batches += 1

#             del mel, teacher_h, teacher_lens, tokens, tok_lens, mel_lens
#             del ctc_sample_weights
#             del kl_feat, ctc_logits, loss_kl, loss_ctc, loss

#             now = time.monotonic()

#             if (spec.canary_updates and not canary_done and val_dataset is not None and
#                     global_step >= spec.canary_updates and time_left_seconds() > 300):
#                 canary_done = True
#                 canary = _run_validation('canary', epoch + (batch_idx + 1) / max(len(loader), 1))
#                 relative_drop = None
#                 if (canary_baseline_loss is not None and
#                         math.isfinite(canary_baseline_loss)):
#                     relative_drop = (
#                         canary_baseline_loss - float(canary['loss'])) / max(
#                             abs(canary_baseline_loss), 1e-8)
#                 has_progress = (
#                     relative_drop is not None and
#                     relative_drop >= spec.canary_min_relative_loss_drop)
#                 loss_log.log(
#                     'canary', stage=stage_name, global_step=global_step,
#                     collapsed=canary['collapse']['collapsed'],
#                     collapse_reason=canary['collapse']['reason'],
#                     val_wer=canary['wer'], val_loss=canary['loss'],
#                     baseline_loss=canary_baseline_loss,
#                     relative_loss_drop=relative_drop,
#                 )
#                 if _is_collapsed(canary) and has_progress:
#                     pbar.write(
#                         f'  Canary output is collapsed but validation loss improved '
#                         f'{relative_drop*100:.1f}% from baseline; continuing.')
#                 elif spec.fail_fast_on_collapse and _is_collapsed(canary):
#                     # Save the diagnostic state before raising; otherwise a
#                     # fail-fast canary discards the only evidence from the run.
#                     save_full_checkpoint(
#                         ckpt_path, model=student, optimizer=optimizer,
#                         scheduler=scheduler, scaler=scaler,
#                         stage_name=stage_name, epoch=epoch,
#                         batch_idx=_checkpoint_batch_idx(batch_idx),
#                         global_step=global_step, best_loss=best_loss,
#                         best_val_metric=best_val_metric,
#                         best_metric_name=spec.best_metric,
#                         epochs_since_improvement=epochs_since_improvement,
#                         last_train_loss=last_train_loss,
#                         resume_tag=spec.resume_tag,
#                         canary_baseline_loss=canary_baseline_loss,
#                         canary_done=canary_done,
#                         evaluation_lineage_clean=spec.evaluation_lineage_clean)
#                     raise RuntimeError(
#                         f'[{stage_name}] Canary collapsed without sufficient loss '
#                         f'progress (relative_drop={relative_drop}, '
#                         f'metrics={canary["collapse"]}).')

#             # A T4 epoch is several hours long. Refresh the inference-best
#             # checkpoint inside the epoch so an 8h session never evaluates a
#             # model thousands of updates behind `latest`.
#             if (spec.validation_interval_updates > 0 and
#                     val_dataset is not None and
#                     global_step - last_validation_step >=
#                     spec.validation_interval_updates and
#                     time_left_seconds() > 600):
#                 midval = _run_validation(
#                     'midval',
#                     epoch + (batch_idx + 1) / max(len(loader), 1),
#                 )
#                 if spec.midval_selects_best:
#                     mid_improved, mid_metric, mid_loss, mid_collapsed, _ = (
#                         _save_best_from_metrics(
#                             midval,
#                             epoch + (batch_idx + 1) / max(len(loader), 1),
#                             'midval',
#                         )
#                     )
#                     epoch_had_improvement = (
#                         epoch_had_improvement or mid_improved)
#                 else:
#                     mid_metric, _ = _metric_for_best(
#                         midval, spec, float('inf'))
#                     mid_loss = float(midval['loss'])
#                     mid_collapsed = _is_collapsed(midval)
#                     mid_improved = False
#                     print(
#                         '  midval probe is monitoring-only; full epoch-end '
#                         'validation selects best checkpoints.')
#                 last_validation_step = global_step
#                 loss_log.log(
#                     'midval', stage=stage_name, epoch=epoch,
#                     global_step=global_step, val_loss=mid_loss,
#                     val_metric=mid_metric, best_metric=spec.best_metric,
#                     improved=mid_improved, collapsed=mid_collapsed,
#                 )
#                 torch.cuda.empty_cache()
#                 student.train()

#             # Validation may take minutes; refresh the timestamp before the
#             # periodic/deadline checks so a canary cannot run past the Kaggle
#             # budget without writing a resumable checkpoint.
#             now = time.monotonic()
#             if now - last_save >= save_interval_s:
#                 save_full_checkpoint(
#                     ckpt_path, model=student, optimizer=optimizer,
#                     scheduler=scheduler, scaler=scaler,
#                     stage_name=stage_name, epoch=epoch,
#                     batch_idx=_checkpoint_batch_idx(batch_idx),
#                     global_step=global_step, best_loss=best_loss,
#                     best_val_metric=best_val_metric,
#                         best_metric_name=spec.best_metric,
#                     epochs_since_improvement=epochs_since_improvement,
#                     last_train_loss=last_train_loss,
#                     resume_tag=spec.resume_tag,
#                     canary_baseline_loss=canary_baseline_loss,
#                     canary_done=canary_done,
#                         evaluation_lineage_clean=spec.evaluation_lineage_clean)
#                 last_save = now
#                 pbar.write(f'saved @ step {global_step} (time left: {fmt_hms(time_left_seconds())})')

#             if now >= SESSION_DEADLINE:
#                 save_full_checkpoint(
#                     ckpt_path, model=student, optimizer=optimizer,
#                     scheduler=scheduler, scaler=scaler,
#                     stage_name=stage_name, epoch=epoch,
#                     batch_idx=_checkpoint_batch_idx(batch_idx),
#                     global_step=global_step, best_loss=best_loss,
#                     best_val_metric=best_val_metric,
#                         best_metric_name=spec.best_metric,
#                     epochs_since_improvement=epochs_since_improvement,
#                     last_train_loss=last_train_loss,
#                     resume_tag=spec.resume_tag,
#                     canary_baseline_loss=canary_baseline_loss,
#                     canary_done=canary_done,
#                         evaluation_lineage_clean=spec.evaluation_lineage_clean)
#                 pbar.write(f'\nSession budget reached. Saved checkpoint to {ckpt_path}.')
#                 time_up = True
#                 break

#             if batch_idx % C.log_every == 0:
#                 pbar.set_postfix_str(
#                     f'loss={epoch_loss/max(n_batches, 1):.3f} '
#                     f'mem={torch.cuda.memory_allocated()/1e9:.1f}G '
#                     f'lr={scheduler.get_last_lr()[0]:.2e} '
#                     f'left={fmt_hms(time_left_seconds())}')

#         pbar.close()
#         if time_up:
#             return _report_loss(), False

#         avg_loss = epoch_loss / max(n_batches, 1)
#         avg_kl = epoch_kl / max(n_batches, 1)
#         avg_ctc = epoch_ctc / max(n_batches, 1)
#         print(f'E{epoch + 1} avg - loss={avg_loss:.4f} kl={avg_kl:.4f} ctc={avg_ctc:.4f}')

#         val_metrics = None
#         epoch_payload = {
#             f'{stage_name}/epoch_avg/loss': avg_loss,
#             f'{stage_name}/epoch_avg/loss_kl': avg_kl,
#             f'{stage_name}/epoch_avg/loss_ctc': avg_ctc,
#             f'{stage_name}/epoch_avg/epoch': epoch + 1,
#             f'{stage_name}/epoch_avg/global_step': global_step,
#         }

#         if val_dataset is not None and time_left_seconds() > 300:
#             val_metrics = _run_validation('val', epoch + 1)
#             collapse = val_metrics['collapse']
#             epoch_payload.update({
#                 f'{stage_name}/val/loss': val_metrics['loss'],
#                 f'{stage_name}/val/loss_kl': val_metrics['loss_kl'],
#                 f'{stage_name}/val/loss_ctc': val_metrics['loss_ctc'],
#                 f'{stage_name}/val/wer': val_metrics['wer'],
#                 f'{stage_name}/val/cer': val_metrics['cer'],
#                 f'{stage_name}/val/macro_cer': val_metrics['macro_cer'],
#                 f'{stage_name}/val/collapsed': int(collapse['collapsed']),
#                 f'{stage_name}/val/blank_frame_pct': collapse['blank_frame_pct'],
#                 f'{stage_name}/val/blank_probability_pct': collapse['blank_probability_pct'],
#                 f'{stage_name}/val/empty_hypothesis_pct': collapse['empty_hypothesis_pct'],
#                 f'{stage_name}/val/top_token_ratio': collapse['top_ratio'],
#                 f'{stage_name}/val/avg_pred_token_len': collapse['avg_pred_token_len'],
#                 f'{stage_name}/val/avg_ref_token_len': collapse['avg_ref_token_len'],
#                 f'{stage_name}/val/epoch': epoch + 1,
#             })
#             for _lang, _m in val_metrics.get('per_lang', {}).items():
#                 epoch_payload[f'{stage_name}/val/cer_{_lang}'] = _m['cer']
#             torch.cuda.empty_cache()

#         if val_metrics is not None:
#             improved, current_metric, current_loss, collapsed, best_source = (
#                 _save_best_from_metrics(
#                     val_metrics, epoch + 1, 'epoch-end validation'))
#             epoch_had_improvement = epoch_had_improvement or improved
#         else:
#             current_metric, best_source = _metric_for_best(
#                 None, spec, avg_loss)
#             current_loss = avg_loss
#             collapsed = False
#             improved = False

#         if es_enabled:
#             if epoch_had_improvement and not collapsed:
#                 epochs_since_improvement = 0
#                 print(f'  early-stop: best val/{spec.best_metric}='
#                       f'{best_val_metric:.5f} - counter reset')
#             else:
#                 epochs_since_improvement += 1
#                 suffix = 'collapsed' if collapsed else 'no improvement'
#                 print(f'  early-stop: {best_source}={current_metric:.5f} '
#                       f'(best={best_val_metric:.5f}) - {suffix} '
#                       f'{epochs_since_improvement}/{es_patience}')
#             epoch_payload[f'{stage_name}/val/best_{spec.best_metric}'] = best_val_metric
#             epoch_payload[f'{stage_name}/val/epochs_without_improvement'] = epochs_since_improvement

#         wandb_run.log(epoch_payload)
#         loss_log.log(
#             'epoch_end', stage=stage_name, epoch=epoch + 1,
#             global_step=global_step, train_loss=avg_loss,
#             train_loss_kl=avg_kl, train_loss_ctc=avg_ctc,
#             val_loss=epoch_payload.get(f'{stage_name}/val/loss'),
#             val_loss_kl=epoch_payload.get(f'{stage_name}/val/loss_kl'),
#             val_loss_ctc=epoch_payload.get(f'{stage_name}/val/loss_ctc'),
#             val_wer=epoch_payload.get(f'{stage_name}/val/wer'),
#             val_cer=epoch_payload.get(f'{stage_name}/val/cer'),
#             val_macro_cer=epoch_payload.get(f'{stage_name}/val/macro_cer'),
#             collapsed=collapsed,
#         )

#         save_full_checkpoint(
#             ckpt_path, model=student, optimizer=optimizer,
#             scheduler=scheduler, scaler=scaler,
#             stage_name=stage_name, epoch=epoch + 1,
#             batch_idx=0, global_step=global_step,
#             best_loss=best_loss,
#             best_val_metric=best_val_metric,
#                         best_metric_name=spec.best_metric,
#             epochs_since_improvement=epochs_since_improvement,
#                     last_train_loss=last_train_loss,
#                     resume_tag=spec.resume_tag,
#                     canary_baseline_loss=canary_baseline_loss,
#                     canary_done=canary_done,
#                         evaluation_lineage_clean=spec.evaluation_lineage_clean)
#         last_save = time.monotonic()

#         if spec.fail_fast_on_collapse and collapsed:
#             _drop = None
#             if canary_baseline_loss is not None and math.isfinite(canary_baseline_loss):
#                 _drop = (canary_baseline_loss - current_loss) / max(
#                     abs(canary_baseline_loss), 1e-8)
#             if _drop is None or _drop < spec.canary_min_relative_loss_drop:
#                 raise RuntimeError(
#                     f'[{stage_name}] Validation collapsed without loss progress; '
#                     f'relative_drop={_drop}.')
#             print(f'  Collapsed greedy output, but loss improved {_drop*100:.1f}% '
#                   'from baseline; continuing and letting early stopping decide.')

#         if es_enabled and epochs_since_improvement >= es_patience:
#             print(f'\n[{stage_name}] EARLY STOPPING at epoch {epoch+1}/{spec.epochs}.')
#             save_full_checkpoint(
#                 ckpt_path, model=student, optimizer=optimizer,
#                 scheduler=scheduler, scaler=scaler,
#                 stage_name=stage_name, epoch=spec.epochs,
#                 batch_idx=0, global_step=global_step,
#                 best_loss=best_loss,
#                 best_val_metric=best_val_metric,
#                         best_metric_name=spec.best_metric,
#                 epochs_since_improvement=epochs_since_improvement,
#                 early_stopped=True,
#                 last_train_loss=last_train_loss,
#                 resume_tag=spec.resume_tag,
#                 canary_baseline_loss=canary_baseline_loss,
#                 canary_done=canary_done,
#                 evaluation_lineage_clean=spec.evaluation_lineage_clean,
#             )
#             loss_log.log('early_stop', stage=stage_name, epoch=epoch + 1,
#                          best_loss=best_loss, best_val_metric=best_val_metric)
#             return best_loss, True

#     print(f'[{stage_name}] Done - best_loss={best_loss:.4f}\n')
#     return best_loss, True


# # Set this only after the entire definition/setup cell succeeds. The launch
# # cell checks it because Kaggle kernels can retain an older train_stage and
# # compute_collapse_metrics after the notebook source has been edited.
# TRAINING_DEFINITION_VERSION = 'scratch-quality-throughput-v7'
# print(f'Training definitions ready: {TRAINING_DEFINITION_VERSION}')


### Smoke test (run after training configuration — gates training process)

Runs ~50 gradient steps on a tiny subset to validate the fast-path Mamba kernels, memory budget, and loss trajectory before committing to a multi-hour run. Sets `_SMOKE_PASSED = True` on success; the training process cell refuses to start otherwise.


In [16]:
# # Smoke test: run after training config; gates the training process cell.
# # Checks kernels, data loading, memory, and loss before a long run.
# from torch.utils.data import Subset, DataLoader
# import statistics
# import time

# N_SMOKE = 50
# SMOKE_BS = C.un_bs
# # The real recovery schedule runs KD-only warmup then joint KD+CTC.
# # Smoke intentionally enables both terms so one short run validates both data
# # paths and both heads; C.a_kl is 0.0 and must not disable this diagnostic.
# SMOKE_A_KL = 1.0
# SMOKE_A_CTC = 0.30
# SMOKE_BACKBONE_LR = 3e-5
# SMOKE_CTC_HEAD_LR = 1e-3
# SMOKE_BLANK_BIAS = 0.0

# # ── Build (mirrors the training cell so the smoke reflects real conditions)
# print('Loading dataset...')
# ds = KDDataset(C.cache_dir, sp, C.mel_dir)
# flush()

# # Reuse persisted splits so train/val/test stay stable across sessions.
# SPLITS_PATH = os.path.join(C.ckpt_dir, 'splits.json')
# _val_size = max(50, min(2000, len(ds) // 50))
# _test_size = _val_size
# train_ds, val_ds, test_ds = get_or_create_splits(
#     ds, val_size=_val_size, test_size=_test_size,
#     seed=C.seed, splits_path=SPLITS_PATH,
# )
# print(f'  train: {len(train_ds):,} samples')
# print(f'  val:   {len(val_ds):,} samples (held out, deterministic)')
# print(f'  test:  {len(test_ds):,} samples (held out for final eval — '
#       f'NOT touched during training)')

# print('\nBuilding student model...')
# smoke_student = ConMambaStudent(C).to(device)
# if C.grad_checkpointing:
#     smoke_student.enable_grad_ckpt()
# if torch.cuda.device_count() > 1:
#     smoke_student = torch.nn.DataParallel(smoke_student)
#     print(f'Smoke wrapped in DataParallel across {torch.cuda.device_count()} GPUs')
# _reset_ctc_head(smoke_student, SMOKE_BLANK_BIAS)

# n_params = sum(p.numel() for p in _unwrap(smoke_student).parameters())
# n_trainable = sum(p.numel() for p in _unwrap(smoke_student).parameters() if p.requires_grad)
# print(f'Student: {n_params/1e6:.1f}M params ({n_trainable/1e6:.1f}M trainable)')
# flush()

# # Mirror training shape with the full trainable model and exercise both losses.
# trainable = sum(p.numel() for p in smoke_student.parameters() if p.requires_grad)
# print(f"Smoke test: {N_SMOKE} steps, bs={SMOKE_BS}, "
#       f"{trainable/1e6:.1f}M trainable (KD+CTC diagnostic)")
# print(f"  smoke loss = {SMOKE_A_KL} * loss_kl + "
#       f"{SMOKE_A_CTC} * loss_ctc")

# # Build buckets over the real training split. Only N_SMOKE batches are
# # loaded, but using all shard buckets guarantees enough bs=32 batches and
# # makes the smoke test representative of the production sampler.
# smoke_ds = train_ds
# _smoke_core = _unwrap(smoke_student)
# _smoke_ctc_params = list(_smoke_core.ctc_head.parameters())
# _smoke_ctc_ids = {id(p) for p in _smoke_ctc_params}
# _smoke_backbone_params = [
#     p for p in _smoke_core.parameters() if id(p) not in _smoke_ctc_ids]
# opt = torch.optim.AdamW([
#     {'params': _smoke_backbone_params, 'lr': SMOKE_BACKBONE_LR},
#     {'params': _smoke_ctc_params, 'lr': SMOKE_CTC_HEAD_LR},
# ], weight_decay=0.01)
# scaler = torch.amp.GradScaler('cuda')

# max_mel_frames = None  # chunk-level CTC: pad only, never crop

# # Match real training: workers + shard buckets avoid FUSE np.load bottlenecks.
# smoke_sampler = ShardBucketBatchSampler(
#     smoke_ds, batch_size=SMOKE_BS,
#     generator=torch.Generator().manual_seed(C.seed),
#     drop_last=True, merge_tails=C.merge_shard_tails,
#     language_temperature=C.train_language_temperature,
# )
# loader = DataLoader(
#     smoke_ds, batch_sampler=smoke_sampler,
#     num_workers=C.workers, persistent_workers=(C.workers > 0),
#     pin_memory=True, prefetch_factor=2 if C.workers > 0 else None,
#     collate_fn=lambda b: collate_kd(b, max_mel_frames=max_mel_frames, allow_crop=False),
# )
# _bucket_sizes = sorted(len(b) for b in smoke_sampler.shard_buckets)
# _nonempty = [s for s in _bucket_sizes if s >= SMOKE_BS]
# print(f'  ShardBucketBatchSampler: {len(smoke_sampler)} batches '
#       f'across {len(smoke_sampler.shard_buckets)} shards '
#       f'({len(_nonempty)} of which have >= {SMOKE_BS} samples)')
# print(f'  bucket-size percentiles: '
#       f'min={_bucket_sizes[0]}, p25={_bucket_sizes[len(_bucket_sizes)//4]}, '
#       f'p50={_bucket_sizes[len(_bucket_sizes)//2]}, '
#       f'p75={_bucket_sizes[3*len(_bucket_sizes)//4]}, max={_bucket_sizes[-1]}')
# print(f'  language counts:  {smoke_sampler.language_counts}')
# print(f'  language repeats: {smoke_sampler.language_repeats}')

# smoke_student.train()
# step_times, load_times, compute_times = [], [], []
# losses_kl, losses_ctc = [], []
# torch.cuda.reset_peak_memory_stats()

# print(f"\n{'step':>4} {'loss':>7} {'l_kl':>8} {'l_ctc':>7} "
#       f"{'mem_GB':>7} {'load':>5} {'cmp':>5} {'tot':>5}")
# print("-" * 60)

# # Time worker-queue dequeue separately from GPU compute.
# it = iter(loader)
# t_prev = time.monotonic()
# for step in range(N_SMOKE):
#     # ── 1. Data load (worker queue dequeue + collate already done in workers)
#     t_load_start = time.monotonic()
#     try:
#         batch = next(it)
#     except StopIteration:
#         break
#     t_load_done = time.monotonic()

#     mel = batch['mel'].to(device, non_blocking=True)
#     teacher_h = batch['teacher_h'].to(device, non_blocking=True)
#     teacher_lens = batch['teacher_lens'].to(device, non_blocking=True)
#     tokens = batch['tokens'].to(device, non_blocking=True)
#     tok_lens = batch['tok_lens'].to(device, non_blocking=True)
#     mel_lens = batch['mel_lens'].to(device, non_blocking=True)

#     # ── 2. Compute
#     t_compute_start = time.monotonic()
#     with torch.amp.autocast('cuda', dtype=torch.float16):
#         ctc_logits, kl_feat, enc_lens = smoke_student(mel, mel_lens)
#         loss_kl = kd_feature_loss(
#             kl_feat, teacher_h, enc_lens, teacher_lens=teacher_lens)
#         bad = describe_invalid_ctc_batch(batch, enc_lens, tok_lens, tokens)
#         if bad:
#             raise RuntimeError(
#                 'Smoke failed: chunk target length exceeds encoder length.')
#         loss_ctc = ctc_loss_fn(ctc_logits, tokens, enc_lens, tok_lens)
#         loss = SMOKE_A_KL * loss_kl + SMOKE_A_CTC * loss_ctc

#     if not torch.isfinite(loss):
#         _SMOKE_PASSED = False
#         raise RuntimeError(
#             f"Non-finite loss at smoke step {step}: "
#             f"loss={loss.item()} kl={loss_kl.item()} ctc={loss_ctc.item()}. "
#             "Mamba SSM likely overflowed fp16 — confirm the fp32 island is in forward()."
#         )

#     scaler.scale(loss).backward()
#     scaler.unscale_(opt)
#     torch.nn.utils.clip_grad_norm_(smoke_student.parameters(), C.gc_norm)
#     scaler.step(opt)
#     scaler.update()
#     opt.zero_grad(set_to_none=True)
#     torch.cuda.synchronize()
#     t_compute_done = time.monotonic()

#     load_t = t_load_done - t_load_start
#     compute_t = t_compute_done - t_compute_start
#     total_t = t_compute_done - t_prev
#     t_prev = t_compute_done

#     load_times.append(load_t)
#     compute_times.append(compute_t)
#     step_times.append(total_t)
#     losses_kl.append(loss_kl.item())
#     losses_ctc.append(loss_ctc.item())

#     if step < 5 or step % 10 == 0 or step == N_SMOKE - 1:
#         mem_gb = torch.cuda.memory_allocated() / 1e9
#         print(f"{step:4d} {loss.item():7.3f} {loss_kl.item():8.4f} "
#               f"{loss_ctc.item():7.3f} {mem_gb:7.2f} "
#               f"{load_t:5.2f} {compute_t:5.2f} {total_t:5.2f}")

# # ── Verdict ─────────────────────────────────────────────────────
# print("\n" + "=" * 60)
# # Skip warmup steps for medians.
# warm = max(2, len(step_times) // 10)


# def med(xs):
#     return statistics.median(xs[warm:]) if len(xs) > warm else (xs[-1] if xs else 0.0)


# med_step = med(step_times)
# med_load = med(load_times)
# med_compute = med(compute_times)
# peak_mem = torch.cuda.max_memory_allocated() / 1e9
# kl_first = sum(losses_kl[:5]) / min(5, len(losses_kl))
# kl_last = sum(losses_kl[-5:]) / min(5, len(losses_kl))
# ctc_first = sum(losses_ctc[:5]) / min(5, len(losses_ctc))
# ctc_last = sum(losses_ctc[-5:]) / min(5, len(losses_ctc))

# print(f"Median sec/step (after warmup): {med_step:.2f}s")
# print(
#     f"  - data load: {med_load:.2f}s  ({100*med_load/max(med_step, 1e-3):.0f}% of step)")
# print(f"  - compute  : {med_compute:.2f}s  ({100 *
#                                              med_compute /
#                                              max(med_step, 1e-3):.0f}% of step)")
# print(f"Peak GPU memory:                {peak_mem:.2f} GB")
# print(
#     f"loss_kl  first 5  : last 5: {
#         kl_first:.4f}  : {
#             kl_last:.4f}  (Δ {
#                 kl_last -
#                 kl_first:+.4f})")
# print(
#     f"loss_ctc first 5  : last 5: {
#         ctc_first:.3f}  : {
#             ctc_last:.3f}  (Δ {
#                 ctc_last -
#                 ctc_first:+.3f})")
# print()

# # Thresholds are smoke-specific: small batch, no long-run optimizer state.
# issues = []
# # Step-time: with workers active and compute at ~0.5 s, we should be ≤ 2 s/step.
# if med_step > 2.0:
#     if med_load > med_compute:
#         issues.append(
#             f"step {med_step:.1f}s is dataloader-bound "
#             f"(load {med_load:.1f}s >> compute {med_compute:.1f}s). "
#             f"Try increasing C.workers (currently {C.workers}) or prefetch_factor."
#         )
#     else:
#         issues.append(
#             f"step {med_step:.1f}s with compute {med_compute:.1f}s — "
#             f"Mamba slow path may be active. Re-run diagnostic_cell."
#         )
# # Compute floor: fast-path Mamba on T4 bs=4 T=300 frozen ≈ 0.3-0.7s.
# if med_compute > 2.0:
#     issues.append(
#         f"compute {med_compute:.2f}s/step too high for fast-path Mamba "
#         "(expected ~0.3-0.7s on T4 bs=4 frozen)"
#     )
# # A fresh random projection should have cosine loss near 1. A zero here means
# # the KD path was skipped (the regression this smoke test is meant to catch).
# if max(losses_kl, default=0.0) <= 1e-6:
#     issues.append(
#         'loss_kl is zero: teacher features were not compared with kl_head output.'
#     )
# # Require finite/moving CTC; KL is also optimized above but its short-run slope
# # is deliberately not a gate because random batches make cosine loss noisy.
# if ctc_last >= ctc_first - 0.01:
#     issues.append(
#         f"loss_ctc not decreasing enough: first5={
#             ctc_first:.3f}, last5={
#             ctc_last:.3f}. "
#         "Run the tiny overfit cell before full training."
#     )
# # Memory: just sanity-check we didn't accidentally blow past T4 capacity.
# if peak_mem > 14.0:
#     issues.append(f"peak mem {peak_mem:.1f} GB — close to T4's 16 GB limit; "
#                   "reduce bs or max_s1")

# if issues:
#     _SMOKE_PASSED = False
#     print("  Issues detected:")
#     for i in issues:
#         print(f"  - {i}")
#     print("\nDo NOT proceed to full training until these are resolved.")
#     print("(_SMOKE_PASSED = False  — the training cell will refuse to start.)")
# else:
#     _SMOKE_PASSED = True
#     print("✅ All checks pass — _SMOKE_PASSED = True")
#     print("   Proceed to the training cell below.")

# # Free only smoke state. The production `student` from cell 25 remains
# # DataParallel-wrapped and untouched for the launch cell.
# del smoke_student, _smoke_core, _smoke_ctc_params, _smoke_ctc_ids
# del _smoke_backbone_params, opt, scaler, loader, smoke_ds, smoke_sampler, it
# torch.cuda.empty_cache()


### CTC length sanity and tiny overfit

Run this before full training on a new chunk cache. It checks chunk CTC lengths and verifies the model can overfit a tiny valid subset.

In [17]:
# # ── Chunk-level CTC sanity + tiny overfit ───────────────────────
# from torch.utils.data import Subset, DataLoader
# import math

# print('\nCTC length sanity:')
# _diag_loader = DataLoader(
#     val_ds if 'val_ds' in globals() else Subset(ds, list(range(min(16, len(ds))))),
#     batch_size=4, shuffle=False, num_workers=0,
#     collate_fn=lambda b: collate_kd(b, max_mel_frames=None, allow_crop=False),
# )
# _diag_model = _unwrap(student).eval() if 'student' in globals(
# ) else ConMambaStudent(C).to(device).eval()
# _batch = next(iter(_diag_loader))
# _mel = _batch['mel'].to(device)
# _mel_lens = _batch['mel_lens'].to(device)
# with torch.no_grad(), torch.amp.autocast('cuda', dtype=torch.float16):
#     _ctc_logits, _kl, _enc_lens = _diag_model(_mel, _mel_lens)
# for j in range(len(_batch['texts'])):
#     print(
#         f'[{j}] mel_len={_batch["mel_lens"][j].item()} '
#         f'enc_len={_enc_lens[j].item()} '
#         f'tok_len={_batch["tok_lens"][j].item()} '
#         f'lang={_batch["langs"][j]} '
#         f'source={_batch["source_ids"][j]} '
#         f'text={_batch["texts"][j][:100]}'
#     )
# _bad = describe_invalid_ctc_batch(
#     _batch, _enc_lens.cpu(), _batch['tok_lens'], _batch['tokens'])
# if _bad:
#     raise RuntimeError(
#         'CTC length sanity failed; regenerate/filter shorter chunk targets before training.')


# def pick_short_valid_subset(ds, n=8, max_mel_frames=800, max_tok_len=80):
#     keep = []
#     for i in range(len(ds)):
#         item = ds[i]
#         mel_len = item['mel'].shape[1]
#         tok_len = len(item['token_ids'])
#         approx_enc_len = math.ceil(mel_len / 4)
#         if mel_len <= max_mel_frames and 0 < tok_len <= max_tok_len and tok_len < approx_enc_len:
#             keep.append(i)
#         if len(keep) >= n:
#             break
#     if len(keep) < n:
#         raise RuntimeError(f'Only found {len(keep)} short valid chunks; need {n}.')
#     return Subset(ds, keep)


# RUN_TINY_OVERFIT = globals().get('RUN_TINY_OVERFIT', True)
# _TINY_OVERFIT_PASSED = False
# if RUN_TINY_OVERFIT:
#     print('\nTiny CTC overfit:')
#     _base_train = train_ds if 'train_ds' in globals() else ds
#     tiny_ds = pick_short_valid_subset(_base_train, n=8)
#     tiny_loader = DataLoader(
#         tiny_ds, batch_size=4, shuffle=True, num_workers=0,
#         collate_fn=lambda b: collate_kd(b, max_mel_frames=None, allow_crop=False),
#     )
#     tiny_student = ConMambaStudent(C).to(device)
#     if C.grad_checkpointing:
#         tiny_student.enable_grad_ckpt()
#     tiny_student.train()
#     opt = torch.optim.AdamW(tiny_student.parameters(), lr=1e-4)
#     losses = []
#     it = iter(tiny_loader)
#     for step in range(300):
#         try:
#             batch = next(it)
#         except StopIteration:
#             it = iter(tiny_loader)
#             batch = next(it)
#         mel = batch['mel'].to(device)
#         mel_lens = batch['mel_lens'].to(device)
#         tokens = batch['tokens'].to(device)
#         tok_lens = batch['tok_lens'].to(device)
#         opt.zero_grad(set_to_none=True)
#         with torch.amp.autocast('cuda', dtype=torch.float16):
#             ctc_logits, _kl, enc_lens = tiny_student(mel, mel_lens)
#             bad = describe_invalid_ctc_batch(batch, enc_lens, tok_lens, tokens)
#             if bad:
#                 raise RuntimeError('Tiny overfit failed: invalid CTC lengths.')
#             loss = ctc_loss_fn(ctc_logits, tokens, enc_lens, tok_lens)
#         loss.backward()
#         torch.nn.utils.clip_grad_norm_(tiny_student.parameters(), 1.0)
#         opt.step()
#         losses.append(float(loss.item()))
#         if step % 50 == 0 or step == 299:
#             pred_ids = greedy_ctc_token_ids(
#                 ctc_logits.float().detach().cpu(), BLANK,
#                 lengths=enc_lens.cpu().tolist(),
#             )
#             decoded = [sp.DecodeIds(ids) for ids in pred_ids]
#             collapse = detect_ctc_collapse(pred_ids, BLANK)
#             print(f'step={step:03d} loss={loss.item():.4f} collapse={collapse}')
#             print('REF:', batch['texts'][0])
#             print('HYP:', decoded[0] if decoded[0] else '(empty)')
#     first = sum(losses[:20]) / min(20, len(losses))
#     last = sum(losses[-20:]) / min(20, len(losses))
#     if not last < first:
#         raise RuntimeError(
#             f'Tiny overfit failed: loss did not decrease ({first:.4f} -> {last:.4f}).')
#     _TINY_OVERFIT_PASSED = True
#     print(f'Tiny overfit passed: loss {first:.4f} -> {last:.4f}')
#     del tiny_student, opt, tiny_loader, tiny_ds
#     torch.cuda.empty_cache()
# else:
#     print('RUN_TINY_OVERFIT=False; skipped tiny overfit loop.')


### Training process (run after smoke passes)

The 2026-07-23 run stopped normally at its old 2.5-hour budget while
`scratch_joint` was still improving (global step 10,741; best normalized CER
60.76%, test WER 86.99%). It was **not converged** and did not collapse.

This continuation uses the measured T4 headroom (`bs=32`, `GA=1`, effective
batch 32), merges shard tails so the larger batch does not discard data,
moderately repeats low-resource languages, selects checkpoints by macro
per-language CER, validates on the complete 2,000-sample validation split, and
allows an 8-hour optimizer window.

Attach the latest `edge_asr` bundle, run the training-configuration cell, rerun
the mandatory smoke test, then run this cell. The v2 joint checkpoint is
migrated in place: model, optimizer, scaler, scheduler, and global step resume;
only the incomparable old sample-weighted best-metric state is reset.


In [18]:
# # Pre-flight: require _SMOKE_PASSED unless SKIP_SMOKE_GUARD=True.
# if not globals().get('SKIP_SMOKE_GUARD', False):
#     assert globals().get('_SMOKE_PASSED', False), (
#         "Run the smoke-test cell FIRST and confirm it passes "
#         "(it sets _SMOKE_PASSED=True). To bypass: SKIP_SMOKE_GUARD = True"
#     )


# # Fail before allocating GPU time if this Kaggle kernel still holds stale
# # checkpoint-lineage or snapshot-only collapse logic.
# _EXPECTED_TRAINING_DEFINITION_VERSION = 'scratch-quality-throughput-v7'
# assert globals().get('TRAINING_DEFINITION_VERSION') == _EXPECTED_TRAINING_DEFINITION_VERSION, (
#     'Training definitions are stale or incomplete. Rerun the Training '
#     'configuration cell, then rerun the smoke test before launching training. '
#     f'Expected {_EXPECTED_TRAINING_DEFINITION_VERSION!r}, got '
#     f'{globals().get("TRAINING_DEFINITION_VERSION")!r}.'
# )

# # Setup and smoke do not consume this budget. Reset on each explicit launch
# # so a saved checkpoint can receive a full fresh budget when this cell reruns.
# start_training_budget(reset=True)


# # Resume preflight: fail before allocating GPU time if the selected
# # read-only edge_asr bundle is not attached or has the wrong layout.
# # Input mounts are intentionally read-only; only C.ckpt_dir is writable.
# if getattr(C, 'resume_from_checkpoint', True):
#     _resume_probe_stage = (
#         'scratch_kd' if C.train_from_scratch else 'recover_ctc')
#     _resume_probe_paths = []
#     for _suffix in ('latest', 'best'):
#         _resume_probe_paths.extend(
#             _load_candidates(_resume_probe_stage, suffix=_suffix,
#                              include_readonly=True))

#     _resume_found = [
#         _path for _path in _resume_probe_paths
#         if os.path.isfile(_path)
#     ]
#     if not _resume_found:
#         _expected_root = (
#             '/kaggle/input/datasets/leviettrieu369/'
#             'distillation-checkpoint/edge_asr'
#         )
#         _searched_dirs = sorted(set(
#             os.path.dirname(_path) for _path in _resume_probe_paths
#         ))
#         raise FileNotFoundError(
#             f'Resume checkpoint preflight failed for stage '
#             f'{_resume_probe_stage!r}: no {_resume_probe_stage}_latest.pt '
#             f'or {_resume_probe_stage}_best.pt was found.\\n'
#             f'Expected the attached read-only dataset under: {_expected_root}\\n'
#             'Expected layout: <edge_asr>/checkpoints/<stage>_latest.pt\\n'
#             f'Searched checkpoint directories: {_searched_dirs}\\n'
#             'Attach the distillation-checkpoint Kaggle dataset and rerun '
#             'the configuration cell. To intentionally start a fresh lineage, '
#             'set C.resume_from_checkpoint = False before launching. '
#             f'New checkpoints are written only under {C.ckpt_dir}.'
#         )

#     print(f'Resume checkpoint preflight passed for {_resume_probe_stage}:')
#     for _path in _resume_found:
#         _size_mb = os.path.getsize(_path) / (1024 ** 2)
#         print(f'  {_path} ({_size_mb:.1f} MiB)')
# fresh_specs = [
#     StageSpec(
#         name='scratch_kd',
#         # No source stage and no legacy fallback: when no matching scratch_kd
#         # resume exists, train_stage keeps the newly initialized ConMamba.
#         lr=C.fr_lr,
#         epochs=1,
#         bs=C.fr_bs,
#         ga=C.fr_ga,
#         max_seconds=None,
#         a_kl=1.0,
#         ctc_start_weight=0.0,
#         ctc_end_weight=0.0,
#         ctc_warmup_steps=0,
#         reset_ctc_head=True,
#         ctc_blank_bias=0.0,
#         best_metric='loss',
#         allow_resume=True,
#         allow_readonly_resume=True,
#         require_source_checkpoint=False,
#         resume_tag=SCRATCH_KD_RECIPE_TAG,
#         evaluation_lineage_clean=True,
#         # A compatible joint checkpoint proves the KD warmup already finished.
#         skip_if_checkpoint_stages=(
#             ('scratch_joint', SCRATCH_JOINT_RECIPE_TAG),),
#         fail_fast_on_collapse=False,
#     ),
#     StageSpec(
#         name='scratch_joint',
#         # Stage 2 may start only from this lineage's completed KD warmup.
#         source_checkpoint_stage='scratch_kd',
#         source_checkpoint_tag=SCRATCH_KD_RECIPE_TAG,
#         fallback_source_stages=(),
#         source_checkpoint_suffix='latest',
#         fallback_source_suffix='best',
#         lr=C.un_lr,
#         ctc_head_lr=1e-3,
#         warmup_steps=200,
#         epochs=C.un_epochs,
#         bs=C.un_bs,
#         ga=C.un_ga,
#         max_seconds=None,
#         a_kl=1.0,
#         kl_end_weight=C.joint_kl_end_weight,
#         kl_hold_steps=C.joint_kl_hold_steps,
#         kl_decay_steps=C.joint_kl_decay_steps,
#         ctc_start_weight=0.10,
#         ctc_end_weight=0.30,
#         ctc_warmup_steps=1000,
#         reset_ctc_head=True,
#         ctc_blank_bias=0.0,
#         best_metric='macro_cer',
#         allow_resume=True,
#         allow_readonly_resume=True,
#         require_source_checkpoint=True,
#         validation_interval_updates=C.mid_epoch_val_updates,
#         validation_max_batches=C.val_max_batches,
#         validation_probe_samples=C.midval_probe_samples,
#         midval_selects_best=False,
#         resume_tag=SCRATCH_JOINT_RECIPE_TAG,
#         # Migrate the supplied 3,978-step v1 checkpoint in place. Effective
#         # batch remains 32, so optimizer/scheduler state stays compatible.
#         resume_source_tags=(
#             SCRATCH_JOINT_RECIPE_TAG,
#             PREVIOUS_SCRATCH_JOINT_RECIPE_TAG,
#             PRIOR_SCRATCH_JOINT_RECIPE_TAG,
#             LEGACY_SCRATCH_JOINT_RECIPE_TAG,
#         ),
#         evaluation_lineage_clean=True,
#         fail_fast_on_collapse=True,
#         canary_updates=300,
#         canary_min_relative_loss_drop=0.05,
#     ),
# ]


# recovery_specs = [
#     StageSpec(
#         name='recover_kd',
#         # Prefer the recovery checkpoint from a prior Kaggle session. If a
#         # a tag-compatible downstream recover_ctc checkpoint is already in the
#         # uploaded bundle, recover_kd is complete and train_stage will skip it.
#         source_checkpoint_stage='recover_kd',
#         source_checkpoint_tag=RECOVER_KD_RECIPE_TAG,
#         fallback_source_stages=('frozen', 'unfrozen', 'chunk_ctc'),
#         source_checkpoint_suffix='best',
#         fallback_source_suffix='latest',
#         lr=C.fr_lr,
#         epochs=1,
#         bs=C.fr_bs,
#         ga=C.fr_ga,
#         max_seconds=None,
#         a_kl=1.0,
#         ctc_start_weight=0.0,
#         ctc_end_weight=0.0,
#         ctc_warmup_steps=0,
#         reset_ctc_head=True,
#         ctc_blank_bias=0.0,
#         best_metric='loss',
#         allow_resume=True,
#         allow_readonly_resume=True,
#         require_source_checkpoint=True,
#         resume_tag=RECOVER_KD_RECIPE_TAG,
#         # Do not spend another multi-hour KD epoch when a downstream CTC
#         # checkpoint proves this stage already completed in an earlier session.
#         skip_if_checkpoint_stages=(
#             ('recover_ctc', RECOVER_CTC_RECIPE_TAG),
#             ('recover_ctc', EDGE_ASR_INPUT_RECOVER_CTC_TAG),),
#         fail_fast_on_collapse=False,
#     ),
#     StageSpec(
#         name='recover_ctc',
#         source_checkpoint_stage='recover_kd',
#         source_checkpoint_tag=RECOVER_KD_RECIPE_TAG,
#         fallback_source_stages=('frozen', 'unfrozen', 'chunk_ctc'),
#         source_checkpoint_suffix='latest',
#         fallback_source_suffix='best',
#         # Protect the KD-aligned backbone while the randomly initialized
#         # 5k-way projection learns CTC. The loss weight below scales the head
#         # gradient, so its nominal LR is raised to preserve a useful effective
#         # update without letting CTC dominate every backbone layer.
#         lr=C.un_lr,
#         ctc_head_lr=1e-3,
#         warmup_steps=200,
#         epochs=C.un_epochs,
#         bs=C.un_bs,
#         ga=C.un_ga,
#         max_seconds=None,
#         # Joint KD+CTC (design doc stage 2), not pure CTC. Cosine feature
#         # supervision anchors the encoder while the CTC weight ramps gradually.
#         a_kl=1.0,
#         kl_end_weight=C.joint_kl_end_weight,
#         kl_hold_steps=C.joint_kl_hold_steps,
#         kl_decay_steps=C.joint_kl_decay_steps,
#         ctc_start_weight=0.10,
#         ctc_end_weight=0.30,
#         ctc_warmup_steps=1000,
#         reset_ctc_head=True,
#         # A -5 blank bias did not prevent collapse; it instead creates a
#         # violent nonblank-to-blank transient. Neutral init is the tested
#         # tiny-overfit path, while temporal canary logic watches real progress.
#         ctc_blank_bias=0.0,
#         best_metric='macro_cer',
#         allow_resume=True,
#         allow_readonly_resume=True,
#         require_source_checkpoint=True,
#         validation_interval_updates=C.mid_epoch_val_updates,
#         validation_max_batches=C.val_max_batches,
#         validation_probe_samples=C.midval_probe_samples,
#         midval_selects_best=False,
#         # The v8 tag invalidates the stale pure-CTC/equal-weight attempts and
#         # guarantees a fresh head plus the tagged v2 KD backbone.
#         resume_tag=RECOVER_CTC_RECIPE_TAG,
#         resume_source_tags=(
#             RECOVER_CTC_RECIPE_TAG, PRIOR_RECOVER_CTC_RECIPE_TAG,
#             EDGE_ASR_INPUT_RECOVER_CTC_TAG),
#         fail_fast_on_collapse=True,
#         # At 300 updates, fail only if a probability-confirmed collapse also
#         # fails to improve validation loss by at least 5% from step zero.
#         canary_updates=300,
#         canary_min_relative_loss_drop=0.05,
#     ),
# ]

# if C.train_from_scratch:
#     stage_specs = fresh_specs
#     print('Training mode: FROM SCRATCH. Legacy recovery checkpoints are ignored.')
# else:
#     stage_specs = recovery_specs
#     print('Training mode: LEGACY RECOVERY. Source checkpoints are required.')

# stage_results = {}
# all_done = True
# for spec in stage_specs:
#     if time_left_seconds() < 600:
#         print(f'\nSession has < 10 min left. Save Version and run {spec.name} next session.')
#         loss_log.log('session_pause', stage=spec.name, completed=False)
#         all_done = False
#         break

#     print('\n' + '=' * 60)
#     print(f'STAGE: {spec.name}  (time left: {fmt_hms(time_left_seconds())})')
#     print('=' * 60)
#     stage_loss, stage_done = train_stage(student, train_ds, spec, val_dataset=val_ds)
#     stage_results[spec.name] = {'loss': stage_loss, 'done': stage_done}
#     if not stage_done:
#         print(f'\n{spec.name} paused. Save Version, reopen, and run again to resume.')
#         loss_log.log('session_pause', stage=spec.name, completed=False, last_loss=stage_loss)
#         all_done = False
#         break

# if all_done:
#     print(f'\nTraining complete - {stage_results}')
#     for _name, _result in stage_results.items():
#         wandb_run.summary[f'final/{_name}_loss'] = _result['loss']
#     loss_log.log('training_complete', stage_results=stage_results)

# loss_log.close()
# wandb_run.finish(exit_code=0, quiet=True)


### Test-set evaluation (run after training completes)

Loads the best compatible checkpoint and runs two passes on the source-grouped
held-out test split:

**Pass 1 — accuracy** (full test set, bs=2): reports normalized and raw WER/CER.
Normalized text uses NFKC, case-folding, and punctuation stripping. Normalized
CER is the checkpoint-selection metric because whitespace WER is not comparable
across Latin, Chinese, and Japanese scripts. Raw metrics remain in the JSON for
exact continuity with the previous 94.98% WER / 77.21% CER run.

**Pass 2 — latency** (bs=1, warmup plus `torch.cuda.synchronize`): mean, p50,
p95, and p99 milliseconds, plus real-time factor when duration is available.

The cell saves `splits.json`, per-sample
`test_results/predictions_<stage>.jsonl`, and
`test_results/results_<stage>.json`. It rebuilds the student from disk and does
not depend on the in-memory training model.


In [19]:
# # ── Test-set evaluation: WER + CER + Latency ──────────────────────
# # Independent of the training cell's in-memory state — rebuilds the
# # student from disk so it works even after a kernel restart.
# import transformers as _transformers_mod
# import json
# import time
# import numpy as np
# import jiwer
# from torch.utils.data import Subset, DataLoader

# # 1. Load splits (test indices).
# SPLITS_PATH = os.path.join(C.ckpt_dir, 'splits.json')
# assert os.path.exists(SPLITS_PATH), (
#     f'No splits.json at {SPLITS_PATH}. Run the training cell at least '
#     'once first — it creates the splits.'
# )
# with open(SPLITS_PATH) as f:
#     splits = json.load(f)

# # 2. Ensure dataset is loaded (might not be if kernel was restarted).
# if 'ds' not in globals():
#     print('Re-loading dataset...')
#     ds = KDDataset(C.cache_dir, sp, C.mel_dir)
# test_ds = Subset(ds, splits['test_indices'])
# print(f'Test set: {len(test_ds):,} samples '
#       f'(fingerprint={splits.get("fingerprint")})')

# # 3. Find the best compatible checkpoint. Prefer validated `best` weights;
# # fall back to `latest` only while a stage has not produced a best snapshot.
# # Search the writable session directory first within each suffix, then the
# # read-only uploaded checkpoint dataset so evaluation survives a kernel restart.
# # Evaluation may inspect a checkpoint produced by the immediately
# # preceding CTC recipe. This compatibility exception is intentionally local
# # to evaluation: training resume still requires the exact current tag.
# _RECOVER_CTC_EVAL_TAGS = (
#     RECOVER_CTC_RECIPE_TAG,
#     PRIOR_RECOVER_CTC_RECIPE_TAG,
#     'recover_ctc_ctc_only_headlr_v6',
# )
# if C.train_from_scratch:
#     _SCRATCH_JOINT_EVAL_TAGS = (
#         SCRATCH_JOINT_RECIPE_TAG,
#         PREVIOUS_SCRATCH_JOINT_RECIPE_TAG,
#         PRIOR_SCRATCH_JOINT_RECIPE_TAG,
#         LEGACY_SCRATCH_JOINT_RECIPE_TAG,
#     )
#     _candidate_specs = [
#         ('scratch_joint', 'best', _SCRATCH_JOINT_EVAL_TAGS),
#         ('scratch_joint', 'latest', _SCRATCH_JOINT_EVAL_TAGS),
#     ]
# else:
#     _candidate_specs = [
#         ('recover_ctc', 'best', _RECOVER_CTC_EVAL_TAGS),
#         ('recover_ctc', 'latest', _RECOVER_CTC_EVAL_TAGS),
#         ('chunk_ctc', 'best', None),
#         ('chunk_ctc', 'latest', None),
#         ('unfrozen', 'best', None),
#         ('unfrozen', 'latest', None),
#     ]
# stage = best_path = None
# _state = None
# for _stage, _suffix, _required_tag in _candidate_specs:
#     for _path in _load_candidates(_stage, suffix=_suffix, include_readonly=True):
#         _candidate, _loaded_from = try_load_compatible_checkpoint(
#             _path, expected_tag=_required_tag,
#             purpose=f'{_stage} evaluation checkpoint')
#         if _candidate is None:
#             continue
#         if _candidate.get('tokenizer_fingerprint') != TOKENIZER_FINGERPRINT:
#             print(f'Ignoring evaluation checkpoint {_loaded_from}: tokenizer '
#                   'fingerprint is missing or incompatible.')
#             continue
#         try:
#             _require_checkpoint_target_recipe(_candidate, _loaded_from)
#         except RuntimeError as exc:
#             print(f'Ignoring evaluation checkpoint {_loaded_from}: {exc}')
#             continue
#         stage, best_path, _state = _stage, _loaded_from, _candidate
#         break
#     if _state is not None:
#         break
# if best_path is None:
#     # A session-budget pause is a normal resumable state, not an evaluation
#     # failure. In particular, a KD-only checkpoint has a deliberately random
#     # CTC head, so evaluating it would produce meaningless WER. Papermill should
#     # finish cleanly and let the next Kaggle Version resume training instead.
#     EVALUATION_SKIPPED = True
#     print(
#         '\nEvaluation skipped: training has not produced a recipe- and '
#         'tokenizer-compatible CTC checkpoint yet.'
#     )
#     print(
#         'The KD-only checkpoint remains resumable. Save this Kaggle Version, '
#         'attach its output checkpoint dataset, and rerun the training cell '
#         'to continue the active KD-to-joint curriculum before evaluation.'
#     )
# else:
#     EVALUATION_SKIPPED = False
#     print(f'Using checkpoint: {best_path} (stage={stage})')

#     # 4. Build a fresh student and load weights.
#     # Always load into the un-wrapped model; strip DataParallel `module.` prefix
#     # from saved state-dict if present.
#     test_student = ConMambaStudent(C).to(device)
#     _require_checkpoint_tokenizer(_state, best_path)
#     _sd = _state['model']
#     # Normalize the on-disk state-dict to no-prefix form. New checkpoints
#     # (post-v7) are already in this form, so this is a no-op then. Legacy
#     # DP-saved checkpoints have `module.` prefixes which we strip. The old
#     # inline comprehension had a bug — it filtered out non-prefixed keys,
#     # which would silently zero-init half the model on mixed dicts.
#     if any(k.startswith('module.') for k in _sd):
#         _sd = {(k[len('module.'):] if k.startswith('module.') else k): v
#                for k, v in _sd.items()}
#     _dropped = _load_model_tolerant(test_student, _sd)
#     if _dropped:
#         raise RuntimeError(
#             f'Evaluation refused to use shape-mismatched weights: {_dropped}')
#     test_student.eval()
#     print(f'  checkpoint meta: epoch={_state.get("epoch")}, '
#           f'global_step={_state.get("global_step")}, '
#           f'loss={_state.get("loss", _state.get("best_loss", float("nan"))):.4f}')
#     evaluation_lineage_clean = bool(_state.get('evaluation_lineage_clean', False))
#     if not evaluation_lineage_clean:
#         print('WARNING: this checkpoint descends from legacy per-chunk-split '
#               'weights. Accuracy is diagnostic only, not an unbiased benchmark. '
#               'Train a fresh source-grouped lineage for publishable test metrics.')

#     # 5. Set up output paths.
#     results_dir = os.path.join(C.ckpt_dir, 'test_results')
#     os.makedirs(results_dir, exist_ok=True)
#     pred_path = os.path.join(results_dir, f'predictions_{stage}.jsonl')
#     results_path = os.path.join(results_dir, f'results_{stage}.json')

#     # ── Pass 1: accuracy over the full test set in merged-tail batches.
#     print(f'\nPass 1/2: WER + CER on {len(test_ds):,} test samples...')
#     max_seconds = None
#     max_mel_frames = None

#     acc_sampler = ShardBucketBatchSampler(
#         test_ds, batch_size=C.validation_batch_size, drop_last=False,
#         shuffle=False, merge_tails=True,
#     )
#     acc_loader = DataLoader(
#         test_ds, batch_sampler=acc_sampler,
#         num_workers=C.workers, persistent_workers=False,
#         pin_memory=True,
#         collate_fn=lambda b: collate_kd(b, max_mel_frames=max_mel_frames, allow_crop=False),
#     )

#     refs, hyps, audio_seconds_list, langs, pred_id_seqs, frame_argmax_seqs = [], [], [], [], [], []
#     with open(pred_path, 'w') as pf, torch.no_grad():
#         for batch in tqdm(acc_loader, desc='accuracy', unit='batch'):
#             mel = batch['mel'].to(device, non_blocking=True)
#             mel_lens = batch['mel_lens'].to(device, non_blocking=True)

#             with torch.amp.autocast('cuda', dtype=torch.float16):
#                 ctc_logits, _kl, enc_lens = test_student(mel, mel_lens)

#             logits_cpu = ctc_logits.float().cpu()
#             lengths = enc_lens.cpu().tolist()
#             pred_ids = greedy_ctc_token_ids(logits_cpu, BLANK, lengths=lengths)
#             frame_argmax = logits_cpu.argmax(dim=-1)
#             for seq, valid_len in zip(frame_argmax, lengths):
#                 frame_argmax_seqs.append(seq[:valid_len].tolist())
#             decoded = [sp.DecodeIds(ids) for ids in pred_ids]
#             pred_id_seqs.extend(pred_ids)
#             batch_secs = batch.get('audio_seconds', [None] * len(decoded))
#             batch_langs = batch.get('langs', ['unknown'] * len(decoded))
#             for ref, hyp, sec, lang in zip(
#                     batch['texts'], decoded, batch_secs, batch_langs):
#                 refs.append(ref)
#                 hyps.append(hyp)
#                 langs.append(lang)
#                 sec_val = float(sec) if sec is not None else None
#                 if sec_val is not None:
#                     audio_seconds_list.append(sec_val)
#                 pf.write(json.dumps(
#                     {'ref': ref, 'hyp': hyp, 'lang': lang, 'audio_seconds': sec_val},
#                     ensure_ascii=False,
#                 ) + '\n')

#     # Report both fair ASR-normalized metrics and exact raw-string metrics.
#     norm_refs, norm_hyps = normalized_metric_texts(refs, hyps)
#     wer = compute_wer(norm_refs, norm_hyps)
#     cer = compute_cer(norm_refs, norm_hyps)
#     wer_raw = compute_wer(refs, hyps)
#     cer_raw = compute_cer(refs, hyps)
#     collapse_metrics = compute_collapse_metrics(
#         pred_id_seqs, BLANK, frame_argmax_ids=frame_argmax_seqs,
#         refs=refs, hyps=hyps, sp_model=sp,
#     )
#     empty_hyp_pct = collapse_metrics['empty_hypothesis_pct']
#     pred_lens = [len(ids) for ids in pred_id_seqs]
#     ref_lens = [len(sp.EncodeAsIds(r)) for r in refs]
#     print(f'  WER(norm) = {wer*100:.2f}%   CER(norm) = {cer*100:.2f}%')
#     print(f'  WER(raw)  = {wer_raw*100:.2f}%   CER(raw)  = {cer_raw*100:.2f}% '
#           f'(n={len(refs)})')
#     print(f'  Collapse: {collapse_metrics}  empty_hyp={empty_hyp_pct*100:.1f}%')

#     per_language = per_language_metrics(refs, hyps, langs)
#     macro_cer, macro_languages = macro_language_cer(
#         per_language, min_samples=C.macro_metric_min_samples)
#     print(f'  Macro language CER = {macro_cer*100:.2f}% '
#           f'over {macro_languages}')
#     print('  Per-language (normalized; CER is primary for zh/ja):')
#     for _lang, _m in per_language.items():
#         print(
#             f'    {_lang:32s} CER={_m["cer"]*100:6.2f}%  '
#             f'WER={_m["wer"]*100:7.2f}%  (n={_m["n"]})'
#         )

#     # ── Pass 2: latency at bs=1 over a subset, with proper warmup ────
#     N_LAT = min(200, len(test_ds))
#     print(f'\nPass 2/2: latency over {N_LAT} samples (bs=1, fp16)...')

#     lat_subset = Subset(test_ds, list(range(N_LAT)))
#     lat_loader = DataLoader(
#         lat_subset, batch_size=1, shuffle=False, num_workers=2,
#         pin_memory=True, drop_last=False,
#         collate_fn=lambda b: collate_kd(b, max_mel_frames=max_mel_frames, allow_crop=False),
#     )

#     # Warmup (first few forwards trigger Triton JIT / kernel caching).
#     with torch.no_grad():
#         for i, batch in enumerate(lat_loader):
#             if i >= 10:
#                 break
#             mel = batch['mel'].to(device, non_blocking=True)
#             mel_lens = batch['mel_lens'].to(device, non_blocking=True)
#             with torch.amp.autocast('cuda', dtype=torch.float16):
#                 test_student(mel, mel_lens)
#             torch.cuda.synchronize()

#     # Measure — torch.cuda.synchronize() before and after to exclude
#     # data-load + queue time from the measurement.
#     latencies_ms, sample_audio_secs = [], []
#     with torch.no_grad():
#         for batch in tqdm(lat_loader, desc='latency', unit='sample'):
#             mel = batch['mel'].to(device, non_blocking=True)
#             mel_lens = batch['mel_lens'].to(device, non_blocking=True)
#             sec = batch.get('audio_seconds', [None])
#             sec_val = float(sec[0]) if sec and sec[0] is not None else None

#             torch.cuda.synchronize()
#             t0 = time.perf_counter()
#             with torch.amp.autocast('cuda', dtype=torch.float16):
#                 test_student(mel, mel_lens)
#             torch.cuda.synchronize()
#             latencies_ms.append((time.perf_counter() - t0) * 1000)
#             sample_audio_secs.append(sec_val)

#     lat_arr = np.array(latencies_ms)

#     # Real-time factor: latency_seconds / audio_seconds. <1.0 is faster than
#     # real-time. Reported only if audio_seconds is available.
#     rtf_stats = None
#     if all(s is not None for s in sample_audio_secs) and sample_audio_secs:
#         sec_arr = np.array(sample_audio_secs)
#         rtf = (lat_arr / 1000.0) / sec_arr
#         rtf_stats = {
#             'mean': float(rtf.mean()),
#             'p50':  float(np.percentile(rtf, 50)),
#             'p95':  float(np.percentile(rtf, 95)),
#         }

#     # Assemble result blob.
#     results = {
#         'stage':           stage,
#         'checkpoint_path': best_path,
#         'checkpoint_meta': {
#             'epoch':       _state.get('epoch'),
#             'global_step': _state.get('global_step'),
#             'loss':        float(_state.get('loss', _state.get('best_loss', float('nan')))),
#         },
#         'n_samples_accuracy': len(refs),
#         'n_samples_latency':  len(latencies_ms),
#         'wer': float(wer),
#         'cer': float(cer),
#         'macro_cer': float(macro_cer),
#         'macro_cer_languages': macro_languages,
#         'wer_raw': float(wer_raw),
#         'cer_raw': float(cer_raw),
#         'metric_normalization': 'NFKC+casefold+punctuation-strip',
#         'per_language': per_language,
#         'collapse': collapse_metrics,
#         'evaluation_valid_for_unbiased_comparison': evaluation_lineage_clean,
#         'checkpoint_split_strategy': _state.get('split_strategy'),
#         'checkpoint_split_fingerprint': _state.get('split_fingerprint'),
#         'active_split_strategy': splits.get('strategy'),
#         'active_split_fingerprint': splits.get('fingerprint'),
#         'empty_hypothesis_pct': empty_hyp_pct,
#         'avg_pred_token_len': float(np.mean(pred_lens)) if pred_lens else 0.0,
#         'avg_ref_token_len': float(np.mean(ref_lens)) if ref_lens else 0.0,
#         'latency_ms': {
#             'mean': float(lat_arr.mean()),
#             'std':  float(lat_arr.std()),
#             'p50':  float(np.percentile(lat_arr, 50)),
#             'p95':  float(np.percentile(lat_arr, 95)),
#             'p99':  float(np.percentile(lat_arr, 99)),
#             'min':  float(lat_arr.min()),
#             'max':  float(lat_arr.max()),
#         },
#         'system': {
#             'gpu_name':              torch.cuda.get_device_name(0),
#             'precision':             'fp16 (autocast)',
#             'torch_version':         torch.__version__,
#             'transformers_version':  _transformers_mod.__version__,
#         },
#         'config': {
#             'batch_size_accuracy':  C.validation_batch_size,
#             'batch_size_latency':   1,
#             'max_audio_seconds':    max_seconds,
#             'student_params_M':     sum(p.numel() for p in test_student.parameters()) / 1e6,
#             'note':                 'Latency = student forward only; mel precomputed.',
#         },
#         'splits': {
#             'splits_path':  SPLITS_PATH,
#             'fingerprint':  splits.get('fingerprint'),
#             'seed':         splits.get('seed'),
#             'test_size':    splits.get('test_size'),
#         },
#     }
#     if rtf_stats is not None:
#         results['rtf'] = rtf_stats

#     with open(results_path, 'w') as f:
#         json.dump(results, f, indent=2)

#     # ── Print summary ──
#     print(f'\n{"=" * 60}')
#     print(f'TEST SET EVALUATION ({stage})')
#     print(f'{"=" * 60}')
#     print(f'  WER (norm)   : {results["wer"]*100:7.2f}%')
#     print(f'  CER (norm)   : {results["cer"]*100:7.2f}%')
#     print(f'  Macro CER    : {results["macro_cer"]*100:7.2f}%')
#     print(f'  WER (raw)    : {results["wer_raw"]*100:7.2f}%')
#     print(f'  CER (raw)    : {results["cer_raw"]*100:7.2f}%')
#     print(f'  Unbiased eval: {results["evaluation_valid_for_unbiased_comparison"]}')
#     print(f'  Latency mean : {results["latency_ms"]["mean"]:7.1f} ms')
#     print(f'  Latency p50  : {results["latency_ms"]["p50"]:7.1f} ms')
#     print(f'  Latency p95  : {results["latency_ms"]["p95"]:7.1f} ms')
#     print(f'  Latency p99  : {results["latency_ms"]["p99"]:7.1f} ms')
#     if rtf_stats is not None:
#         print(f'  RTF mean     : {rtf_stats["mean"]:7.3f}  '
#               '(student forward only; <1.0 = faster than real-time)')
#     print(f'\nArtifacts saved (use these to compare other methods on the same set):')
#     print(f'  splits.json      : {SPLITS_PATH}')
#     print(f'  predictions.jsonl: {pred_path}')
#     print(f'  results.json     : {results_path}')

#     del test_student
#     torch.cuda.empty_cache()


### CTC output diagnostic (run if WER stays at 100%)

After training has saved at least one checkpoint, this cell loads it, runs the model on 10 validation samples, and prints the per-frame argmax distribution. WER=100% has four possible causes — this output tells you which:

- **Blank-dominant collapse** (>95% of frames argmax to BLANK): expected during frozen stage. The CTC head can't overcome the trivial "predict blank everywhere" local minimum without backbone gradients. Not a bug — move to unfrozen stage.
- **Mode collapse** on a single non-blank token: training instability, often a vocab/temperature issue.
- **Diverse outputs but wrong text**: model is learning, just hasn't converged yet on alignment.
- **Tokenizer mismatch**: token IDs look right but decoded text doesn't match — bug in the encode/decode round-trip.


In [20]:
# # ── CTC output diagnostic ─────────────────────────────────────
# # Distinguishes blank-collapse vs mode-collapse vs genuine slow learning vs
# # tokenizer bug. Run after at least one stage has saved a checkpoint.
# from torch.utils.data import Subset, DataLoader
# from collections import Counter

# # Prefer recovery checkpoints; legacy fallbacks are only for comparison.
# _diag_candidates = [
#     os.path.join(C.ckpt_dir,'scratch_joint_latest.pt'),
#     os.path.join(C.ckpt_dir,'scratch_joint_best.pt'),
#     os.path.join(C.ckpt_dir,'recover_ctc_latest.pt'),
#     os.path.join(C.ckpt_dir,'recover_ctc_best.pt'),
#     os.path.join(C.ckpt_dir,'recover_kd_latest.pt'),
#     os.path.join(C.ckpt_dir,'chunk_ctc_latest.pt'),
#     os.path.join(C.ckpt_dir,'unfrozen_latest.pt'),
#     os.path.join(C.ckpt_dir,'frozen_latest.pt'),
# ]
# ckpt_path = next((p for p in _diag_candidates if os.path.exists(p)), None)
# if ckpt_path is None:
#     raise FileNotFoundError('No checkpoint found. Train at least partway through a stage first.')

# # Build a fresh student so we don't disturb the in-memory training state.
# diag = ConMambaStudent(C).to(device)
# _state = torch.load(ckpt_path, map_location='cpu', weights_only=False)
# _require_checkpoint_tokenizer(_state, ckpt_path, allow_missing=True)
# _sd = _state['model']
# if any(k.startswith('module.') for k in _sd):
#     _sd = {(k[len('module.'):] if k.startswith('module.') else k): v
#            for k, v in _sd.items()}
# _load_model_tolerant(diag, _sd)
# diag.eval()
# print(f'Loaded {ckpt_path}')
# print(f'  epoch={_state.get("epoch")}, step={_state.get("global_step")}, '
#       f'best_loss={_state.get("best_loss", float("nan")):.4f}')

# # Reference summary of vocab structure.
# print(f'\nVocab structure:')
# print(
#     f'  sp.GetPieceSize()  = {
#         sp.GetPieceSize()}  (SPM token IDs 0..{
#             sp.GetPieceSize() -
#             1})')
# print(f'  BLANK              = {BLANK}  (CTC blank index)')
# print(f'  C.vocab_size       = {C.vocab_size}  (ctc_head output dim)')
# assert BLANK == sp.GetPieceSize(), "BLANK should be one past the SPM range"
# assert C.vocab_size == BLANK + 1, "ctc_head must output (vocab + 1) classes"

# # Pull 10 val samples from the persisted splits.
# SPLITS_PATH = os.path.join(C.ckpt_dir, 'splits.json')
# with open(SPLITS_PATH) as f:
#     splits_data = json.load(f)
# val_subset = Subset(ds, splits_data['val_indices'][:10])
# loader = DataLoader(
#     val_subset, batch_size=1, shuffle=False, num_workers=0,
#     collate_fn=lambda b: collate_kd(b, max_mel_frames=None, allow_crop=False),
# )

# print('\n' + '─' * 80)

# all_argmax = Counter()
# all_pred_ids = []
# all_frame_ids = []
# all_hyps = []
# all_refs = []
# for i, batch in enumerate(loader):
#     mel = batch['mel'].to(device)
#     mel_lens = batch['mel_lens'].to(device)
#     n_tokens = batch['tok_lens'][0].item()

#     with torch.no_grad(), torch.amp.autocast('cuda', dtype=torch.float16):
#         ctc_logits, _kl, enc_lens = diag(mel, mel_lens)
#     valid_len = enc_lens[0].item()

#     # Per-frame argmax (after masking to valid frames).
#     argmax_ids = ctc_logits[0, :valid_len].float().argmax(dim=-1).cpu().tolist()
#     counter = Counter(argmax_ids)
#     all_argmax.update(counter)
#     all_frame_ids.append(argmax_ids)

#     # First-frame top-5 — shows whether non-blank tokens are even in contention.
#     probs = ctc_logits[0, 0].float().softmax(dim=-1)
#     top5 = probs.topk(5)

#     ref = batch['texts'][0]
#     pred_ids = greedy_ctc_token_ids(
#         ctc_logits.float().cpu(), BLANK, lengths=[valid_len],
#     )[0]
#     all_pred_ids.append(pred_ids)
#     decoded = sp.DecodeIds(pred_ids)
#     all_hyps.append(decoded)
#     all_refs.append(ref)
#     blank_pct = 100 * counter.get(BLANK, 0) / valid_len

#     print(f'\n[{i}] frames={valid_len}  target_tokens={n_tokens}  '
#           f'blank_pct={blank_pct:5.1f}%')
#     print(f'    REF: {ref[:100]}')
#     print(f'    HYP: {decoded[:100] if decoded else "(empty — only blanks)"}')
#     print(f'    argmax top-3: ', end='')
#     for tid, cnt in counter.most_common(3):
#         piece = sp.id_to_piece(tid) if tid < sp.GetPieceSize() else f'<BLANK>'
#         print(f'{piece}({cnt})  ', end='')
#     print()
#     print(f'    frame-0 top-5: ', end='')
#     for j in range(5):
#         tid = top5.indices[j].item()
#         prob = top5.values[j].item()
#         piece = sp.id_to_piece(tid) if tid < sp.GetPieceSize() else '<BLANK>'
#         print(f'{piece}({prob:.2f})  ', end='')
#     print()

# print('\n' + '═' * 80)
# print('AGGREGATE across 10 samples:')
# total = sum(all_argmax.values())
# print(f'  Total frames examined: {total}')
# print(f'  Top-10 most-emitted tokens:')
# top10 = all_argmax.most_common(10)
# for tid, count in top10:
#     piece = sp.id_to_piece(tid) if tid < sp.GetPieceSize() else '<BLANK>'
#     marker = '  <-- BLANK' if tid == BLANK else ''
#     print(f'    id={tid:5d}  {piece:25s} {count:7d}  ({100*count/total:5.1f}%){marker}')

# # Verdict
# print('\n' + '?' * 80)
# print('DIAGNOSIS:')
# metrics = compute_collapse_metrics(
#     all_pred_ids, BLANK, frame_argmax_ids=all_frame_ids,
#     refs=all_refs, hyps=all_hyps, sp_model=sp,
# )
# blank_pct = metrics['blank_frame_pct'] * 100
# top1_id, top1_count = top10[0]
# top1_pct = 100 * top1_count / total
# print(f'  collapsed={metrics["collapsed"]} reason={metrics["reason"]}')
# print(f'  blank_frame_pct={blank_pct:.1f}%')
# print(f'  empty_hypothesis_pct={metrics["empty_hypothesis_pct"]*100:.1f}%')
# print(f'  top_token_ratio={metrics["top_ratio"]*100:.1f}%')
# print(f'  avg_pred_token_len={metrics["avg_pred_token_len"]:.2f}')
# print(f'  avg_ref_token_len={metrics["avg_ref_token_len"]:.2f}')

# if metrics['reason'] == 'blank_dominant':
#     print('  (a) BLANK-DOMINANT COLLAPSE: reset the CTC head and restart recover_ctc.')
# elif metrics['reason'] == 'nonblank_mode':
#     piece = sp.id_to_piece(metrics['top_id']) if metrics['top_id'] is not None else '?'
#     print(f'  (b) MODE COLLAPSE on token "{piece}" (id={metrics["top_id"]}).')
# elif metrics['reason'] == 'empty_hypothesis':
#     print('  (c) EMPTY-HYPOTHESIS COLLAPSE: decoded outputs are blank/empty.')
# else:
#     print('  (d) Non-collapsed: alignment is forming; continue training/evaluate WER trend.')
# # Bonus diagnostic: feature distillation health.
# # loss_kl ~0.004 is suspicious — verify the student\'s kl_feat is actually
# # varying across frames rather than predicting the per-sample mean.
# print('\n' + '─' * 80)
# print('Bonus: kl_feat diversity check (verify it\'s not just predicting mean):')
# # The front-end conv runs under autocast fp16; the encoder forces fp32 internally
# # (the selective-scan fp32 island). Run under autocast so the conv input matches.
# with torch.no_grad(), torch.amp.autocast('cuda', dtype=torch.float16):
#     batch = next(iter(loader))
#     mel = batch['mel'].to(device)
#     teacher_h = batch['teacher_h'].to(device)
#     mel_lens = batch['mel_lens'].to(device)
#     _ctc, kl_feat, enc_lens = diag(mel, mel_lens)
#     s_len = enc_lens[0].item()
#     t_len = teacher_h.shape[1]
#     student_var = kl_feat[0, :s_len].float().var(dim=0).mean().item()
#     teacher_var = teacher_h[0, :t_len].float().var(dim=0).mean().item()
#     print(f'  per-frame variance (avg over feature dim):')
#     print(f'    teacher_h    : {teacher_var:.8e}')
#     print(f'    student_kl   : {student_var:.8e}  '
#           f'(ratio {student_var/max(teacher_var, 1e-9):.2f})')
#     if student_var < 0.1 * teacher_var:
#         print('  WARN: student_kl variance << teacher. The low loss_kl may be')
#         print('        partially from predicting the per-sample mean rather than')
#         print('        frame-level details. Not catastrophic but worth noting.')

# del diag
# torch.cuda.empty_cache()
